<a href="https://colab.research.google.com/github/KJJA/IVMOS/blob/main/IVMOS_Market_Engine_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IVMOS Market Engine — Clean Production Candidate

Executable pipeline only:

1. Bootstrap and Drive contract
2. Data dictionary
3. Price ingestion
4. FRED API macro ingestion
5. Feature engineering with integrated dtype normalization
6. Evidence Registry and Calibration
7. Evidence Runtime
8. Confidence v2.1 with Credit FULL/PROXY mode
9. Final validation

Removed: Git setup, Drive repair experiments, research audits, v1.1 archive, and FRED CSV repair attempts.


In [3]:
# =============================================================================
# IVMOS — BOOTSTRAP (FAIL-FAST)
# =============================================================================

from google.colab import drive
from pathlib import Path
import importlib.util
import os
import random
import subprocess
import sys

MOUNT_POINT = Path("/content/drive")
MY_DRIVE = MOUNT_POINT / "MyDrive"

if not MY_DRIVE.exists():
    drive.mount(str(MOUNT_POINT), force_remount=False)

PROJECT_ROOT = MY_DRIVE / "IVMOS"

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"IVMOS project not found at {PROJECT_ROOT}. "
        "Check the mounted Google account. Local fallback is intentionally disabled."
    )

os.environ["IVMOS_PROJECT_ROOT"] = str(PROJECT_ROOT)

for relative in [
    "config",
    "raw/prices",
    "raw/macro",
    "processed",
    "outputs",
    "logs",
    "tests",
]:
    (PROJECT_ROOT / relative).mkdir(parents=True, exist_ok=True)

PACKAGE_IMPORTS = {
    "yfinance": "yfinance",
    "pyarrow": "pyarrow",
    "pandas_market_calendars": "pandas-market-calendars",
    "yaml": "pyyaml",
}

missing_packages = [
    package_name
    for module_name, package_name in PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
    )

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

report = {
    "status": "PASS",
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "errors": [],
    "warnings": [],
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Drive contract: PASS")
print("Local fallback: DISABLED")
print("Bootstrap: PASS")


PROJECT_ROOT: /content/drive/MyDrive/IVMOS
Drive contract: PASS
Local fallback: DISABLED
Bootstrap: PASS


## Step 2 — Data Dictionary


In [4]:
import pandas as pd
import os

data_dictionary_data = []

# Helper function to add instruments to the data dictionary
def add_instrument(instrument_id, group, asset_type, source_primary, required, notes=None, unit=None):
    # Determine common attributes
    mechanism = 'yfinance' if source_primary == 'yfinance' else 'FRED API'
    adjusted_required = True if source_primary == 'yfinance' and asset_type != 'Index' else False
    frequency = 'Daily'
    max_staleness_days = 1
    source_fallback = None # Not specified, so default to None

    # Set display_name, use instrument_id as default
    display_name = instrument_id

    # Specific unit adjustments
    if unit is None:
        if asset_type == 'Macro':
            if instrument_id in ['DGS10', 'DGS2', 'DFII10', 'T10Y2Y', 'SOFR']:
                unit = 'Percent'
            elif instrument_id == 'BAMLH0A0HYM2':
                unit = 'Percent'
            elif instrument_id in ['WALCL', 'WTREGEN']:
                unit = 'Millions USD'
            elif instrument_id == 'RRPONTSYD':
                unit = 'Billions USD'
            else:
                unit = 'Index' # Default for other FRED indices like VIXCLS
        else: # Price instruments
            unit = 'USD'

    data_dictionary_data.append({
        'instrument_id': instrument_id,
        'display_name': display_name,
        'asset_type': asset_type,
        'group': group,
        'mechanism': mechanism,
        'source_primary': source_primary,
        'source_fallback': source_fallback,
        'frequency': frequency,
        'unit': unit,
        'adjusted_required': adjusted_required,
        'required': required,
        'max_staleness_days': max_staleness_days,
        'notes': notes
    })

# A. FAST_MARKET_CORE
fast_market_core = ['SPY', 'QQQ', 'RSP', 'IWM', 'SOXX', 'HYG', 'LQD', 'TLT', 'UUP', 'GLD', '^VIX']
for instr in fast_market_core:
    asset_type = 'Index' if instr == '^VIX' else ('ETF' if instr != 'GLD' else 'Commodity')
    add_instrument(instr, 'FAST_MARKET_CORE', asset_type, 'yfinance', True)

# B. SECTOR_ROTATION_CORE
sector_rotation_core = ['XLK', 'XLV', 'XLF', 'XLE', 'XLI', 'XLU']
for instr in sector_rotation_core:
    add_instrument(instr, 'SECTOR_ROTATION_CORE', 'ETF', 'yfinance', True)

# C. SECTOR_ROTATION_OPTIONAL
sector_rotation_optional = ['XLB', 'XLY', 'XLP', 'XLRE']
for instr in sector_rotation_optional:
    add_instrument(instr, 'SECTOR_ROTATION_OPTIONAL', 'ETF', 'yfinance', False)

# D. DIAGNOSTIC_PRICE
diagnostic_price = ['TIP', 'IEF', 'SHY', 'USO', 'CPER', 'SMH']
for instr in diagnostic_price:
    add_instrument(instr, 'DIAGNOSTIC_PRICE', 'ETF', 'yfinance', False)

# E. AI_COMPUTE
ai_compute = ['NVDA', 'TSM', 'ASML', 'MU', 'AVGO']
for instr in ai_compute:
    add_instrument(instr, 'AI_COMPUTE', 'Equity', 'yfinance', False)

# F. AI_CONNECTIVITY
ai_connectivity = ['ANET', 'CRDO', 'MRVL', 'COHR', 'LITE']
for instr in ai_connectivity:
    add_instrument(instr, 'AI_CONNECTIVITY', 'Equity', 'yfinance', False)

# G. AI_POWER_GRID
ai_power_grid = ['ETN', 'VRT', 'GEV', 'HUBB', 'APH']
for instr in ai_power_grid:
    add_instrument(instr, 'AI_POWER_GRID', 'Equity', 'yfinance', False)

# H. AI_SOFTWARE
ai_software = ['MSFT', 'GOOGL', 'ORCL', 'NOW', 'PLTR']
for instr in ai_software:
    add_instrument(instr, 'AI_SOFTWARE', 'Equity', 'yfinance', False)

# I. AI_SPECULATIVE
ai_speculative = ['RKLB', 'ASTS', 'OKLO', 'LEU', 'IREN']
for instr in ai_speculative:
    add_instrument(instr, 'AI_SPECULATIVE', 'Equity', 'yfinance', False)

# J. FRED_DAILY
fred_daily = ['DGS10', 'DGS2', 'T10Y2Y', 'DFII10', 'BAMLH0A0HYM2', 'VIXCLS', 'SOFR']
required_fred_daily = ['DGS10', 'DGS2', 'DFII10', 'BAMLH0A0HYM2', 'SOFR']
for instr in fred_daily:
    is_required = instr in required_fred_daily
    add_instrument(instr, 'FRED_DAILY', 'Macro', 'FRED', is_required)

# K. FRED_LIQUIDITY_OPTIONAL
fred_liquidity_optional = ['WALCL', 'WTREGEN', 'RRPONTSYD']
for instr in fred_liquidity_optional:
    add_instrument(instr, 'FRED_LIQUIDITY_OPTIONAL', 'Macro', 'FRED', False)

# Create the DataFrame
data_dictionary = pd.DataFrame(data_dictionary_data)

# Source-of-truth corrections
data_dictionary.loc[
    data_dictionary["instrument_id"].eq("BAMLH0A0HYM2"),
    "notes",
] = (
    "FRED/ICE publication may expose only limited trailing history. "
    "Use HYG/LQD as a lower-confidence historical Credit proxy when HY OAS is unavailable."
)

data_dictionary.loc[
    data_dictionary["instrument_id"].isin(["WALCL", "WTREGEN"]),
    "frequency",
] = "Weekly"

data_dictionary.loc[
    data_dictionary["instrument_id"].isin(["WALCL", "WTREGEN"]),
    "max_staleness_days",
] = 10

data_dictionary.loc[
    data_dictionary["instrument_id"].eq("RRPONTSYD"),
    "max_staleness_days",
] = 3

print("Data Dictionary created successfully.")

# Export to CSV and JSON
csv_path = os.path.join(PROJECT_ROOT, 'config', 'data_dictionary.csv')
json_path = os.path.join(PROJECT_ROOT, 'config', 'data_dictionary.json')

data_dictionary.to_csv(csv_path, index=False)
data_dictionary.to_json(json_path, orient='records', indent=4)

print(f"Data Dictionary exported to: {csv_path}")
print(f"Data Dictionary exported to: {json_path}")

# --- Validation Checks ---
validation_errors = []

# 1. No duplicate instrument_id
if data_dictionary['instrument_id'].duplicated().any():
    duplicates = data_dictionary[data_dictionary['instrument_id'].duplicated(keep=False)]
    validation_errors.append(f"Duplicate instrument_id found: {duplicates['instrument_id'].tolist()}")

# 2. Every Instrument has a Group
if data_dictionary['group'].isnull().any():
    validation_errors.append("Instruments found with null 'group'.")

# 3. Every Instrument has a Source (source_primary)
if data_dictionary['source_primary'].isnull().any():
    validation_errors.append("Instruments found with null 'source_primary'.")

# 4. 'required' has no null values
if data_dictionary['required'].isnull().any():
    validation_errors.append("The 'required' column contains null values.")

# Report Validation results
print("\n--- STEP 2 VALIDATION REPORT ---")
if validation_errors:
    report['status'] = 'FAIL'
    report['errors'].extend(validation_errors)
    print("Validation FAILED. Errors found:")
    for error in validation_errors:
        print(f"- {error}")
else:
    print("Validation PASSED: All checks successful.")
    # Update overall report status if no errors found in this step and previous was PASS or PARTIAL
    if report['status'] != 'FAIL':
        report['status'] = 'PASS'

print("\n--- STEP 2 REPORT ---")
print(f"Status: {report['status']}")
print("Created Files:")
print(f"- {csv_path}")
print(f"- {json_path}")
if report['errors']:
    print("Errors:")
    for error in report['errors']:
        print(f"- {error}")
else:
    print("No Errors.")
print("---------------------")


Data Dictionary created successfully.
Data Dictionary exported to: /content/drive/MyDrive/IVMOS/config/data_dictionary.csv
Data Dictionary exported to: /content/drive/MyDrive/IVMOS/config/data_dictionary.json

--- STEP 2 VALIDATION REPORT ---
Validation PASSED: All checks successful.

--- STEP 2 REPORT ---
Status: PASS
Created Files:
- /content/drive/MyDrive/IVMOS/config/data_dictionary.csv
- /content/drive/MyDrive/IVMOS/config/data_dictionary.json
No Errors.
---------------------


## Step 3 — Price Data Engine


In [5]:
# ============================================================
# STEP 3 — FETCH PRICE DATA (CORRECTED / ROBUST VERSION)
# ============================================================

from pathlib import Path
import datetime as dt
import json
import os
import time

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import yfinance as yf


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

START_DATE = "2015-01-01"

# yfinance ใช้ end แบบ exclusive จึงบวกหนึ่งวัน
END_DATE = (
    pd.Timestamp.now(tz="America/New_York") + pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

INTERVAL = "1d"
MAX_RETRIES = 3
INITIAL_BACKOFF_SEC = 2
CHUNK_SIZE = 12


# Fail fast after runtime restart; never create a shadow local project.
if "PROJECT_ROOT" not in globals():
    raise RuntimeError("Run the Bootstrap cell before Step 3.")

if "report" not in globals():
    report = {
        "status": "PASS",
        "errors": [],
    }


PROJECT_ROOT = str(PROJECT_ROOT)

CONFIG_DIR = Path(PROJECT_ROOT) / "config"
RAW_PRICE_DIR = Path(PROJECT_ROOT) / "raw" / "prices"
LOG_DIR = Path(PROJECT_ROOT) / "logs"
OUTPUT_DIR = Path(PROJECT_ROOT) / "outputs"

for directory in [CONFIG_DIR, RAW_PRICE_DIR, LOG_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("yfinance version:", yf.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Download period:", START_DATE, "to", END_DATE)


# ------------------------------------------------------------
# 2. LOAD DATA DICTIONARY
# ------------------------------------------------------------

dictionary_path = CONFIG_DIR / "data_dictionary.csv"

if not dictionary_path.exists():
    raise FileNotFoundError(
        f"ไม่พบ Data Dictionary: {dictionary_path}\n"
        "ให้รัน Step 2 ใหม่ก่อน"
    )

data_dictionary = pd.read_csv(dictionary_path)

required_columns = {
    "instrument_id",
    "asset_type",
    "source_primary",
    "required",
}

missing_columns = required_columns - set(data_dictionary.columns)

if missing_columns:
    raise ValueError(
        f"Data Dictionary ขาด columns: {sorted(missing_columns)}"
    )


def parse_bool(value) -> bool:
    """แปลงค่า bool ที่อ่านมาจาก CSV ให้แน่นอน"""
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true", "1", "yes", "y", "required"
    }


data_dictionary["required_bool"] = (
    data_dictionary["required"].apply(parse_bool)
)

source_text = (
    data_dictionary["source_primary"]
    .fillna("")
    .astype(str)
    .str.lower()
)

asset_type_text = (
    data_dictionary["asset_type"]
    .fillna("")
    .astype(str)
    .str.lower()
)

# รองรับทั้งคำว่า yfinance และ Yahoo Finance
price_source_mask = source_text.str.contains(
    r"yfinance|yahoo",
    regex=True,
)

macro_mask = asset_type_text.str.contains(
    r"macro|fred",
    regex=True,
)

price_instruments_df = data_dictionary[
    price_source_mask & ~macro_mask
].copy()

tickers_to_fetch = (
    price_instruments_df["instrument_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

required_tickers = set(
    price_instruments_df.loc[
        price_instruments_df["required_bool"],
        "instrument_id",
    ].astype(str)
)

if not tickers_to_fetch:
    raise ValueError(
        "ไม่พบ Price instruments ใน Data Dictionary\n"
        "ตรวจค่า source_primary ว่าเป็น yfinance หรือ Yahoo Finance"
    )

print(
    f"\nพบ Price instruments จำนวน {len(tickers_to_fetch)} ตัว"
)
print(", ".join(tickers_to_fetch))


# ------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ------------------------------------------------------------

FETCHED_AT = pd.Timestamp.now(tz="UTC")


def chunked(items, size):
    """แบ่ง Ticker เป็นชุดย่อย ลดโอกาส Batch ใหญ่ล้มเหลว"""
    for position in range(0, len(items), size):
        yield items[position:position + size]


def extract_ticker_frame(
    raw: pd.DataFrame,
    ticker: str,
) -> pd.DataFrame:
    """
    แปลงผลลัพธ์ yfinance ให้เป็น Long-format มาตรฐาน

    ใช้ auto_adjust=True:
    - Open/High/Low/Close ถูกปรับ Split/Dividend แล้ว
    - ใช้ Close ไม่ใช้ Adj Close
    """

    if raw is None or raw.empty:
        return pd.DataFrame()

    frame = raw.copy()

    # รองรับ MultiIndex ทั้ง (Ticker, Price) และ (Price, Ticker)
    if isinstance(frame.columns, pd.MultiIndex):
        level_0 = frame.columns.get_level_values(0)
        level_last = frame.columns.get_level_values(-1)

        if ticker in level_0:
            frame = frame[ticker].copy()

        elif ticker in level_last:
            frame = frame.xs(
                ticker,
                axis=1,
                level=-1,
                drop_level=True,
            ).copy()

        else:
            return pd.DataFrame()

    # Normalize column names
    frame.columns = [
        str(column).strip().lower().replace(" ", "_")
        for column in frame.columns
    ]

    # Fallback เฉพาะกรณี Source คืน Adj Close
    if "close" not in frame.columns and "adj_close" in frame.columns:
        frame["close"] = frame["adj_close"]

    # Volume อาจไม่มีสำหรับ Index เช่น ^VIX
    for column in ["open", "high", "low", "close", "volume"]:
        if column not in frame.columns:
            frame[column] = np.nan

    frame = frame[
        ["open", "high", "low", "close", "volume"]
    ].copy()

    # Close เป็น field ขั้นต่ำที่จำเป็น
    frame = frame.loc[frame["close"].notna()].copy()

    if frame.empty:
        return pd.DataFrame()

    # Normalize date ให้เป็น timezone-naive Trading date
    date_index = pd.to_datetime(frame.index)

    if getattr(date_index, "tz", None) is not None:
        date_index = (
            date_index
            .tz_convert("America/New_York")
            .tz_localize(None)
        )

    frame.index = date_index.normalize()
    frame.index.name = "date"
    frame = frame.reset_index()

    frame["ticker"] = ticker
    frame["source"] = "yfinance"
    frame["fetched_at"] = FETCHED_AT
    frame["adjusted"] = True

    return frame[
        [
            "date",
            "ticker",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "source",
            "fetched_at",
            "adjusted",
        ]
    ]


def download_batch(batch_tickers):
    """ดาวน์โหลด Batch โดยกำหนดรูปแบบให้ชัดเจน"""
    return yf.download(
        tickers=batch_tickers,
        start=START_DATE,
        end=END_DATE,
        interval=INTERVAL,
        auto_adjust=True,
        actions=False,
        group_by="ticker",
        threads=True,
        progress=False,
        ignore_tz=True,
        timeout=30,
        multi_level_index=True,
    )


def download_single(ticker):
    """ดาวน์โหลดรายตัวโดยบังคับไม่ใช้ MultiIndex"""
    return yf.download(
        tickers=ticker,
        start=START_DATE,
        end=END_DATE,
        interval=INTERVAL,
        auto_adjust=True,
        actions=False,
        group_by="column",
        threads=False,
        progress=False,
        ignore_tz=True,
        timeout=30,
        multi_level_index=False,
    )


# ------------------------------------------------------------
# 4. BATCH DOWNLOAD
# ------------------------------------------------------------

fetched_frames = []
fetch_log = []
successful_tickers = set()
failed_tickers = set()

batches = list(chunked(tickers_to_fetch, CHUNK_SIZE))

print(f"\nเริ่ม Batch download จำนวน {len(batches)} ชุด")

for batch_number, batch_tickers in enumerate(batches, start=1):

    print(
        f"\nBatch {batch_number}/{len(batches)}: "
        f"{', '.join(batch_tickers)}"
    )

    try:
        batch_raw = download_batch(batch_tickers)

        for ticker in batch_tickers:
            frame = extract_ticker_frame(batch_raw, ticker)

            if frame.empty:
                print(f"  {ticker}: ไม่มีข้อมูลใน Batch")
                continue

            fetched_frames.append(frame)
            successful_tickers.add(ticker)

            fetch_log.append({
                "ticker": ticker,
                "status": "SUCCESS",
                "method": "batch",
                "attempts": 1,
                "rows": len(frame),
                "start_date": str(frame["date"].min().date()),
                "end_date": str(frame["date"].max().date()),
            })

            print(
                f"  {ticker}: OK — {len(frame):,} rows"
            )

    except Exception as error:
        print(f"Batch {batch_number} ล้มเหลว: {error}")

        fetch_log.append({
            "tickers": batch_tickers,
            "status": "BATCH_FAILURE",
            "method": "batch",
            "error": repr(error),
        })


# ------------------------------------------------------------
# 5. INDIVIDUAL RETRIES
# ------------------------------------------------------------

remaining_tickers = [
    ticker
    for ticker in tickers_to_fetch
    if ticker not in successful_tickers
]

print(
    f"\nต้อง Retry รายตัวจำนวน {len(remaining_tickers)} ตัว"
)

for ticker in remaining_tickers:

    ticker_success = False
    final_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            print(
                f"{ticker}: attempt {attempt}/{MAX_RETRIES}"
            )

            single_raw = download_single(ticker)
            frame = extract_ticker_frame(single_raw, ticker)

            if frame.empty:
                raise ValueError(
                    f"No usable price data returned for {ticker}"
                )

            fetched_frames.append(frame)
            successful_tickers.add(ticker)
            ticker_success = True

            fetch_log.append({
                "ticker": ticker,
                "status": "SUCCESS",
                "method": "individual",
                "attempts": attempt,
                "rows": len(frame),
                "start_date": str(frame["date"].min().date()),
                "end_date": str(frame["date"].max().date()),
            })

            print(
                f"{ticker}: OK — {len(frame):,} rows"
            )

            break

        except Exception as error:
            final_error = error

            fetch_log.append({
                "ticker": ticker,
                "status": "RETRY_FAILURE",
                "method": "individual",
                "attempt": attempt,
                "error": repr(error),
            })

            if attempt < MAX_RETRIES:
                wait_seconds = INITIAL_BACKOFF_SEC * (
                    2 ** (attempt - 1)
                )

                print(
                    f"{ticker}: error={error}; "
                    f"รอ {wait_seconds} วินาที"
                )

                time.sleep(wait_seconds)

    if not ticker_success:
        failed_tickers.add(ticker)

        fetch_log.append({
            "ticker": ticker,
            "status": "PERMANENT_FAILURE",
            "method": "individual",
            "attempts": MAX_RETRIES,
            "error": repr(final_error),
        })

        print(f"{ticker}: FAILED")


# ------------------------------------------------------------
# 6. COMBINE LONG-FORM DATA
# ------------------------------------------------------------

if not fetched_frames:
    raise RuntimeError(
        "ไม่สามารถดึงข้อมูลราคาได้แม้แต่ Ticker เดียว"
    )

all_price_data = pd.concat(
    fetched_frames,
    ignore_index=True,
)

# ป้องกันข้อมูลซ้ำจาก Batch และ Retry
all_price_data = (
    all_price_data
    .drop_duplicates(
        subset=["date", "ticker"],
        keep="last",
    )
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

successful_tickers = sorted(
    all_price_data["ticker"].unique().tolist()
)

failed_tickers = sorted(
    set(tickers_to_fetch) - set(successful_tickers)
)

print("\n--- FETCH SUMMARY ---")
print("Requested:", len(tickers_to_fetch))
print("Successful:", len(successful_tickers))
print("Failed:", len(failed_tickers))
print("Total rows:", f"{len(all_price_data):,}")

if failed_tickers:
    print("Failed tickers:", ", ".join(failed_tickers))


# ------------------------------------------------------------
# 7. VALIDATION
# ------------------------------------------------------------

quality = {
    "step": 3,
    "validation_status": "PASS",
    "fetched_at": str(FETCHED_AT),
    "requested_count": len(tickers_to_fetch),
    "successful_count": len(successful_tickers),
    "failed_count": len(failed_tickers),
    "successful_tickers": successful_tickers,
    "failed_tickers": failed_tickers,
    "missing_required_tickers": [],
    "stale_required_tickers": [],
    "critical_errors": {
        "duplicate_date_ticker": False,
        "non_positive_close": [],
        "high_below_low": [],
        "missing_required_ohlc": [],
    },
    "warnings": {
        "missing_volume": [],
        "less_than_200_rows": [],
        "history_starts_after_requested_date": [],
    },
    "summary_per_ticker": {},
    "files_created": [],
}


# 7.1 Duplicate
duplicate_mask = all_price_data.duplicated(
    subset=["date", "ticker"],
    keep=False,
)

quality["critical_errors"]["duplicate_date_ticker"] = bool(
    duplicate_mask.any()
)


# 7.2 Price consistency
non_positive_close = sorted(
    all_price_data.loc[
        all_price_data["close"] <= 0,
        "ticker",
    ].unique().tolist()
)

quality["critical_errors"]["non_positive_close"] = (
    non_positive_close
)

high_below_low = sorted(
    all_price_data.loc[
        all_price_data["high"] < all_price_data["low"],
        "ticker",
    ].unique().tolist()
)

quality["critical_errors"]["high_below_low"] = (
    high_below_low
)


# 7.3 Required OHLC and Volume
for ticker, ticker_frame in all_price_data.groupby("ticker"):

    missing_ohlc = [
        column
        for column in ["open", "high", "low", "close"]
        if ticker_frame[column].isna().any()
    ]

    if missing_ohlc:
        quality["critical_errors"][
            "missing_required_ohlc"
        ].append({
            "ticker": ticker,
            "columns": missing_ohlc,
        })

    # Volume เป็น Warning ไม่ใช่ Error โดยเฉพาะ Index
    if ticker_frame["volume"].isna().any():
        quality["warnings"]["missing_volume"].append(ticker)


# 7.4 Trading-day staleness
nyse = mcal.get_calendar("XNYS")

today_ny = pd.Timestamp.now(
    tz="America/New_York"
).date()

recent_trading_days = nyse.valid_days(
    start_date=today_ny - dt.timedelta(days=14),
    end_date=today_ny,
)

# เปลี่ยนเป็น timezone-naive เพื่อเทียบกับ yfinance daily date
recent_trading_days = (
    recent_trading_days
    .tz_convert(None)
    .normalize()
)

if len(recent_trading_days) >= 3:
    freshness_threshold = recent_trading_days[-3]
elif len(recent_trading_days) > 0:
    freshness_threshold = recent_trading_days[0]
else:
    freshness_threshold = pd.Timestamp(today_ny)


for ticker in required_tickers:

    ticker_frame = all_price_data[
        all_price_data["ticker"] == ticker
    ]

    if ticker_frame.empty:
        quality["missing_required_tickers"].append(ticker)
        continue

    latest_date = ticker_frame["date"].max()

    if latest_date < freshness_threshold:
        quality["stale_required_tickers"].append({
            "ticker": ticker,
            "latest_date": str(latest_date.date()),
            "threshold_date": str(
                freshness_threshold.date()
            ),
        })


# 7.5 History length and summary
requested_start = pd.Timestamp(START_DATE)

for ticker, ticker_frame in all_price_data.groupby("ticker"):

    ticker_frame = ticker_frame.sort_values("date")

    start_date = ticker_frame["date"].min()
    end_date = ticker_frame["date"].max()
    row_count = len(ticker_frame)

    quality["summary_per_ticker"][ticker] = {
        "start_date": str(start_date.date()),
        "end_date": str(end_date.date()),
        "row_count": row_count,
    }

    if row_count < 200:
        quality["warnings"][
            "less_than_200_rows"
        ].append({
            "ticker": ticker,
            "row_count": row_count,
        })

    if start_date > requested_start:
        quality["warnings"][
            "history_starts_after_requested_date"
        ].append({
            "ticker": ticker,
            "start_date": str(start_date.date()),
        })


# 7.6 Final status
critical_failure = any([
    quality["critical_errors"]["duplicate_date_ticker"],
    bool(quality["critical_errors"]["non_positive_close"]),
    bool(quality["critical_errors"]["high_below_low"]),
    bool(quality["critical_errors"]["missing_required_ohlc"]),
    bool(quality["missing_required_tickers"]),
    bool(quality["stale_required_tickers"]),
])

has_warnings = any([
    bool(failed_tickers),
    bool(quality["warnings"]["missing_volume"]),
    bool(quality["warnings"]["less_than_200_rows"]),
    bool(
        quality["warnings"][
            "history_starts_after_requested_date"
        ]
    ),
])

if critical_failure:
    quality["validation_status"] = "FAIL"
elif has_warnings:
    quality["validation_status"] = "PARTIAL"
else:
    quality["validation_status"] = "PASS"

report["status"] = quality["validation_status"]

if critical_failure:
    report["errors"].append(
        "Step 3 critical validation failure"
    )


# ------------------------------------------------------------
# 8. EXPORT FILES
# ------------------------------------------------------------

parquet_path = RAW_PRICE_DIR / "prices_daily.parquet"
csv_path = RAW_PRICE_DIR / "prices_daily.csv.gz"
fetch_log_path = LOG_DIR / "price_fetch_log.json"
quality_path = OUTPUT_DIR / "price_data_quality.json"

all_price_data.to_parquet(
    parquet_path,
    index=False,
    compression="snappy",
)

all_price_data.to_csv(
    csv_path,
    index=False,
    compression="gzip",
)

with open(fetch_log_path, "w", encoding="utf-8") as file:
    json.dump(
        fetch_log,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

quality["files_created"] = [
    str(parquet_path),
    str(csv_path),
    str(fetch_log_path),
    str(quality_path),
]

with open(quality_path, "w", encoding="utf-8") as file:
    json.dump(
        quality,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3 EXECUTION REPORT")
print("=" * 60)

print("Status:", quality["validation_status"])
print(
    f"Successful: {quality['successful_count']}/"
    f"{quality['requested_count']}"
)
print("Failed:", quality["failed_tickers"] or "None")
print(
    "Missing required:",
    quality["missing_required_tickers"] or "None",
)
print(
    "Stale required:",
    quality["stale_required_tickers"] or "None",
)
print("Total rows:", f"{len(all_price_data):,}")

print("\nLatest dates:")
for ticker in successful_tickers:
    item = quality["summary_per_ticker"][ticker]
    print(
        f"  {ticker:8s} "
        f"{item['start_date']} → {item['end_date']} "
        f"({item['row_count']:,} rows)"
    )

print("\nFiles:")
for path in quality["files_created"]:
    print(" ", path)

print("=" * 60)


yfinance version: 0.2.66
PROJECT_ROOT: /content/drive/MyDrive/IVMOS
Download period: 2015-01-01 to 2026-08-08

พบ Price instruments จำนวน 52 ตัว
SPY, QQQ, RSP, IWM, SOXX, HYG, LQD, TLT, UUP, GLD, ^VIX, XLK, XLV, XLF, XLE, XLI, XLU, XLB, XLY, XLP, XLRE, TIP, IEF, SHY, USO, CPER, SMH, NVDA, TSM, ASML, MU, AVGO, ANET, CRDO, MRVL, COHR, LITE, ETN, VRT, GEV, HUBB, APH, MSFT, GOOGL, ORCL, NOW, PLTR, RKLB, ASTS, OKLO, LEU, IREN

เริ่ม Batch download จำนวน 5 ชุด

Batch 1/5: SPY, QQQ, RSP, IWM, SOXX, HYG, LQD, TLT, UUP, GLD, ^VIX, XLK
  SPY: OK — 2,915 rows
  QQQ: OK — 2,915 rows
  RSP: OK — 2,915 rows
  IWM: OK — 2,915 rows
  SOXX: OK — 2,915 rows
  HYG: OK — 2,915 rows
  LQD: OK — 2,915 rows
  TLT: OK — 2,915 rows
  UUP: OK — 2,915 rows
  GLD: OK — 2,915 rows
  ^VIX: OK — 2,916 rows
  XLK: OK — 2,915 rows

Batch 2/5: XLV, XLF, XLE, XLI, XLU, XLB, XLY, XLP, XLRE, TIP, IEF, SHY
  XLV: OK — 2,915 rows
  XLF: OK — 2,915 rows
  XLE: OK — 2,915 rows
  XLI: OK — 2,915 rows
  XLU: OK — 2,915 rows
  X

## Step 4 — FRED REST API Macro Engine


In [6]:
# ============================================================
# STEP 4 — FRED REST API MACRO DATA ENGINE
# ขั้นตอนที่ 4 — ระบบดึงข้อมูลเศรษฐกิจมหภาคจาก FRED REST API
# ============================================================

from pathlib import Path
import json, re, time
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import userdata

# ============================================================
# 1. CONFIGURATION (การตั้งค่า)
# ============================================================

# วันที่เริ่มต้นการดาวน์โหลดข้อมูล
START_DATE = "2015-01-01"
# วันที่สิ้นสุดการดาวน์โหลดข้อมูล (วันนี้ในโซนเวลา America/New_York)
END_DATE = pd.Timestamp.now(tz="America/New_York").strftime("%Y-%m-%d")
# เวลาที่รันสคริปต์นี้ (UTC)
RUN_TS = pd.Timestamp.now(tz="UTC")

# Fail fast after runtime restart.
if "PROJECT_ROOT" not in globals():
    raise RuntimeError("Run the Bootstrap cell before Step 4.")

ROOT = Path(PROJECT_ROOT)
# พาธสำหรับเก็บข้อมูลดิบเศรษฐกิจมหภาค
RAW = ROOT / "raw" / "macro"
# พาธสำหรับเก็บผลลัพธ์
OUT = ROOT / "outputs"
# พาธสำหรับเก็บไฟล์บันทึก (logs)
LOG = ROOT / "logs"
# สร้าง directories ถ้ายังไม่มี
for p in (RAW, OUT, LOG):
    p.mkdir(parents=True, exist_ok=True)

# รายการ Series ID ที่ต้องการดาวน์โหลดจาก FRED พร้อมการตั้งค่า:
# (เป็นข้อมูลที่จำเป็นหรือไม่, ประเภทวันหยุด (business/calendar), จำนวนวันที่ข้อมูลห่างจากปัจจุบันได้สูงสุด)
SERIES = {
    "DGS10": (True, "business", 3), # อัตราผลตอบแทนพันธบัตรสหรัฐ 10 ปี
    "DGS2": (True, "business", 3),  # อัตราผลตอบแทนพันธบัตรสหรัฐ 2 ปี
    "DFII10": (True, "business", 3), # อัตราผลตอบแทนพันธบัตร TIPS 10 ปี
    "T10Y2Y": (True, "business", 3), # ส่วนต่างอัตราผลตอบแทนพันธบัตร 10 ปี กับ 2 ปี
    # ICE/FRED may expose only a limited trailing history for this series.
    # Historical Credit fallback is handled by HYG/LQD in Step 6.
    "BAMLH0A0HYM2": (True, "business", 3), # High Yield OAS
    "SOFR": (True, "business", 3),  # Secured Overnight Financing Rate
    "VIXCLS": (True, "business", 3), # VIX Index
    "WALCL": (False, "calendar", 10), # Total Assets of Federal Reserve (สินทรัพย์รวมของ Fed)
    "WTREGEN": (False, "calendar", 10), # Repurchase Agreements (การซื้อคืนพันธบัตร)
    "RRPONTSYD": (False, "business", 3), # Overnight Reverse Repurchase Agreements (การขายคืนพันธบัตรชั่วคราว)
}

# ดึง FRED API Key จาก Colab Secrets
API_KEY = str(userdata.get("FRED_API_KEY") or "").strip()
# ตรวจสอบความถูกต้องของ API Key
if not re.fullmatch(r"[a-z0-9]{32}", API_KEY):
    raise ValueError(
        "FRED_API_KEY ไม่ถูกต้อง ให้ตรวจ Colab Secrets "
        "และเปิด Notebook access" # ตรวจสอบ FRED_API_KEY ใน Colab Secrets
    )

print("PROJECT_ROOT:", ROOT)
print("Download period:", START_DATE, "to", END_DATE)
print("Requested series:", len(SERIES))
print("FRED API key: AVAILABLE (hidden)") # แสดงว่า API Key พร้อมใช้งาน

# ตั้งค่าการ Retry สำหรับ Request ที่อาจล้มเหลวชั่วคราว
retry = Retry(
    total=3, # จำนวนครั้งที่พยายามใหม่
    connect=3, # จำนวนครั้งที่พยายามเชื่อมต่อใหม่
    read=3, # จำนวนครั้งที่พยายามอ่านข้อมูลใหม่
    status=3, # จำนวนครั้งที่พยายามใหม่เมื่อได้รับ HTTP status code บางอย่าง
    backoff_factor=1, # ตัวคูณเวลาหน่วงในการ retry
    status_forcelist=[429, 500, 502, 503, 504], # HTTP status codes ที่จะ retry
    allowed_methods=frozenset(["GET"]), # HTTP methods ที่อนุญาตให้ retry
    raise_on_status=False, # ไม่ยกเว้น error สำหรับ status codes ที่กำหนด
)
session = requests.Session() # สร้าง session สำหรับ requests
session.mount("https://", HTTPAdapter(max_retries=retry)) # mount adapter สำหรับ HTTPS
session.headers.update({
    "User-Agent": "IVMOS-FRED-Engine/1.0", # User-Agent สำหรับ request
    "Accept": "application/json", # ต้องการ response เป็น JSON
})

# FRED API endpoints
META_URL = "https://api.stlouisfed.org/fred/series" # URL สำหรับ metadata ของ series
OBS_URL = "https://api.stlouisfed.org/fred/series/observations" # URL สำหรับข้อมูล observations ของ series

# ============================================================
# 2. HELPER FUNCTIONS (ฟังก์ชันช่วยเหลือ)
# ============================================================

def api_get(url, params):
    """
    ส่ง HTTP GET request ไปยัง FRED API และจัดการข้อผิดพลาด
    """
    response = session.get(
        url,
        params={**params, "api_key": API_KEY, "file_type": "json"}, # เพิ่ม API key และระบุ file_type เป็น json
        timeout=(10, 60), # กำหนด timeout สำหรับการเชื่อมต่อและอ่านข้อมูล
    )
    payload = response.json()
    if response.status_code != 200: # ถ้า status code ไม่ใช่ 200 (OK)
        message = str(payload.get("error_message", payload)).replace(
            API_KEY, "[REDACTED]" # ซ่อน API Key จาก error message
        )
        raise RuntimeError(
            f"FRED API HTTP {response.status_code}: {message}" # ยกเว้น error
        )
    return payload

def business_age(start, end):
    """
    คำนวณจำนวนวันทำการระหว่างสองวันที่
    """
    a = np.datetime64(pd.Timestamp(start).date())
    b = np.datetime64(pd.Timestamp(end).date())
    return 0 if b <= a else int(np.busday_count(a, b))

# ============================================================
# 3. AUTHENTICATION TEST (ทดสอบการยืนยันตัวตน)
# ============================================================

print("\nTesting FRED API authentication...")
# ทดสอบการเชื่อมต่อ API ด้วยการดึง metadata ของ DGS10
test = api_get(META_URL, {"series_id": "DGS10"})
if not test.get("seriess"):
    raise RuntimeError("Authentication ผ่าน แต่ไม่พบ DGS10 metadata") # หากไม่พบ metadata ให้ยกเว้น error
print("Authentication test: PASS")

# ============================================================
# 4. FRED DOWNLOADS (การดาวน์โหลดข้อมูล FRED)
# ============================================================

frames = [] # สำหรับเก็บ DataFrame ของข้อมูลที่ดาวน์โหลดมา
metadata_rows = [] # สำหรับเก็บ metadata ของ series
fetch_log = [] # สำหรับบันทึกสถานะการดาวน์โหลด
successful = [] # รายชื่อ series ที่ดาวน์โหลดสำเร็จ
failed = [] # รายชื่อ series ที่ดาวน์โหลดไม่สำเร็จ

print("\nStarting FRED downloads...")

# วนลูปดาวน์โหลดข้อมูลสำหรับแต่ละ series ในรายการ SERIES
for i, series_id in enumerate(SERIES, 1):
    print(f"[{i}/{len(SERIES)}] {series_id}", flush=True)
    started = time.perf_counter()

    try:
        # ดึง metadata ของ series
        meta_payload = api_get(META_URL, {"series_id": series_id})
        meta_list = meta_payload.get("seriess", [])
        if not meta_list:
            raise ValueError("No metadata returned") # หากไม่มี metadata ให้ยกเว้น error

        m = meta_list[0]
        # เพิ่ม metadata เข้าไปในรายการ metadata_rows
        metadata_rows.append({
            "series_id": series_id,
            "title": m.get("title"),
            "frequency": m.get("frequency"),
            "frequency_short": m.get("frequency_short"),
            "units": m.get("units"),
            "units_short": m.get("units_short"),
            "seasonal_adjustment": m.get("seasonal_adjustment"),
            "observation_start": m.get("observation_start"),
            "observation_end": m.get("observation_end"),
            "last_updated": m.get("last_updated"),
            "required": SERIES[series_id][0],
            "source": "FRED",
            "download_timestamp": RUN_TS,
        })

        # ดึงข้อมูล observations ของ series
        obs_payload = api_get(
            OBS_URL,
            {
                "series_id": series_id,
                "observation_start": START_DATE,
                "observation_end": END_DATE,
                "sort_order": "asc",
                "limit": 100000,
            },
        )
        observations = obs_payload.get("observations", [])
        if not observations:
            raise ValueError("No observations returned") # หากไม่มี observations ให้ยกเว้น error

        # สร้าง DataFrame จากข้อมูล observations
        raw = pd.DataFrame(observations)
        df = pd.DataFrame({
            "series_id": series_id,
            "observation_date": pd.to_datetime(
                raw["date"], errors="coerce" # แปลงเป็น datetime, จัดการ error ด้วย coerce
            ),
            "value_raw": raw["value"].astype("string"),
            "value": pd.to_numeric(raw["value"], errors="coerce"), # แปลงเป็นตัวเลข, จัดการ error ด้วย coerce
            "realtime_start": pd.to_datetime(
                raw["realtime_start"], errors="coerce"
            ),
            "realtime_end": pd.to_datetime(
                raw["realtime_end"], errors="coerce"
            ),
        })

        # เพิ่มข้อมูล metadata ลงใน DataFrame
        df["title"] = m.get("title")
        df["frequency"] = m.get("frequency")
        df["frequency_short"] = m.get("frequency_short")
        df["units"] = m.get("units")
        df["units_short"] = m.get("units_short")
        df["source"] = "FRED"
        df["download_method"] = "FRED_REST_API"
        df["download_timestamp"] = RUN_TS
        df["status"] = np.where(
            df["value"].notna(), "VALID", "MISSING_VALUE" # กำหนด status เป็น VALID หรือ MISSING_VALUE
        )

        # กรองข้อมูลตามช่วงวันที่และเรียงลำดับ
        df = df.loc[
            df["observation_date"].notna()
            & df["observation_date"].between(
                pd.Timestamp(START_DATE),
                pd.Timestamp(END_DATE),
            )
        ].sort_values("observation_date").reset_index(drop=True)

        numeric = df.loc[df["value"].notna()] # กรองเฉพาะข้อมูลที่เป็นตัวเลข
        if numeric.empty:
            raise ValueError("No numeric observations") # หากไม่มีข้อมูลตัวเลขให้ยกเว้น error

        frames.append(df) # เพิ่ม DataFrame เข้าไปในรายการ frames
        successful.append(series_id) # เพิ่ม series ID ลงในรายการที่ดาวน์โหลดสำเร็จ
        fetch_log.append({
            "series_id": series_id,
            "status": "SUCCESS",
            "rows_total": len(df),
            "rows_numeric": len(numeric),
            "rows_missing": int(df["value"].isna().sum()),
            "latest_observation": str(
                numeric["observation_date"].max().date()
            ),
            "elapsed_seconds": round(
                time.perf_counter() - started, 4
            ),
        })
        print(f"  OK — {len(df):,} rows")

    except Exception as exc:
        failed.append(series_id) # เพิ่ม series ID ลงในรายการที่ดาวน์โหลดไม่สำเร็จ
        fetch_log.append({
            "series_id": series_id,
            "status": "FAIL",
            "error_type": type(exc).__name__,
            "error": str(exc).replace(API_KEY, "[REDACTED]"), # ซ่อน API Key จาก error message
        })
        print(f"  FAIL — {type(exc).__name__}: {exc}")

# รวม DataFrame ทั้งหมดที่ดาวน์โหลดมา
fred_raw = (
    pd.concat(frames, ignore_index=True) # รวม DataFrame
    .sort_values(["series_id", "observation_date"])
    .reset_index(drop=True)
    if frames else pd.DataFrame() # หากไม่มี frames ให้สร้าง DataFrame ว่าง
)
fred_metadata = pd.DataFrame(metadata_rows) # สร้าง DataFrame จาก metadata

# ============================================================
# 5. POST-DOWNLOAD PROCESSING & VALIDATION (การประมวลผลหลังดาวน์โหลดและตรวจสอบ)
# ============================================================

# แยก series ที่จำเป็นและไม่จำเป็น
required = {s for s, cfg in SERIES.items() if cfg[0]}
optional = set(SERIES) - required

# ตรวจสอบ series ที่หายไป
missing_required = sorted(required - set(successful))
missing_optional = sorted(optional - set(successful))

duplicates = []
future_series = []
# ตรวจสอบข้อมูลซ้ำและข้อมูลในอนาคต
if not fred_raw.empty:
    duplicates = sorted(
        fred_raw.loc[
            fred_raw.duplicated(
                ["series_id", "observation_date"], keep=False # ตรวจสอบข้อมูลซ้ำกันของ series_id และ observation_date
            ),
            "series_id",
        ].unique().tolist()
    )
    future_series = sorted(
        fred_raw.loc[
            fred_raw["observation_date"] > pd.Timestamp(END_DATE), # ตรวจสอบข้อมูลที่มี observation_date อยู่ในอนาคต
            "series_id",
        ].unique().tolist()
    )

# กำหนดวันที่ปัจจุบันในโซน America/New_York
today = pd.Timestamp.now(
    tz="America/New_York"
).tz_localize(None).normalize()

summary = []
stale_required = [] # รายการ series ที่จำเป็นแต่ข้อมูลไม่สดใหม่
stale_optional = [] # รายการ series ที่ไม่จำเป็นแต่ข้อมูลไม่สดใหม่

# ตรวจสอบความสดใหม่ของข้อมูลแต่ละ series
for series_id, (is_required, age_type, max_age) in SERIES.items():
    sdf = (
        fred_raw.loc[fred_raw["series_id"] == series_id]
        if not fred_raw.empty else pd.DataFrame()
    )
    mdf = fred_metadata.loc[
        fred_metadata["series_id"] == series_id
    ]

    if sdf.empty:
        summary.append({
            "series_id": series_id,
            "required": is_required,
            "frequency": None,
            "units": None,
            "rows_total": 0,
            "rows_numeric": 0,
            "latest_observation": None,
            "calendar_age_days": None,
            "business_age_days": None,
            "status": "MISSING",
        })
        continue

    numeric = sdf.loc[sdf["value"].notna()] # กรองเฉพาะข้อมูลที่เป็นตัวเลข
    latest = numeric["observation_date"].max() # วันที่ observations ล่าสุด
    calendar_days = int((today - latest.normalize()).days) # จำนวนวันตามปฏิทินที่ข้อมูลห่างจากปัจจุบัน
    business_days = business_age(latest, today) # จำนวนวันทำการที่ข้อมูลห่างจากปัจจุบัน
    tested_age = (
        business_days if age_type == "business" else calendar_days # อายุข้อมูลที่ใช้ทดสอบตามประเภท
    )
    status = "STALE" if tested_age > max_age else "VALID" # กำหนด status เป็น STALE หรือ VALID

    if status == "STALE":
        item = {
            "series_id": series_id,
            "latest_observation": str(latest.date()),
            "age_type": age_type,
            "tested_age": tested_age,
            "max_age": max_age,
        }
        (stale_required if is_required else stale_optional).append(item) # เพิ่มข้อมูลลงใน stale_required หรือ stale_optional

    summary.append({
        "series_id": series_id,
        "required": is_required,
        "frequency": mdf["frequency"].iloc[0]
            if not mdf.empty else None,
        "units": mdf["units"].iloc[0]
            if not mdf.empty else None,
        "rows_total": len(sdf),
        "rows_numeric": len(numeric),
        "latest_observation": str(latest.date()),
        "calendar_age_days": calendar_days,
        "business_age_days": business_days,
        "status": status,
    })

macro_summary = pd.DataFrame(summary) # สร้าง DataFrame สรุปข้อมูล Macro

# ============================================================
# 6. QUALITY ASSESSMENT (การประเมินคุณภาพ)
# ============================================================

# ตรวจสอบข้อผิดพลาดที่สำคัญ (Critical errors)
critical = any([
    missing_required,
    stale_required,
    duplicates,
    future_series,
])
# ตรวจสอบข้อผิดพลาดที่ไม่สำคัญ (Partial errors)
partial = any([
    missing_optional,
    stale_optional,
])
# กำหนดสถานะโดยรวมของขั้นตอน
overall_status = (
    "FAIL" if critical else
    "PARTIAL" if partial else
    "PASS"
)

# สร้าง dictionary สำหรับรายงานคุณภาพ
quality = {
    "step": 4,
    "overall_status": overall_status,
    "download_timestamp": str(RUN_TS),
    "series_successful": successful,
    "series_failed": failed,
    "missing_required": missing_required,
    "missing_optional": missing_optional,
    "stale_required": stale_required,
    "stale_optional": stale_optional,
    "duplicate_series_dates": duplicates,
    "future_observation_series": future_series,
    "total_rows": len(fred_raw),
    "numeric_rows": int(fred_raw["value"].notna().sum())
        if not fred_raw.empty else 0,
    "missing_value_rows": int(fred_raw["value"].isna().sum())
        if not fred_raw.empty else 0,
    "api_key_written_to_output": False,
    "forward_fill": False,
    "backward_fill": False,
    "interpolation": False,
}

# ============================================================
# 7. EXPORT (ส่งออกไฟล์)
# ============================================================

# กำหนดพาธสำหรับไฟล์ที่จะ export
paths = {
    "raw_parquet": RAW / "fred_raw.parquet",
    "raw_csv": RAW / "fred_raw.csv.gz",
    "metadata_csv": RAW / "fred_series_metadata.csv",
    "summary_csv": OUT / "macro_summary.csv",
    "quality_json": OUT / "macro_data_quality.json",
    "fetch_log_json": LOG / "fred_fetch_log.json",
}

# Export ข้อมูลเป็นไฟล์ Parquet และ CSV
fred_raw.to_parquet(
    paths["raw_parquet"], index=False, compression="snappy"
)
fred_raw.to_csv(
    paths["raw_csv"], index=False, compression="gzip"
)
fred_metadata.to_csv(paths["metadata_csv"], index=False)
macro_summary.to_csv(paths["summary_csv"], index=False)

# Export log การดึงข้อมูลเป็น JSON
with open(paths["fetch_log_json"], "w", encoding="utf-8") as f:
    json.dump(fetch_log, f, ensure_ascii=False, indent=2, default=str)

quality["files_created"] = [str(p) for p in paths.values()] # เพิ่มรายชื่อไฟล์ที่สร้างลงในรายงานคุณภาพ
# Export รายงานคุณภาพเป็น JSON
with open(paths["quality_json"], "w", encoding="utf-8") as f:
    json.dump(quality, f, ensure_ascii=False, indent=2, default=str)

# อัปเดตสถานะในรายงานรวมของ Colab
if "report" not in globals():
    report = {"status": "PASS", "errors": [], "warnings": []}
report["status"] = overall_status

# ============================================================
# 8. EXECUTION REPORT (รายงานการรัน)
# ============================================================

print("\n" + "=" * 70)
print("STEP 4 EXECUTION REPORT: FRED REST API")
print("=" * 70)
print("Overall Status:", overall_status)
print(f"Series Successful: {len(successful)}/{len(SERIES)}")
print("Series Failed:", failed or "None")
print("Missing Required:", missing_required or "None")
print("Missing Optional:", missing_optional or "None")
print("Stale Required:", stale_required or "None")
print("Stale Optional:", stale_optional or "None")
print("Duplicate Series-Date:", duplicates or "None")
print("Future Observations:", future_series or "None")
print("Total Rows:", f"{len(fred_raw):,}")
print("\nSeries Summary:")
print(macro_summary.to_string(index=False))
print("\nFiles Created:")
for p in quality["files_created"]:
    print(" ", p)
print("=" * 70)


PROJECT_ROOT: /content/drive/MyDrive/IVMOS
Download period: 2015-01-01 to 2026-08-07
Requested series: 10
FRED API key: AVAILABLE (hidden)

Testing FRED API authentication...
Authentication test: PASS

Starting FRED downloads...
[1/10] DGS10
  OK — 3,025 rows
[2/10] DGS2
  OK — 3,025 rows
[3/10] DFII10
  OK — 3,025 rows
[4/10] T10Y2Y
  OK — 3,026 rows
[5/10] BAMLH0A0HYM2
  OK — 795 rows
[6/10] SOFR
  OK — 2,177 rows
[7/10] VIXCLS
  OK — 3,025 rows
[8/10] WALCL
  OK — 605 rows
[9/10] WTREGEN
  OK — 605 rows
[10/10] RRPONTSYD
  OK — 3,026 rows

STEP 4 EXECUTION REPORT: FRED REST API
Overall Status: PASS
Series Successful: 10/10
Series Failed: None
Missing Required: None
Missing Optional: None
Stale Required: None
Stale Optional: None
Duplicate Series-Date: None
Future Observations: None
Total Rows: 22,334

Series Summary:
   series_id  required                frequency                    units  rows_total  rows_numeric latest_observation  calendar_age_days  business_age_days status
     

## Step 5 — Feature Engine


In [7]:
# ============================================================
# IVMOS STEP 5 — FEATURE ENGINE v2
# Vectorized macro alignment / explicit progress / data only
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Any
from contextlib import contextmanager
import json
import math
import os
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

DEFAULT_PROJECT_ROOT = "/content/drive/MyDrive/IVMOS"
PROJECT_ROOT = Path(
    os.environ.get(
        "IVMOS_PROJECT_ROOT",
        str(globals().get("PROJECT_ROOT", DEFAULT_PROJECT_ROOT)),
    )
)

RAW_PRICE_PATH = PROJECT_ROOT / "raw" / "prices" / "prices_daily.parquet"
RAW_MACRO_PATH = PROJECT_ROOT / "raw" / "macro" / "fred_raw.parquet"
MACRO_METADATA_PATH = PROJECT_ROOT / "raw" / "macro" / "fred_series_metadata.csv"

PROCESSED_DIR = PROJECT_ROOT / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"
TEST_DIR = PROJECT_ROOT / "tests"

for directory in [PROCESSED_DIR, OUTPUT_DIR, LOG_DIR, TEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now(tz="UTC")
MIN_BASKET_COVERAGE = 0.60
EXPORT_FULL_CSV_GZ = False  # Parquet is the full feature store; latest row is exported to CSV.

REQUIRED_PRICE_TICKERS = {
    "SPY", "QQQ", "RSP", "IWM", "SOXX", "HYG", "LQD", "TLT",
    "UUP", "GLD", "^VIX", "XLK", "XLV", "XLF", "XLE", "XLI", "XLU",
}

REQUIRED_MACRO_SERIES = {
    "DGS10", "DGS2", "DFII10", "T10Y2Y", "BAMLH0A0HYM2", "SOFR", "VIXCLS",
}

RELATIVE_PAIRS = {
    "RSP_SPY": ("RSP", "SPY"),
    "IWM_SPY": ("IWM", "SPY"),
    "QQQ_SPY": ("QQQ", "SPY"),
    "SOXX_SPY": ("SOXX", "SPY"),
    "SOXX_QQQ": ("SOXX", "QQQ"),
    "SMH_SOXX": ("SMH", "SOXX"),
    "XLK_SPY": ("XLK", "SPY"),
    "XLV_SPY": ("XLV", "SPY"),
    "XLF_SPY": ("XLF", "SPY"),
    "XLE_SPY": ("XLE", "SPY"),
    "XLI_SPY": ("XLI", "SPY"),
    "XLU_SPY": ("XLU", "SPY"),
    "XLB_SPY": ("XLB", "SPY"),
    "XLY_SPY": ("XLY", "SPY"),
    "XLP_SPY": ("XLP", "SPY"),
    "XLRE_SPY": ("XLRE", "SPY"),
    "HYG_LQD": ("HYG", "LQD"),
    "GLD_UUP": ("GLD", "UUP"),
    "GLD_TLT": ("GLD", "TLT"),
    "GLD_SPY": ("GLD", "SPY"),
    "USO_SPY": ("USO", "SPY"),
    "CPER_SPY": ("CPER", "SPY"),
}

AI_BASKETS = {
    "AI_COMPUTE": ["NVDA", "TSM", "ASML", "MU", "AVGO"],
    "AI_CONNECTIVITY": ["ANET", "CRDO", "MRVL", "COHR", "LITE"],
    "AI_POWER_GRID": ["ETN", "VRT", "GEV", "HUBB", "APH"],
    "AI_SOFTWARE": ["MSFT", "GOOGL", "ORCL", "NOW", "PLTR"],
    "AI_SPECULATIVE": ["RKLB", "ASTS", "OKLO", "LEU", "IREN"],
}

FREQUENCY_FALLBACK = {
    "DGS10": "D",
    "DGS2": "D",
    "DFII10": "D",
    "T10Y2Y": "D",
    "BAMLH0A0HYM2": "D",
    "SOFR": "D",
    "VIXCLS": "D",
    "WALCL": "W",
    "WTREGEN": "W",
    "RRPONTSYD": "D",
}

print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)
print("Price input:", RAW_PRICE_PATH, flush=True)
print("Macro input:", RAW_MACRO_PATH, flush=True)
print("Run timestamp:", RUN_TIMESTAMP, flush=True)


# ============================================================
# 2. HELPERS
# ============================================================

STAGE_TIMINGS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    started = time.perf_counter()
    print(f"\n{name}...", flush=True)
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        STAGE_TIMINGS[name] = round(elapsed, 3)
        print(f"{name}: completed in {elapsed:.2f}s", flush=True)


def clean_json_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(k): clean_json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json_value(v) for v in value]
    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_json(path: Path, payload: Any) -> None:
    with open(path, "w", encoding="utf-8") as file:
        json.dump(clean_json_value(payload), file, ensure_ascii=False, indent=2)


def safe_pct_change(series: pd.Series, periods: int) -> pd.Series:
    return series.pct_change(periods=periods, fill_method=None)


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)


def rolling_zscore(series: pd.Series, window: int, min_periods: int) -> pd.Series:
    rolling = series.rolling(window=window, min_periods=min_periods)
    mean = rolling.mean()
    std = rolling.std(ddof=0).replace(0, np.nan)
    return (series - mean) / std


def rolling_percentile(series: pd.Series, window: int, min_periods: int) -> pd.Series:
    # pandas rolling.rank is implemented in C and is much faster than Python rolling.apply.
    return series.rolling(window=window, min_periods=min_periods).rank(pct=True)


def calculate_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - 100 / (1 + rs)
    rsi = rsi.where(~((avg_loss == 0) & (avg_gain > 0)), 100.0)
    rsi = rsi.where(~((avg_loss == 0) & (avg_gain == 0)), 50.0)
    return rsi


def flatten_pivot_columns(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    flattened = frame.copy()
    flattened.columns = [
        f"{prefix}__{str(entity)}__{str(feature)}"
        for feature, entity in flattened.columns
    ]
    return flattened


def count_infinite(frame: pd.DataFrame) -> int:
    numeric = frame.select_dtypes(include=[np.number])
    if numeric.empty:
        return 0
    return int(np.isinf(numeric.to_numpy(dtype=float, copy=False)).sum())


def normalized_index(return_series: pd.Series, base: float = 100.0) -> pd.Series:
    result = (1.0 + return_series).cumprod()
    first_valid = result.first_valid_index()
    if first_valid is None:
        return result
    return result / result.loc[first_valid] * base


def frequency_short_map() -> dict[str, str]:
    mapping = dict(FREQUENCY_FALLBACK)
    if MACRO_METADATA_PATH.exists():
        metadata = pd.read_csv(MACRO_METADATA_PATH)
        if {"series_id", "frequency_short"}.issubset(metadata.columns):
            for row in metadata[["series_id", "frequency_short"]].dropna().itertuples(index=False):
                mapping[str(row.series_id)] = str(row.frequency_short).upper()
    return mapping


def staleness_limit_days(frequency_short: str) -> tuple[str, int]:
    frequency_short = str(frequency_short).upper()
    if frequency_short.startswith("D"):
        return "business", 3
    if frequency_short.startswith("W"):
        return "calendar", 10
    if frequency_short.startswith("M"):
        return "calendar", 45
    if frequency_short.startswith("Q"):
        return "calendar", 120
    return "calendar", 10


# ============================================================
# 3. LOAD AND VALIDATE INPUTS
# ============================================================

with stage("Loading input data"):
    if not RAW_PRICE_PATH.exists():
        raise FileNotFoundError(f"Price input not found: {RAW_PRICE_PATH}")
    if not RAW_MACRO_PATH.exists():
        raise FileNotFoundError(f"Macro input not found: {RAW_MACRO_PATH}")

    price_raw = pd.read_parquet(RAW_PRICE_PATH)
    macro_raw = pd.read_parquet(RAW_MACRO_PATH)

    price_required_columns = {"date", "ticker", "open", "high", "low", "close", "volume"}
    macro_required_columns = {"observation_date", "series_id", "value"}

    missing_price_columns = sorted(price_required_columns - set(price_raw.columns))
    missing_macro_columns = sorted(macro_required_columns - set(macro_raw.columns))
    if missing_price_columns:
        raise ValueError(f"Price input missing columns: {missing_price_columns}")
    if missing_macro_columns:
        raise ValueError(f"Macro input missing columns: {missing_macro_columns}")

    price_raw["date"] = pd.to_datetime(price_raw["date"], errors="coerce").dt.tz_localize(None)
    macro_raw["observation_date"] = pd.to_datetime(
        macro_raw["observation_date"], errors="coerce"
    ).dt.tz_localize(None)

    price_raw["ticker"] = price_raw["ticker"].astype(str)
    macro_raw["series_id"] = macro_raw["series_id"].astype(str)
    macro_raw["value"] = pd.to_numeric(macro_raw["value"], errors="coerce")

    price_duplicate_count_before = int(price_raw.duplicated(["date", "ticker"]).sum())
    macro_duplicate_count_before = int(
        macro_raw.duplicated(["observation_date", "series_id"]).sum()
    )

    price_raw = (
        price_raw.dropna(subset=["date", "ticker"])
        .drop_duplicates(["date", "ticker"], keep="last")
        .sort_values(["ticker", "date"])
        .reset_index(drop=True)
    )
    macro_raw = (
        macro_raw.dropna(subset=["observation_date", "series_id"])
        .drop_duplicates(["observation_date", "series_id"], keep="last")
        .sort_values(["series_id", "observation_date"])
        .reset_index(drop=True)
    )

    available_price_tickers = set(price_raw["ticker"].unique())
    available_macro_series = set(macro_raw["series_id"].unique())
    missing_required_price = sorted(REQUIRED_PRICE_TICKERS - available_price_tickers)
    missing_required_macro = sorted(REQUIRED_MACRO_SERIES - available_macro_series)

    print("Price rows loaded:", f"{len(price_raw):,}", flush=True)
    print("Macro rows loaded:", f"{len(macro_raw):,}", flush=True)
    print("Price tickers:", len(available_price_tickers), flush=True)
    print("Macro series:", len(available_macro_series), flush=True)


# ============================================================
# 4. PRICE FEATURES
# ============================================================


def calculate_price_features(ticker_frame: pd.DataFrame) -> pd.DataFrame:
    ticker_frame = ticker_frame.sort_values("date").copy().set_index("date")
    ticker = str(ticker_frame["ticker"].iloc[0])

    close = pd.to_numeric(ticker_frame["close"], errors="coerce")
    high = pd.to_numeric(ticker_frame["high"], errors="coerce")
    low = pd.to_numeric(ticker_frame["low"], errors="coerce")
    volume = pd.to_numeric(ticker_frame["volume"], errors="coerce")

    output = pd.DataFrame(index=ticker_frame.index)
    output["ticker"] = ticker
    for column in ["open", "high", "low", "close", "volume"]:
        output[column] = pd.to_numeric(ticker_frame[column], errors="coerce")

    for period in [1, 5, 20, 60, 126, 252]:
        output[f"return_{period}d"] = safe_pct_change(close, period)

    for span in [20, 50, 100, 200]:
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        output[f"ema_{span}"] = ema
        output[f"distance_ema{span}_pct"] = safe_divide(close, ema) - 1

    high_252 = close.rolling(252, min_periods=126).max()
    low_252 = close.rolling(252, min_periods=126).min()
    output["high_252d"] = high_252
    output["low_252d"] = low_252
    output["distance_high_252d_pct"] = safe_divide(close, high_252) - 1
    output["distance_low_252d_pct"] = safe_divide(close, low_252) - 1

    previous_close = close.shift(1)
    true_range = pd.concat(
        [(high - low), (high - previous_close).abs(), (low - previous_close).abs()],
        axis=1,
    ).max(axis=1)
    output["atr_20"] = true_range.rolling(20, min_periods=20).mean()
    output["atr_20_pct"] = safe_divide(output["atr_20"], close)

    daily_return = safe_pct_change(close, 1)
    output["volatility_20d_annualized"] = daily_return.rolling(20, min_periods=20).std() * np.sqrt(252)
    output["volatility_60d_annualized"] = daily_return.rolling(60, min_periods=40).std() * np.sqrt(252)

    output["volume_ma20"] = volume.rolling(20, min_periods=10).mean()
    output["volume_ratio_20d"] = safe_divide(volume, output["volume_ma20"])
    output["rsi_14"] = calculate_rsi(close, 14)
    output["roc_20d"] = safe_pct_change(close, 20)
    output["roc_60d"] = safe_pct_change(close, 60)

    ema_12 = close.ewm(span=12, adjust=False, min_periods=12).mean()
    ema_26 = close.ewm(span=26, adjust=False, min_periods=26).mean()
    output["macd"] = ema_12 - ema_26
    output["macd_signal"] = output["macd"].ewm(span=9, adjust=False, min_periods=9).mean()
    output["macd_histogram"] = output["macd"] - output["macd_signal"]

    for window in [20, 60, 252]:
        rolling_peak = close.rolling(window, min_periods=max(2, window // 2)).max()
        output[f"drawdown_{window}d"] = safe_divide(close, rolling_peak) - 1
    output["drawdown_from_all_time_high"] = safe_divide(close, close.cummax()) - 1

    output["return_zscore_252d"] = rolling_zscore(daily_return, 252, 126)
    output["history_days"] = np.arange(1, len(output) + 1)
    output["feature_ready_20d"] = output["history_days"] >= 20
    output["feature_ready_50d"] = output["history_days"] >= 50
    output["feature_ready_200d"] = output["history_days"] >= 200
    output["feature_ready_252d"] = output["history_days"] >= 252

    output.index.name = "date"
    return output.reset_index()


with stage("Calculating price features"):
    price_feature_frames: list[pd.DataFrame] = []
    price_groups = list(price_raw.groupby("ticker", sort=True))
    for number, (ticker, ticker_frame) in enumerate(price_groups, start=1):
        price_feature_frames.append(calculate_price_features(ticker_frame))
        if number % 10 == 0 or number == len(price_groups):
            print(f"  Processed {number}/{len(price_groups)} tickers", flush=True)

    price_features_long = pd.concat(price_feature_frames, ignore_index=True)
    price_features_long = price_features_long.replace([np.inf, -np.inf], np.nan)

    price_value_columns = [
        column
        for column in price_features_long.columns
        if column not in {"date", "ticker"}
        and (
            pd.api.types.is_numeric_dtype(price_features_long[column])
            or pd.api.types.is_bool_dtype(price_features_long[column])
        )
    ]
    price_features_wide = price_features_long.pivot(
        index="date", columns="ticker", values=price_value_columns
    )
    price_features_wide = flatten_pivot_columns(price_features_wide, "price")
    price_features_wide = price_features_wide.sort_index()

    close_wide = price_raw.pivot(index="date", columns="ticker", values="close").sort_index()
    trading_dates = pd.DatetimeIndex(close_wide.index.unique()).sort_values()


# ============================================================
# 5. RELATIVE FEATURES
# ============================================================

with stage("Calculating relative features"):
    relative_column_data: dict[str, pd.Series] = {}
    relative_pair_log: list[dict[str, Any]] = []

    for pair_name, (asset, benchmark) in RELATIVE_PAIRS.items():
        if asset not in close_wide.columns or benchmark not in close_wide.columns:
            relative_pair_log.append({
                "pair": pair_name,
                "asset": asset,
                "benchmark": benchmark,
                "status": "MISSING_INPUT",
            })
            continue

        asset_close = close_wide[asset]
        benchmark_close = close_wide[benchmark]
        ratio = safe_divide(asset_close, benchmark_close)
        first_valid = ratio.first_valid_index()
        normalized_ratio = ratio.copy()
        if first_valid is not None and pd.notna(ratio.loc[first_valid]) and ratio.loc[first_valid] != 0:
            normalized_ratio = ratio / ratio.loc[first_valid] * 100

        prefix = f"relative__{pair_name}"
        relative_column_data[f"{prefix}__ratio"] = ratio
        relative_column_data[f"{prefix}__ratio_index_100"] = normalized_ratio
        relative_column_data[f"{prefix}__ratio_ema20"] = ratio.ewm(
            span=20, adjust=False, min_periods=20
        ).mean()
        relative_column_data[f"{prefix}__ratio_ema50"] = ratio.ewm(
            span=50, adjust=False, min_periods=50
        ).mean()
        relative_column_data[f"{prefix}__ratio_roc20"] = safe_pct_change(ratio, 20)

        for period in [1, 5, 20, 60]:
            asset_return = safe_pct_change(asset_close, period)
            benchmark_return = safe_pct_change(benchmark_close, period)
            relative_column_data[f"{prefix}__relative_return_{period}d"] = (
                asset_return - benchmark_return
            )

        relative_pair_log.append({
            "pair": pair_name,
            "asset": asset,
            "benchmark": benchmark,
            "status": "CREATED",
        })

    relative_features_wide = pd.DataFrame(relative_column_data, index=trading_dates)
    relative_features_wide = relative_features_wide.replace([np.inf, -np.inf], np.nan)
    print(
        f"  Created {sum(item['status'] == 'CREATED' for item in relative_pair_log)}/"
        f"{len(relative_pair_log)} relative pairs",
        flush=True,
    )


# ============================================================
# 6. MACRO NATIVE-FREQUENCY FEATURES
# ============================================================


def calculate_macro_native_features(series_frame: pd.DataFrame, frequency_short: str) -> pd.DataFrame:
    series_frame = series_frame.sort_values("observation_date").copy()
    series_id = str(series_frame["series_id"].iloc[0])
    series_frame = series_frame.loc[series_frame["value"].notna()].copy()
    series_frame = series_frame.drop_duplicates("observation_date", keep="last")
    series_frame = series_frame.set_index("observation_date")
    value = series_frame["value"].astype(float)

    output = pd.DataFrame(index=value.index)
    output["series_id"] = series_id
    output["frequency_short"] = frequency_short
    output["value"] = value

    for period in [1, 5, 20, 60]:
        output[f"diff_{period}obs"] = value.diff(period)
        output[f"pct_change_{period}obs"] = safe_pct_change(value, period)

    output["ema_20obs"] = value.ewm(span=20, adjust=False, min_periods=20).mean()
    output["ema_50obs"] = value.ewm(span=50, adjust=False, min_periods=50).mean()
    output["distance_ema20"] = value - output["ema_20obs"]
    output["distance_ema50"] = value - output["ema_50obs"]
    output["value_zscore_252obs"] = rolling_zscore(value, 252, 126)
    output["value_percentile_252obs"] = rolling_percentile(value, 252, 126)

    periods_per_year = 52 if str(frequency_short).upper().startswith("W") else 252
    output["yoy_diff"] = value.diff(periods_per_year)
    output["yoy_pct_change"] = safe_pct_change(value, periods_per_year)
    output["history_observations"] = np.arange(1, len(output) + 1)

    output.index.name = "observation_date"
    return output.reset_index()


with stage("Calculating macro native-frequency features"):
    macro_frequency_map = frequency_short_map()
    macro_native_frames: list[pd.DataFrame] = []
    macro_groups = list(macro_raw.groupby("series_id", sort=True))

    for number, (series_id, series_frame) in enumerate(macro_groups, start=1):
        frequency_short = macro_frequency_map.get(str(series_id), "D")
        numeric_rows = int(series_frame["value"].notna().sum())
        if numeric_rows > 0:
            macro_native_frames.append(
                calculate_macro_native_features(series_frame, frequency_short)
            )
        print(
            f"  Processed {number}/{len(macro_groups)}: {series_id} "
            f"({numeric_rows:,} numeric rows)",
            flush=True,
        )

    macro_features_native_long = (
        pd.concat(macro_native_frames, ignore_index=True)
        if macro_native_frames
        else pd.DataFrame()
    )
    macro_features_native_long = macro_features_native_long.replace(
        [np.inf, -np.inf], np.nan
    )
# ============================================================
# 7. MACRO ALIGNMENT — ONE WIDE REINDEX/FORWARD AS-OF PASS
# ============================================================

with stage("Aligning macro features to trading dates"):
    macro_aligned_wide = pd.DataFrame(index=trading_dates)
    macro_alignment_log: list[dict[str, Any]] = []

    if not macro_features_native_long.empty:
        macro_value_columns = [
            column
            for column in macro_features_native_long.columns
            if column not in {"observation_date", "series_id", "frequency_short"}
            and pd.api.types.is_numeric_dtype(macro_features_native_long[column])
        ]

        macro_native_wide = macro_features_native_long.pivot(
            index="observation_date",
            columns="series_id",
            values=macro_value_columns,
        )
        macro_native_wide = flatten_pivot_columns(macro_native_wide, "macro")
        macro_native_wide = macro_native_wide.sort_index()

        union_index = macro_native_wide.index.union(trading_dates).sort_values()
        macro_aligned_wide = (
            macro_native_wide.reindex(union_index).ffill().reindex(trading_dates)
        )

        # Add provenance and staleness per series. This loop is only 10 series;
        # it does not merge each series against each ticker.
        for series_id, series_frame in macro_raw.groupby("series_id", sort=True):
            valid_dates = pd.DatetimeIndex(
                series_frame.loc[series_frame["value"].notna(), "observation_date"]
                .drop_duplicates()
                .sort_values()
            )
            if valid_dates.empty:
                continue

            source_dates_native = pd.Series(valid_dates, index=valid_dates)
            source_dates_aligned = (
                source_dates_native.reindex(valid_dates.union(trading_dates).sort_values())
                .ffill()
                .reindex(trading_dates)
            )

            target_series = pd.Series(trading_dates, index=trading_dates)
            calendar_age = (target_series - source_dates_aligned).dt.days.astype("float64")

            business_age = pd.Series(np.nan, index=trading_dates, dtype="float64")
            valid_mask = source_dates_aligned.notna()
            if valid_mask.any():
                start_days = source_dates_aligned.loc[valid_mask].to_numpy(dtype="datetime64[D]")
                end_days = trading_dates[valid_mask.to_numpy()].to_numpy(dtype="datetime64[D]")
                business_age.loc[valid_mask] = np.busday_count(start_days, end_days)

            frequency_short = macro_frequency_map.get(str(series_id), "D")
            age_type, maximum_age = staleness_limit_days(frequency_short)
            age_for_test = business_age if age_type == "business" else calendar_age
            stale_mask = age_for_test > maximum_age

            series_feature_columns = [
                column
                for column in macro_aligned_wide.columns
                if column.startswith(f"macro__{series_id}__")
            ]
            if series_feature_columns:
                macro_aligned_wide.loc[stale_mask.fillna(True), series_feature_columns] = np.nan

            macro_aligned_wide[f"macro__{series_id}__source_observation_date"] = (
                source_dates_aligned
            )
            macro_aligned_wide[f"macro__{series_id}__calendar_age_days"] = calendar_age
            macro_aligned_wide[f"macro__{series_id}__business_age_days"] = business_age
            macro_aligned_wide[f"macro__{series_id}__is_stale"] = stale_mask.astype("boolean")

            macro_alignment_log.append({
                "series_id": str(series_id),
                "frequency_short": frequency_short,
                "age_type": age_type,
                "maximum_age": maximum_age,
                "latest_source_observation": str(valid_dates.max().date()),
                "aligned_rows": len(trading_dates),
            })

    print(
        f"  Macro aligned shape: {macro_aligned_wide.shape[0]:,} rows x "
        f"{macro_aligned_wide.shape[1]:,} columns",
        flush=True,
    )
# ============================================================
# 8. AI BASKET FEATURES
# ============================================================

with stage("Calculating AI basket features"):
    basket_column_data: dict[str, pd.Series | int | bool] = {}
    close_returns_1d = close_wide.pct_change(fill_method=None)
    close_returns_20d = close_wide.pct_change(20, fill_method=None)
    close_ema20 = close_wide.ewm(span=20, adjust=False, min_periods=20).mean()
    close_ema50 = close_wide.ewm(span=50, adjust=False, min_periods=50).mean()
    spy_return_20d = close_returns_20d.get("SPY", pd.Series(index=trading_dates, dtype=float))
    qqq_return_20d = close_returns_20d.get("QQQ", pd.Series(index=trading_dates, dtype=float))
    basket_log: list[dict[str, Any]] = []

    for basket_name, configured_members in AI_BASKETS.items():
        members = [member for member in configured_members if member in close_wide.columns]
        minimum_members = max(1, math.ceil(len(configured_members) * MIN_BASKET_COVERAGE))
        prefix = f"basket__{basket_name}"

        if not members:
            basket_log.append({
                "basket": basket_name,
                "status": "NO_MEMBERS_AVAILABLE",
                "available_members": [],
            })
            continue

        member_returns_1d = close_returns_1d[members]
        member_returns_20d = close_returns_20d[members]
        member_close = close_wide[members]
        available_count = member_returns_1d.notna().sum(axis=1)
        sufficient = available_count >= minimum_members

        mean_return_1d = member_returns_1d.mean(axis=1, skipna=True).where(sufficient)
        median_return_1d = member_returns_1d.median(axis=1, skipna=True).where(sufficient)
        mean_index = normalized_index(mean_return_1d)
        median_index = normalized_index(median_return_1d)

        positive_denominator = member_returns_1d.notna().sum(axis=1).replace(0, np.nan)
        percent_positive = member_returns_1d.gt(0).sum(axis=1) / positive_denominator

        ema20_denominator = member_close.notna().sum(axis=1).replace(0, np.nan)
        percent_above_ema20 = member_close.gt(close_ema20[members]).sum(axis=1) / ema20_denominator
        percent_above_ema50 = member_close.gt(close_ema50[members]).sum(axis=1) / ema20_denominator

        outperform_denominator = member_returns_20d.notna().sum(axis=1).replace(0, np.nan)
        percent_outperform_spy20 = member_returns_20d.gt(spy_return_20d, axis=0).sum(axis=1) / outperform_denominator

        basket_column_data[f"{prefix}__mean_return_1d"] = mean_return_1d
        basket_column_data[f"{prefix}__median_return_1d"] = median_return_1d
        basket_column_data[f"{prefix}__mean_index_100"] = mean_index
        basket_column_data[f"{prefix}__median_index_100"] = median_index
        basket_column_data[f"{prefix}__return_5d"] = safe_pct_change(mean_index, 5)
        basket_column_data[f"{prefix}__return_20d"] = safe_pct_change(mean_index, 20)
        basket_column_data[f"{prefix}__return_60d"] = safe_pct_change(mean_index, 60)
        basket_column_data[f"{prefix}__relative_spy_20d"] = safe_pct_change(mean_index, 20) - spy_return_20d
        basket_column_data[f"{prefix}__relative_qqq_20d"] = safe_pct_change(mean_index, 20) - qqq_return_20d
        basket_column_data[f"{prefix}__member_count_available"] = available_count
        basket_column_data[f"{prefix}__member_count_total"] = len(configured_members)
        basket_column_data[f"{prefix}__percent_members_positive"] = percent_positive.where(sufficient)
        basket_column_data[f"{prefix}__percent_members_above_ema20"] = percent_above_ema20.where(sufficient)
        basket_column_data[f"{prefix}__percent_members_above_ema50"] = percent_above_ema50.where(sufficient)
        basket_column_data[f"{prefix}__percent_members_outperform_spy_20d"] = percent_outperform_spy20.where(sufficient)
        basket_column_data[f"{prefix}__cross_sectional_dispersion_20d"] = member_returns_20d.std(axis=1, ddof=0).where(sufficient)
        basket_column_data[f"{prefix}__insufficient_members"] = ~sufficient

        basket_log.append({
            "basket": basket_name,
            "status": "CREATED",
            "configured_members": configured_members,
            "available_members": members,
            "minimum_members": minimum_members,
        })
        print(
            f"  {basket_name}: {len(members)}/{len(configured_members)} members available",
            flush=True,
        )

    basket_features_wide = pd.DataFrame(basket_column_data, index=trading_dates)
    basket_features_wide = basket_features_wide.replace([np.inf, -np.inf], np.nan)


# ============================================================
# 9. COMBINE FEATURE STORE
# ============================================================

with stage("Combining feature store"):
    features_wide = pd.concat(
        [
            price_features_wide.reindex(trading_dates),
            relative_features_wide.reindex(trading_dates),
            macro_aligned_wide.reindex(trading_dates),
            basket_features_wide.reindex(trading_dates),
        ],
        axis=1,
    )
    features_wide = features_wide.loc[:, ~features_wide.columns.duplicated()].sort_index()
    features_wide.index.name = "date"
    numeric_feature_columns = features_wide.select_dtypes(include=[np.number]).columns
    features_wide.loc[:, numeric_feature_columns] = features_wide.loc[:, numeric_feature_columns].replace(
        [np.inf, -np.inf], np.nan
    )
# ============================================================
# 10. FEATURE METADATA AND VALIDATION
# ============================================================

with stage("Validating feature store"):
    feature_metadata_rows: list[dict[str, Any]] = []
    for column in features_wide.columns:
        series = features_wide[column]
        first_valid = series.first_valid_index()
        last_valid = series.last_valid_index()
        category = column.split("__", 1)[0] if "__" in column else "unknown"
        feature_metadata_rows.append({
            "feature_name": column,
            "category": category,
            "dtype": str(series.dtype),
            "non_null_count": int(series.notna().sum()),
            "null_count": int(series.isna().sum()),
            "first_valid_date": str(first_valid.date()) if first_valid is not None else None,
            "last_valid_date": str(last_valid.date()) if last_valid is not None else None,
            "updated_at": str(RUN_TIMESTAMP),
        })
    feature_metadata = pd.DataFrame(feature_metadata_rows)

    infinite_count = count_infinite(features_wide)
    duplicate_date_count = int(features_wide.index.duplicated().sum())
    future_date_count = int((features_wide.index > pd.Timestamp.now(tz="UTC").tz_localize(None)).sum())

    rsi_values = price_features_long["rsi_14"].dropna()
    rsi_out_of_range = int(((rsi_values < 0) | (rsi_values > 100)).sum())
    atr_negative = int((price_features_long["atr_20"].dropna() < 0).sum())

    macro_lookahead_violations = 0
    for column in [c for c in features_wide.columns if c.endswith("__source_observation_date")]:
        values = pd.to_datetime(features_wide[column], errors="coerce")
        macro_lookahead_violations += int((values > features_wide.index.to_series()).sum())

    critical_errors = {
        "missing_required_price_tickers": missing_required_price,
        "missing_required_macro_series": missing_required_macro,
        "duplicate_feature_dates": duplicate_date_count,
        "infinite_values": infinite_count,
        "future_feature_dates": future_date_count,
        "macro_lookahead_violations": macro_lookahead_violations,
        "rsi_out_of_range": rsi_out_of_range,
        "negative_atr_values": atr_negative,
    }

    has_critical_error = any(
        bool(value) if isinstance(value, list) else int(value) > 0
        for value in critical_errors.values()
    )
    overall_status = "FAIL" if has_critical_error else "PASS"

    test_results = [
        ("required_price_tickers_present", len(missing_required_price) == 0),
        ("required_macro_series_present", len(missing_required_macro) == 0),
        ("no_duplicate_feature_dates", duplicate_date_count == 0),
        ("no_infinite_values", infinite_count == 0),
        ("no_future_feature_dates", future_date_count == 0),
        ("no_macro_lookahead", macro_lookahead_violations == 0),
        ("rsi_within_0_100", rsi_out_of_range == 0),
        ("atr_non_negative", atr_negative == 0),
    ]
    tests_passed = sum(passed for _, passed in test_results)
    tests_failed = len(test_results) - tests_passed


# ============================================================
# 11. EXPORT
# ============================================================

with stage("Exporting feature files"):
    price_features_path = PROCESSED_DIR / "price_features.parquet"
    relative_features_path = PROCESSED_DIR / "relative_features.parquet"
    macro_native_path = PROCESSED_DIR / "macro_features_native.parquet"
    macro_aligned_path = PROCESSED_DIR / "macro_features_aligned.parquet"
    basket_features_path = PROCESSED_DIR / "ai_basket_features.parquet"
    features_path = PROCESSED_DIR / "features.parquet"
    features_latest_path = PROCESSED_DIR / "features_latest.csv"
    feature_metadata_path = OUTPUT_DIR / "feature_metadata.csv"
    feature_quality_path = OUTPUT_DIR / "feature_quality.json"
    feature_log_path = LOG_DIR / "feature_engine_log.json"
    tests_path = TEST_DIR / "step5_test_results.txt"

    price_features_long.to_parquet(price_features_path, index=False, compression="snappy")
    relative_features_wide.to_parquet(relative_features_path, index=True, compression="snappy")
    macro_features_native_long.to_parquet(macro_native_path, index=False, compression="snappy")
    macro_aligned_wide.to_parquet(macro_aligned_path, index=True, compression="snappy")
    basket_features_wide.to_parquet(basket_features_path, index=True, compression="snappy")
    features_wide.to_parquet(features_path, index=True, compression="snappy")

    latest_row = features_wide.tail(1).reset_index()
    latest_row.to_csv(features_latest_path, index=False)
    feature_metadata.to_csv(feature_metadata_path, index=False)

    if EXPORT_FULL_CSV_GZ:
        features_wide.to_csv(PROCESSED_DIR / "features.csv.gz", compression="gzip")

    with open(tests_path, "w", encoding="utf-8") as file:
        for test_name, passed in test_results:
            file.write(f"{'PASS' if passed else 'FAIL'} | {test_name}\n")
        file.write(f"\nPassed: {tests_passed}\nFailed: {tests_failed}\n")

    quality_report = {
        "step": 5,
        "engine": "IVMOS_FEATURE_ENGINE_V2",
        "overall_status": overall_status,
        "run_timestamp": str(RUN_TIMESTAMP),
        "input_rows": {
            "price": len(price_raw),
            "macro": len(macro_raw),
        },
        "input_duplicates_removed": {
            "price": price_duplicate_count_before,
            "macro": macro_duplicate_count_before,
        },
        "output_shapes": {
            "price_features_long": list(price_features_long.shape),
            "relative_features_wide": list(relative_features_wide.shape),
            "macro_features_native_long": list(macro_features_native_long.shape),
            "macro_features_aligned_wide": list(macro_aligned_wide.shape),
            "ai_basket_features_wide": list(basket_features_wide.shape),
            "features_wide": list(features_wide.shape),
        },
        "critical_errors": critical_errors,
        "tests_passed": tests_passed,
        "tests_failed": tests_failed,
        "stage_timings_seconds": STAGE_TIMINGS,
        "macro_alignment_method": "single wide backward-as-of reindex and forward propagation from past observation dates only",
        "macro_revision_limitation": "Current FRED vintage; not ALFRED point-in-time vintage history.",
        "interpretation_fields_created": False,
        "files_created": [],
    }

    engine_log = {
        "run_timestamp": str(RUN_TIMESTAMP),
        "relative_pairs": relative_pair_log,
        "macro_alignment": macro_alignment_log,
        "ai_baskets": basket_log,
        "stage_timings_seconds": STAGE_TIMINGS,
    }

    files_created = [
        price_features_path,
        relative_features_path,
        macro_native_path,
        macro_aligned_path,
        basket_features_path,
        features_path,
        features_latest_path,
        feature_metadata_path,
        feature_quality_path,
        feature_log_path,
        tests_path,
    ]
    if EXPORT_FULL_CSV_GZ:
        files_created.append(PROCESSED_DIR / "features.csv.gz")

    quality_report["files_created"] = [str(path) for path in files_created]
    write_json(feature_quality_path, quality_report)
    write_json(feature_log_path, engine_log)
# ============================================================
# 12. EXECUTION REPORT
# ============================================================

print("\n" + "=" * 78)
print("STEP 5 EXECUTION REPORT: IVMOS FEATURE ENGINE v2")
print("=" * 78)
print("Overall Status:", overall_status)
print("Price Tickers:", len(available_price_tickers))
print("Macro Series:", len(available_macro_series))
print("Price Feature Rows:", f"{len(price_features_long):,}")
print("Relative Feature Shape:", relative_features_wide.shape)
print("Macro Native Feature Rows:", f"{len(macro_features_native_long):,}")
print("Macro Aligned Feature Shape:", macro_aligned_wide.shape)
print("AI Basket Feature Shape:", basket_features_wide.shape)
print("Wide Feature Shape:", features_wide.shape)
print("Tests Passed:", tests_passed)
print("Tests Failed:", tests_failed)
print("Critical Errors:", critical_errors)
print("\nStage timings:")
for name, seconds in STAGE_TIMINGS.items():
    print(f"  {name}: {seconds:.3f}s")
print("\nFiles Created:")
for path in files_created:
    print(" ", path)
print("=" * 78)

# =============================================================================
# 13. LOSSLESS DTYPE NORMALIZATION (INTEGRATED)
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/IVMOS")
FEATURE_PATH = ROOT / "processed/features.parquet"
METADATA_PATH = ROOT / "outputs/feature_metadata.csv"
QUALITY_PATH = ROOT / "outputs/feature_quality.json"

print("Loading feature store...")
features = pd.read_parquet(FEATURE_PATH)

object_cols = features.select_dtypes(include=["object"]).columns.tolist()

converted_numeric = []
converted_boolean = []
conversion_failures = {}

for col in object_cols:
    s = features[col]

    # Detect boolean-like columns
    non_null_unique = set(s.dropna().unique().tolist())

    boolean_like_values = {
        True, False, 1, 0, 1.0, 0.0,
        "True", "False", "true", "false",
        "1", "0"
    }

    if non_null_unique and non_null_unique.issubset(boolean_like_values):
        mapped = s.map({
            True: True,
            False: False,
            1: True,
            0: False,
            1.0: True,
            0.0: False,
            "True": True,
            "False": False,
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })

        features[col] = mapped.astype("boolean")
        converted_boolean.append(col)
        continue

    # Otherwise require lossless numeric conversion
    numeric = pd.to_numeric(s, errors="coerce")

    original_non_null = int(s.notna().sum())
    numeric_non_null = int(numeric.notna().sum())

    if original_non_null == numeric_non_null:
        features[col] = numeric.astype("float64")
        converted_numeric.append(col)
    else:
        conversion_failures[col] = {
            "original_non_null": original_non_null,
            "numeric_non_null": numeric_non_null,
            "loss": original_non_null - numeric_non_null,
        }

if conversion_failures:
    print("Conversion failures found:")
    for key, value in list(conversion_failures.items())[:20]:
        print(key, value)

    raise ValueError(
        f"Unsafe dtype conversion detected in "
        f"{len(conversion_failures)} columns."
    )

# Validation after conversion
remaining_object_cols = (
    features.select_dtypes(include=["object"]).columns.tolist()
)

numeric_cols = features.select_dtypes(include=[np.number]).columns
boolean_cols = features.select_dtypes(include=["bool", "boolean"]).columns

infinite_values = int(
    np.isinf(features[numeric_cols].to_numpy(dtype="float64")).sum()
)

print("\nDtype normalization summary")
print("Converted numeric:", len(converted_numeric))
print("Converted boolean:", len(converted_boolean))
print("Remaining object:", len(remaining_object_cols))
print("Numeric columns:", len(numeric_cols))
print("Boolean columns:", len(boolean_cols))
print("Infinite values:", infinite_values)

if remaining_object_cols:
    print("Remaining object columns:")
    print(remaining_object_cols[:30])
    raise ValueError("Object columns remain after normalization.")

if infinite_values > 0:
    raise ValueError(
        f"Feature store contains {infinite_values} infinite values."
    )

# Write normalized feature store
features.to_parquet(
    FEATURE_PATH,
    index=True,
    compression="snappy",
)

# Refresh latest snapshot
features.tail(1).reset_index().to_csv(
    ROOT / "processed/features_latest.csv",
    index=False,
)

# Rebuild metadata using actual post-normalization dtypes
metadata_rows = []

for col in features.columns:
    s = features[col]

    first_valid = s.first_valid_index()
    last_valid = s.last_valid_index()

    metadata_rows.append({
        "feature_name": col,
        "category": col.split("__", 1)[0] if "__" in col else "other",
        "dtype": str(s.dtype),
        "non_null_count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "first_valid_date": (
            str(first_valid) if first_valid is not None else None
        ),
        "last_valid_date": (
            str(last_valid) if last_valid is not None else None
        ),
        "updated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    })

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(METADATA_PATH, index=False)

# Update quality report
quality = {}

if QUALITY_PATH.exists():
    with open(QUALITY_PATH, "r", encoding="utf-8") as f:
        quality = json.load(f)

quality["dtype_normalization"] = {
    "status": "PASS",
    "converted_numeric_columns": len(converted_numeric),
    "converted_boolean_columns": len(converted_boolean),
    "remaining_object_columns": len(remaining_object_cols),
    "numeric_columns": len(numeric_cols),
    "boolean_columns": len(boolean_cols),
    "infinite_values": infinite_values,
    "normalized_at": pd.Timestamp.now(tz="UTC").isoformat(),
}

with open(QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        quality,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("\nFiles updated:")
print(FEATURE_PATH)
print(ROOT / "processed/features_latest.csv")
print(METADATA_PATH)
print(QUALITY_PATH)
print("\nDTYPE NORMALIZATION: PASS")


PROJECT_ROOT: /content/drive/MyDrive/IVMOS
Price input: /content/drive/MyDrive/IVMOS/raw/prices/prices_daily.parquet
Macro input: /content/drive/MyDrive/IVMOS/raw/macro/fred_raw.parquet
Run timestamp: 2026-08-07 06:24:06.925580+00:00

Loading input data...
Price rows loaded: 138,725
Macro rows loaded: 22,334
Price tickers: 52
Macro series: 10
Loading input data: completed in 0.56s

Calculating price features...
  Processed 10/52 tickers
  Processed 20/52 tickers
  Processed 30/52 tickers
  Processed 40/52 tickers
  Processed 50/52 tickers
  Processed 52/52 tickers
Calculating price features: completed in 9.17s

Calculating relative features...
  Created 22/22 relative pairs
Calculating relative features: completed in 0.17s

Calculating macro native-frequency features...
  Processed 1/10: BAMLH0A0HYM2 (787 numeric rows)
  Processed 2/10: DFII10 (2,899 numeric rows)
  Processed 3/10: DGS10 (2,899 numeric rows)
  Processed 4/10: DGS2 (2,899 numeric rows)
  Processed 5/10: RRPONTSYD (2,890

## Step 6A — Evidence Registry and Calibration


In [8]:
# =============================================================================
# IVMOS PHASE 2.0
# Evidence Source Registry v2 — Draft + Feature Preflight
#
# Purpose:
# 1) Correct liquidity unit contracts
# 2) Create the 21-source Evidence Registry draft
# 3) Validate every referenced feature against Step 5 features.parquet
#
# This cell DOES NOT modify Step 6 v1.1 outputs.
# =============================================================================

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

FEATURE_PATH = PROJECT_ROOT / "processed" / "features.parquet"
DATA_DICTIONARY_PATH = PROJECT_ROOT / "config" / "data_dictionary.csv"

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_source_registry_v2_draft.yaml"
)

PREFLIGHT_CSV_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_registry_preflight.csv"
)

PREFLIGHT_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_registry_preflight.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step6_v2_registry_preflight.txt"
)

for directory in [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Validate inputs
# =============================================================================

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Missing Step 5 feature store: {FEATURE_PATH}"
    )

if not DATA_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Missing data dictionary: {DATA_DICTIONARY_PATH}"
    )

features = pd.read_parquet(FEATURE_PATH)

if "date" in features.columns:
    features["date"] = pd.to_datetime(
        features["date"],
        errors="coerce",
    )
    features = features.set_index("date")

features.index = pd.to_datetime(
    features.index,
    errors="coerce",
)

feature_columns = set(features.columns.astype(str))

print("Feature store loaded")
print("Shape:", features.shape)
print("Feature columns:", len(feature_columns))


# =============================================================================
# 2. Enforce and validate liquidity unit contracts
# =============================================================================

data_dictionary = pd.read_csv(DATA_DICTIONARY_PATH)

unit_corrections = {
    "WALCL": "Millions USD",
    "WTREGEN": "Millions USD",
    "RRPONTSYD": "Billions USD",
}

for instrument_id, correct_unit in unit_corrections.items():
    mask = data_dictionary["instrument_id"].eq(instrument_id)

    if not mask.any():
        raise ValueError(
            f"{instrument_id} missing from data dictionary"
        )

    data_dictionary.loc[mask, "unit"] = correct_unit

data_dictionary.to_csv(
    DATA_DICTIONARY_PATH,
    index=False,
)

print("\nLiquidity unit contracts validated:")
for instrument_id, correct_unit in unit_corrections.items():
    print(f"  {instrument_id}: {correct_unit}")


# =============================================================================
# Registry helper
# =============================================================================

def registry_source(
    source_id: str,
    name: str,
    owner_layer: str,
    descriptors: dict[str, list[str]],
    source_group: str,
    independence_cluster: str,
    quality_tier: str,
    underlying_series: list[str],
    secondary_references: list[dict[str, Any]] | None = None,
    curated_bias_flag: bool = False,
    structural_break_sensitive: bool = False,
    expected_behavior: str = "",
    failure_modes: list[str] | None = None,
) -> dict[str, Any]:

    return {
        "source_id": source_id,
        "name": name,
        "owner_layer": owner_layer,
        "secondary_references": secondary_references or [],
        "source_group": source_group,
        "independence_cluster": independence_cluster,
        "quality_tier": quality_tier,
        "underlying_series": underlying_series,
        "descriptors": descriptors,
        "curated_bias_flag": curated_bias_flag,
        "structural_break_sensitive": structural_break_sensitive,
        "expected_behavior": expected_behavior,
        "failure_modes": failure_modes or [],
        "status": "ACTIVE",
        "mapping_version": "0.1.0-draft",
    }


# =============================================================================
# 3. Evidence Source Registry — 21 independent source objects
# =============================================================================

ai_baskets = [
    "AI_COMPUTE",
    "AI_CONNECTIVITY",
    "AI_POWER_GRID",
    "AI_SOFTWARE",
    "AI_SPECULATIVE",
]

registry_sources = [

    # -------------------------------------------------------------------------
    # TREND — 3 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_SPY_TREND",
        name="SPY Trend Structure",
        owner_layer="TREND",
        descriptors={
            "short_trend": [
                "price__SPY__close",
                "price__SPY__ema_20",
            ],
            "medium_trend": [
                "price__SPY__close",
                "price__SPY__ema_50",
            ],
            "long_trend": [
                "price__SPY__close",
                "price__SPY__ema_200",
            ],
            "ema_structure": [
                "price__SPY__ema_20",
                "price__SPY__ema_50",
                "price__SPY__ema_200",
            ],
            "momentum_20d": [
                "price__SPY__return_20d",
            ],
            "momentum_60d": [
                "price__SPY__return_60d",
            ],
        },
        source_group="PRICE_TREND",
        independence_cluster="US_LARGE_CAP_TREND",
        quality_tier="B",
        underlying_series=["SPY"],
        secondary_references=[
            {
                "layer": "STRESS_BUFFER",
                "weight": 0.20,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher score means stronger and more durable broad-market trend."
        ),
        failure_modes=[
            "ETF price-feed anomaly",
            "index concentration masks internal weakness",
        ],
    ),

    registry_source(
        source_id="ES_QQQ_TREND",
        name="QQQ Long-Term Trend",
        owner_layer="TREND",
        descriptors={
            "long_trend_distance": [
                "price__QQQ__distance_ema200_pct",
            ],
        },
        source_group="PRICE_TREND",
        independence_cluster="US_GROWTH_TREND",
        quality_tier="B",
        underlying_series=["QQQ"],
        expected_behavior=(
            "Higher score means growth equities are stronger relative "
            "to their long-term trend."
        ),
    ),

    registry_source(
        source_id="ES_IWM_TREND",
        name="IWM Long-Term Trend",
        owner_layer="TREND",
        descriptors={
            "long_trend_distance": [
                "price__IWM__distance_ema200_pct",
            ],
        },
        source_group="PRICE_TREND",
        independence_cluster="US_SMALL_CAP_TREND",
        quality_tier="B",
        underlying_series=["IWM"],
        expected_behavior=(
            "Higher score means small caps are stronger relative "
            "to their long-term trend."
        ),
    ),

    # -------------------------------------------------------------------------
    # PARTICIPATION — 3 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_RSP_SPY",
        name="Equal-Weight Participation",
        owner_layer="PARTICIPATION",
        descriptors={
            "momentum": [
                "relative__RSP_SPY__ratio_roc20",
            ],
            "trend_state": [
                "relative__RSP_SPY__ratio",
                "relative__RSP_SPY__ratio_ema50",
            ],
        },
        source_group="MARKET_PARTICIPATION",
        independence_cluster="EQUAL_WEIGHT_PARTICIPATION",
        quality_tier="B",
        underlying_series=["RSP", "SPY"],
        secondary_references=[
            {
                "layer": "ROTATION",
                "weight": 0.30,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher score means equal-weight stocks are participating "
            "more strongly than cap-weight SPY."
        ),
    ),

    registry_source(
        source_id="ES_IWM_SPY",
        name="Small-Cap Participation",
        owner_layer="PARTICIPATION",
        descriptors={
            "momentum": [
                "relative__IWM_SPY__ratio_roc20",
            ],
            "trend_state": [
                "relative__IWM_SPY__ratio",
                "relative__IWM_SPY__ratio_ema50",
            ],
        },
        source_group="MARKET_PARTICIPATION",
        independence_cluster="SMALL_CAP_PARTICIPATION",
        quality_tier="B",
        underlying_series=["IWM", "SPY"],
        secondary_references=[
            {
                "layer": "ROTATION",
                "weight": 0.30,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher score means small caps are participating "
            "more strongly than SPY."
        ),
    ),

    registry_source(
        source_id="ES_SOXX_SPY",
        name="Semiconductor Participation",
        owner_layer="PARTICIPATION",
        descriptors={
            "momentum": [
                "relative__SOXX_SPY__ratio_roc20",
            ],
            "trend_state": [
                "relative__SOXX_SPY__ratio",
                "relative__SOXX_SPY__ratio_ema50",
            ],
        },
        source_group="MARKET_PARTICIPATION",
        independence_cluster="SEMICONDUCTOR_PARTICIPATION",
        quality_tier="B",
        underlying_series=["SOXX", "SPY"],
        expected_behavior=(
            "Higher score means semiconductors are participating "
            "more strongly than SPY."
        ),
    ),

    # -------------------------------------------------------------------------
    # ROTATION — 2 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_SECTOR_CYCLICAL",
        name="Cyclical Sector Rotation",
        owner_layer="ROTATION",
        descriptors={
            "relative_momentum": [
                "relative__XLK_SPY__ratio_roc20",
                "relative__XLI_SPY__ratio_roc20",
                "relative__XLF_SPY__ratio_roc20",
                "relative__XLE_SPY__ratio_roc20",
            ],
        },
        source_group="SECTOR_ROTATION",
        independence_cluster="CYCLICAL_ROTATION",
        quality_tier="B",
        underlying_series=[
            "XLK", "XLI", "XLF", "XLE", "SPY"
        ],
        secondary_references=[
            {
                "layer": "PARTICIPATION",
                "weight": 0.35,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher score means cyclical sectors are leading SPY."
        ),
    ),

    registry_source(
        source_id="ES_SECTOR_DEFENSIVE",
        name="Defensive Sector Rotation",
        owner_layer="ROTATION",
        descriptors={
            "relative_momentum": [
                "relative__XLU_SPY__ratio_roc20",
                "relative__XLV_SPY__ratio_roc20",
            ],
        },
        source_group="SECTOR_ROTATION",
        independence_cluster="DEFENSIVE_ROTATION",
        quality_tier="B",
        underlying_series=[
            "XLU", "XLV", "SPY"
        ],
        secondary_references=[
            {
                "layer": "PARTICIPATION",
                "weight": 0.35,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher raw relative strength means defensive sectors "
            "are leading; scoring polarity will be defined in calibration."
        ),
    ),

    # -------------------------------------------------------------------------
    # LEADERSHIP — 3 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_AI_LEADERSHIP",
        name="Curated AI Leadership Basket",
        owner_layer="LEADERSHIP",
        descriptors={
            "relative_spy_20d": [
                f"basket__{basket}__relative_spy_20d"
                for basket in ai_baskets
            ],
            "member_outperformance": [
                f"basket__{basket}__percent_members_outperform_spy_20d"
                for basket in ai_baskets
            ],
            "members_above_ema50": [
                f"basket__{basket}__percent_members_above_ema50"
                for basket in ai_baskets
            ],
        },
        source_group="THEMATIC_LEADERSHIP",
        independence_cluster="CURATED_AI_BASKETS",
        quality_tier="C",
        underlying_series=ai_baskets,
        curated_bias_flag=True,
        expected_behavior=(
            "Higher score means the curated AI universe is leading SPY."
        ),
        failure_modes=[
            "hindsight-selection bias",
            "non-point-in-time membership",
            "basket composition drift",
        ],
    ),

    registry_source(
        source_id="ES_QQQ_SPY",
        name="Growth Leadership",
        owner_layer="LEADERSHIP",
        descriptors={
            "relative_momentum": [
                "relative__QQQ_SPY__ratio_roc20",
            ],
        },
        source_group="TECH_LEADERSHIP",
        independence_cluster="GROWTH_LEADERSHIP",
        quality_tier="B",
        underlying_series=["QQQ", "SPY"],
        secondary_references=[
            {
                "layer": "TREND",
                "weight": 0.25,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Higher score means Nasdaq growth is leading SPY."
        ),
    ),

    registry_source(
        source_id="ES_SOXX_QQQ",
        name="Semiconductor Leadership",
        owner_layer="LEADERSHIP",
        descriptors={
            "relative_momentum": [
                "relative__SOXX_QQQ__ratio_roc20",
            ],
        },
        source_group="TECH_LEADERSHIP",
        independence_cluster="SEMICONDUCTOR_LEADERSHIP",
        quality_tier="B",
        underlying_series=["SOXX", "QQQ"],
        expected_behavior=(
            "Higher score means semiconductors are leading Nasdaq growth."
        ),
    ),

    # -------------------------------------------------------------------------
    # CREDIT — 2 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_HY_SPREAD",
        name="High-Yield Credit Spread",
        owner_layer="CREDIT",
        descriptors={
            "level": [
                "macro__BAMLH0A0HYM2__value",
            ],
            "momentum": [
                "macro__BAMLH0A0HYM2__diff_20obs",
            ],
            "percentile": [
                "macro__BAMLH0A0HYM2__value_percentile_252obs",
            ],
        },
        source_group="CREDIT",
        independence_cluster="HY_OPTION_ADJUSTED_SPREAD",
        quality_tier="A",
        underlying_series=["BAMLH0A0HYM2"],
        expected_behavior=(
            "Lower and falling spreads indicate healthier credit conditions."
        ),
    ),

    registry_source(
        source_id="ES_HYG_LQD",
        name="High-Yield versus Investment-Grade Price Signal",
        owner_layer="CREDIT",
        descriptors={
            "trend_state": [
                "relative__HYG_LQD__ratio",
                "relative__HYG_LQD__ratio_ema50",
            ],
            "momentum": [
                "relative__HYG_LQD__ratio_roc20",
            ],
        },
        source_group="CREDIT",
        independence_cluster="CREDIT_ETF_RELATIVE_PRICE",
        quality_tier="B",
        underlying_series=["HYG", "LQD"],
        expected_behavior=(
            "Higher HYG/LQD relative strength indicates improving "
            "market-implied credit risk appetite."
        ),
    ),

    # -------------------------------------------------------------------------
    # LIQUIDITY — 5 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_WALCL",
        name="Federal Reserve Balance Sheet",
        owner_layer="LIQUIDITY",
        descriptors={
            "year_over_year_change": [
                "macro__WALCL__yoy_pct_change",
            ],
            "trend_state": [
                "macro__WALCL__distance_ema20",
            ],
        },
        source_group="MONETARY_LIQUIDITY",
        independence_cluster="FED_BALANCE_SHEET",
        quality_tier="A",
        underlying_series=["WALCL"],
        structural_break_sensitive=True,
        expected_behavior=(
            "Balance-sheet expansion generally supports system liquidity."
        ),
        failure_modes=[
            "Fed operating-framework change",
            "accounting reclassification",
        ],
    ),

    registry_source(
        source_id="ES_TGA",
        name="Treasury General Account",
        owner_layer="LIQUIDITY",
        descriptors={
            "change_20obs": [
                "macro__WTREGEN__diff_20obs",
            ],
        },
        source_group="FISCAL_LIQUIDITY",
        independence_cluster="TREASURY_ACCOUNT",
        quality_tier="A",
        underlying_series=["WTREGEN"],
        structural_break_sensitive=True,
        expected_behavior=(
            "A falling TGA generally releases liquidity; "
            "a rising TGA generally drains liquidity."
        ),
    ),

    registry_source(
        source_id="ES_RRP",
        name="Overnight Reverse Repo",
        owner_layer="LIQUIDITY",
        descriptors={
            "change_20obs": [
                "macro__RRPONTSYD__diff_20obs",
            ],
        },
        source_group="MONETARY_LIQUIDITY",
        independence_cluster="REVERSE_REPO",
        quality_tier="A",
        underlying_series=["RRPONTSYD"],
        structural_break_sensitive=True,
        expected_behavior=(
            "A falling RRP balance can release liquidity, "
            "subject to the prevailing Fed operating framework."
        ),
    ),

    registry_source(
        source_id="ES_DFII10",
        name="Ten-Year Real Yield",
        owner_layer="LIQUIDITY",
        descriptors={
            "change_20obs": [
                "macro__DFII10__diff_20obs",
            ],
        },
        source_group="COST_OF_CAPITAL",
        independence_cluster="REAL_YIELD",
        quality_tier="A",
        underlying_series=["DFII10"],
        expected_behavior=(
            "Falling real yields generally ease financial conditions."
        ),
    ),

    registry_source(
        source_id="ES_SOFR",
        name="Secured Overnight Financing Rate",
        owner_layer="LIQUIDITY",
        descriptors={
            "change_20obs": [
                "macro__SOFR__diff_20obs",
            ],
        },
        source_group="FUNDING_LIQUIDITY",
        independence_cluster="SECURED_FUNDING_RATE",
        quality_tier="A",
        underlying_series=["SOFR"],
        structural_break_sensitive=True,
        expected_behavior=(
            "Falling SOFR generally eases short-term secured "
            "funding conditions."
        ),
    ),

    # -------------------------------------------------------------------------
    # STRESS BUFFER — 3 sources
    # -------------------------------------------------------------------------

    registry_source(
        source_id="ES_VIX",
        name="VIX Implied Volatility",
        owner_layer="STRESS_BUFFER",
        descriptors={
            "level": [
                "macro__VIXCLS__value",
            ],
            "momentum": [
                "macro__VIXCLS__diff_20obs",
            ],
            "percentile": [
                "macro__VIXCLS__value_percentile_252obs",
            ],
        },
        source_group="MARKET_STRESS",
        independence_cluster="IMPLIED_EQUITY_VOLATILITY",
        quality_tier="A",
        underlying_series=["VIXCLS"],
        expected_behavior=(
            "Lower and falling VIX indicates greater market risk capacity."
        ),
    ),

    registry_source(
        source_id="ES_SPY_VOL",
        name="SPY Realized Volatility",
        owner_layer="STRESS_BUFFER",
        descriptors={
            "realized_volatility": [
                "price__SPY__volatility_20d_annualized",
            ],
        },
        source_group="MARKET_STRESS",
        independence_cluster="REALIZED_EQUITY_VOLATILITY",
        quality_tier="B",
        underlying_series=["SPY"],
        secondary_references=[
            {
                "layer": "TREND",
                "weight": 0.20,
                "count_in_confidence": False,
            }
        ],
        expected_behavior=(
            "Lower realized volatility indicates greater market "
            "risk-bearing capacity."
        ),
    ),

    registry_source(
        source_id="ES_SPY_DRAWDOWN",
        name="SPY Drawdown",
        owner_layer="STRESS_BUFFER",
        descriptors={
            "drawdown_60d": [
                "price__SPY__drawdown_60d",
            ],
        },
        source_group="MARKET_STRESS",
        independence_cluster="EQUITY_DRAWDOWN",
        quality_tier="B",
        underlying_series=["SPY"],
        expected_behavior=(
            "Smaller drawdown indicates greater market stress capacity."
        ),
    ),
]


registry_document = {
    "schema_name": "IVMOS Evidence Source Registry",
    "schema_version": "0.1.0-draft",
    "target_engine_version": "2.0.0",
    "neutral_score": 50.0,
    "score_range": [0.0, 100.0],
    "single_owner_required": True,
    "reference_counts_in_confidence": False,
    "calibration_status": "INITIAL_REQUIRES_BACKTEST",
    "sources": registry_sources,
}


# =============================================================================
# 4. Registry validation
# =============================================================================

source_ids = [
    source["source_id"]
    for source in registry_sources
]

duplicate_source_ids = sorted({
    source_id
    for source_id in source_ids
    if source_ids.count(source_id) > 1
})

preflight_rows = []
all_missing_features = []

for source in registry_sources:

    referenced_features = []

    for descriptor_features in source["descriptors"].values():
        referenced_features.extend(descriptor_features)

    referenced_features = sorted(set(referenced_features))

    missing_features = sorted(
        feature
        for feature in referenced_features
        if feature not in feature_columns
    )

    all_missing_features.extend(missing_features)

    available_count = (
        len(referenced_features)
        - len(missing_features)
    )

    availability_ratio = (
        available_count / len(referenced_features)
        if referenced_features
        else 0.0
    )

    preflight_rows.append({
        "source_id": source["source_id"],
        "owner_layer": source["owner_layer"],
        "quality_tier": source["quality_tier"],
        "independence_cluster": source["independence_cluster"],
        "referenced_feature_count": len(referenced_features),
        "available_feature_count": available_count,
        "missing_feature_count": len(missing_features),
        "availability_ratio": availability_ratio,
        "missing_features": "|".join(missing_features),
        "curated_bias_flag": source["curated_bias_flag"],
        "structural_break_sensitive": (
            source["structural_break_sensitive"]
        ),
    })

preflight_df = pd.DataFrame(preflight_rows)

owner_layers = sorted(
    preflight_df["owner_layer"].unique()
)

expected_layers = {
    "TREND",
    "PARTICIPATION",
    "LEADERSHIP",
    "ROTATION",
    "CREDIT",
    "LIQUIDITY",
    "STRESS_BUFFER",
}

actual_layers = set(owner_layers)

layer_contract_pass = (
    actual_layers == expected_layers
)

feature_contract_pass = (
    len(set(all_missing_features)) == 0
)

source_count_pass = (
    len(registry_sources) == 21
)

duplicate_source_pass = (
    len(duplicate_source_ids) == 0
)

unit_contract = {
    instrument_id: data_dictionary.loc[
        data_dictionary["instrument_id"].eq(instrument_id),
        "unit",
    ].iloc[0]
    for instrument_id in unit_corrections
}

unit_contract_pass = (
    unit_contract == unit_corrections
)

tests = {
    "source_count_is_21": source_count_pass,
    "no_duplicate_source_id": duplicate_source_pass,
    "expected_owner_layers_present": layer_contract_pass,
    "all_referenced_features_exist": feature_contract_pass,
    "liquidity_units_correct": unit_contract_pass,
}

overall_status = (
    "PASS"
    if all(tests.values())
    else "FAIL"
)


# =============================================================================
# 5. Export draft registry and preflight results
# =============================================================================

with open(
    REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        registry_document,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=120,
    )

preflight_df.to_csv(
    PREFLIGHT_CSV_PATH,
    index=False,
)

preflight_report = {
    "engine": "IVMOS Evidence Registry Preflight",
    "registry_version": "0.1.0-draft",
    "overall_status": overall_status,
    "feature_store_shape": [
        int(features.shape[0]),
        int(features.shape[1]),
    ],
    "source_count": len(registry_sources),
    "owner_layers": owner_layers,
    "missing_features": sorted(set(all_missing_features)),
    "duplicate_source_ids": duplicate_source_ids,
    "liquidity_unit_contract": unit_contract,
    "tests": tests,
    "files_created": [
        str(REGISTRY_PATH),
        str(PREFLIGHT_CSV_PATH),
        str(PREFLIGHT_JSON_PATH),
        str(TEST_PATH),
    ],
}

with open(
    PREFLIGHT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preflight_report,
        file,
        ensure_ascii=False,
        indent=2,
    )

test_lines = [
    "=" * 78,
    "IVMOS STEP 6 v2 — EVIDENCE REGISTRY PREFLIGHT",
    "=" * 78,
    f"Overall Status: {overall_status}",
    f"Evidence Sources: {len(registry_sources)}",
    f"Owner Layers: {', '.join(owner_layers)}",
    f"Missing Features: {len(set(all_missing_features))}",
    "",
]

for test_name, passed in tests.items():
    test_lines.append(
        f"{'PASS' if passed else 'FAIL'} — {test_name}"
    )

if all_missing_features:
    test_lines.extend([
        "",
        "Missing feature list:",
        *[
            f"  - {feature}"
            for feature in sorted(set(all_missing_features))
        ],
    ])

test_lines.extend([
    "",
    "Files created:",
    f"  {REGISTRY_PATH}",
    f"  {PREFLIGHT_CSV_PATH}",
    f"  {PREFLIGHT_JSON_PATH}",
    f"  {TEST_PATH}",
    "=" * 78,
])

TEST_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Final report
# =============================================================================

print("\n" + "=" * 78)
print("IVMOS STEP 6 v2 — EVIDENCE REGISTRY PREFLIGHT")
print("=" * 78)
print("Overall Status:", overall_status)
print("Evidence Sources:", len(registry_sources))
print("Owner Layers:", ", ".join(owner_layers))
print("Missing Features:", len(set(all_missing_features)))

print("\nTests:")
for test_name, passed in tests.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {test_name}"
    )

if all_missing_features:
    print("\nMissing feature list:")
    for feature in sorted(set(all_missing_features)):
        print(" ", feature)

print("\nLiquidity unit contract:")
for instrument_id, unit in unit_contract.items():
    print(f"  {instrument_id}: {unit}")

print("\nFiles created:")
for path in preflight_report["files_created"]:
    print(" ", path)

print("=" * 78)


Feature store loaded
Shape: (2916, 2843)
Feature columns: 2843

Liquidity unit contracts validated:
  WALCL: Millions USD
  WTREGEN: Millions USD
  RRPONTSYD: Billions USD

IVMOS STEP 6 v2 — EVIDENCE REGISTRY PREFLIGHT
Overall Status: PASS
Evidence Sources: 21
Owner Layers: CREDIT, LEADERSHIP, LIQUIDITY, PARTICIPATION, ROTATION, STRESS_BUFFER, TREND
Missing Features: 0

Tests:
  PASS — source_count_is_21
  PASS — no_duplicate_source_id
  PASS — expected_owner_layers_present
  PASS — all_referenced_features_exist
  PASS — liquidity_units_correct

Liquidity unit contract:
  WALCL: Millions USD
  WTREGEN: Millions USD
  RRPONTSYD: Billions USD

Files created:
  /content/drive/MyDrive/IVMOS/config/evidence_source_registry_v2_draft.yaml
  /content/drive/MyDrive/IVMOS/outputs/evidence_v2_registry_preflight.csv
  /content/drive/MyDrive/IVMOS/outputs/evidence_v2_registry_preflight.json
  /content/drive/MyDrive/IVMOS/tests/step6_v2_registry_preflight.txt


In [9]:
# =============================================================================
# IVMOS STEP 6 v2
# Evidence Calibration Registry — Draft v0.1.0
#
# Purpose:
# 1) Define descriptor transforms and weights for all 21 evidence sources
# 2) Define source weights for all 7 evidence layers
# 3) Freeze confidence, coverage, freshness and stability contracts
# 4) Validate calibration against Evidence Source Registry
#
# This cell creates configuration only.
# It does NOT replace Evidence Engine v1.1 yet.
# =============================================================================

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_source_registry_v2_draft.yaml"
)

CALIBRATION_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_calibration_v2_draft.yaml"
)

VALIDATION_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_calibration_validation.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step6_v2_calibration_validation.txt"
)

for directory in [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Load Evidence Source Registry
# =============================================================================

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(
        f"Missing Evidence Source Registry: {REGISTRY_PATH}"
    )

with open(REGISTRY_PATH, "r", encoding="utf-8") as file:
    registry = yaml.safe_load(file)

registry_sources = registry.get("sources", [])

if not registry_sources:
    raise ValueError("Evidence Source Registry contains no sources")

registry_by_id = {
    source["source_id"]: source
    for source in registry_sources
}

print("Evidence Source Registry loaded")
print("Registry version:", registry.get("schema_version"))
print("Sources:", len(registry_by_id))


# =============================================================================
# Transform helpers
# =============================================================================

def rolling_percentile(
    polarity: str,
    window: int = 756,
    min_periods: int = 252,
) -> dict[str, Any]:
    """
    Convert a continuous descriptor into a historical percentile score.

    positive:
        Higher raw value = more risk-supportive.

    negative:
        Lower raw value = more risk-supportive.
    """
    return {
        "type": "rolling_percentile",
        "polarity": polarity,
        "window_observations": window,
        "minimum_observations": min_periods,
        "winsorize_quantiles": [0.01, 0.99],
        "fallback": {
            "type": "expanding_percentile",
            "minimum_observations": 60,
        },
    }


def pair_relative_distance(
    polarity: str = "positive",
    scale_percent: float = 3.0,
) -> dict[str, Any]:
    """
    Score the relative distance between descriptor input 0 and input 1.

    Typical usage:
        ratio versus ratio EMA
        price versus EMA
    """
    return {
        "type": "pair_relative_distance",
        "polarity": polarity,
        "left_input_index": 0,
        "right_input_index": 1,
        "scale_percent": scale_percent,
        "neutral_band_percent": 0.25,
        "clip_score": [0.0, 100.0],
    }


def ordered_structure() -> dict[str, Any]:
    """
    Used for EMA20 / EMA50 / EMA200 structure.

    Bullish:
        EMA20 > EMA50 > EMA200

    Bearish:
        EMA20 < EMA50 < EMA200
    """
    return {
        "type": "ordered_structure",
        "bullish_relation": "input_0 > input_1 > input_2",
        "bearish_relation": "input_0 < input_1 < input_2",
        "scores": {
            "fully_bullish": 100.0,
            "partially_bullish": 70.0,
            "mixed": 50.0,
            "partially_bearish": 30.0,
            "fully_bearish": 0.0,
        },
    }


def multi_series_mean_percentile(
    polarity: str,
) -> dict[str, Any]:
    """
    Take the cross-series mean, then convert it into a rolling percentile.
    """
    return {
        "type": "multi_series_mean_percentile",
        "polarity": polarity,
        "window_observations": 756,
        "minimum_observations": 252,
        "minimum_series_coverage": 0.60,
        "winsorize_quantiles": [0.01, 0.99],
    }


def percentile_value(
    polarity: str,
) -> dict[str, Any]:
    """
    Descriptor is already a percentile-like field.
    Engine must automatically detect 0–1 versus 0–100 scale.
    """
    return {
        "type": "percentile_value",
        "polarity": polarity,
        "input_scale": "AUTO",
        "clip_score": [0.0, 100.0],
    }


def source_spec(
    descriptor_weights: dict[str, float],
    transforms: dict[str, dict[str, Any]],
    freshness_profile: str,
    minimum_descriptor_coverage: float = 0.50,
) -> dict[str, Any]:

    return {
        "descriptor_weights": descriptor_weights,
        "transforms": transforms,
        "aggregation": {
            "method": "weighted_mean",
            "renormalize_available_weights": True,
            "minimum_descriptor_coverage": (
                minimum_descriptor_coverage
            ),
            "neutral_score_when_unavailable": 50.0,
        },
        "freshness_profile": freshness_profile,
    }


# =============================================================================
# Source-level calibration
#
# Score orientation contract:
#   100 = strongly risk-supportive
#    50 = neutral
#     0 = strongly risk-adverse
# =============================================================================

source_scoring = {

    # -------------------------------------------------------------------------
    # TREND
    # -------------------------------------------------------------------------

    "ES_SPY_TREND": source_spec(
        descriptor_weights={
            "short_trend": 0.10,
            "medium_trend": 0.20,
            "long_trend": 0.25,
            "ema_structure": 0.20,
            "momentum_20d": 0.10,
            "momentum_60d": 0.15,
        },
        transforms={
            "short_trend": pair_relative_distance(
                polarity="positive",
                scale_percent=3.0,
            ),
            "medium_trend": pair_relative_distance(
                polarity="positive",
                scale_percent=5.0,
            ),
            "long_trend": pair_relative_distance(
                polarity="positive",
                scale_percent=10.0,
            ),
            "ema_structure": ordered_structure(),
            "momentum_20d": rolling_percentile(
                polarity="positive",
            ),
            "momentum_60d": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
        minimum_descriptor_coverage=0.60,
    ),

    "ES_QQQ_TREND": source_spec(
        descriptor_weights={
            "long_trend_distance": 1.00,
        },
        transforms={
            "long_trend_distance": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_IWM_TREND": source_spec(
        descriptor_weights={
            "long_trend_distance": 1.00,
        },
        transforms={
            "long_trend_distance": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    # -------------------------------------------------------------------------
    # PARTICIPATION
    # -------------------------------------------------------------------------

    "ES_RSP_SPY": source_spec(
        descriptor_weights={
            "momentum": 0.60,
            "trend_state": 0.40,
        },
        transforms={
            "momentum": rolling_percentile(
                polarity="positive",
            ),
            "trend_state": pair_relative_distance(
                polarity="positive",
                scale_percent=3.0,
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_IWM_SPY": source_spec(
        descriptor_weights={
            "momentum": 0.60,
            "trend_state": 0.40,
        },
        transforms={
            "momentum": rolling_percentile(
                polarity="positive",
            ),
            "trend_state": pair_relative_distance(
                polarity="positive",
                scale_percent=4.0,
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_SOXX_SPY": source_spec(
        descriptor_weights={
            "momentum": 0.60,
            "trend_state": 0.40,
        },
        transforms={
            "momentum": rolling_percentile(
                polarity="positive",
            ),
            "trend_state": pair_relative_distance(
                polarity="positive",
                scale_percent=5.0,
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    # -------------------------------------------------------------------------
    # ROTATION
    # -------------------------------------------------------------------------

    "ES_SECTOR_CYCLICAL": source_spec(
        descriptor_weights={
            "relative_momentum": 1.00,
        },
        transforms={
            "relative_momentum": (
                multi_series_mean_percentile(
                    polarity="positive",
                )
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_SECTOR_DEFENSIVE": source_spec(
        descriptor_weights={
            "relative_momentum": 1.00,
        },
        transforms={
            "relative_momentum": (
                multi_series_mean_percentile(
                    polarity="negative",
                )
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    # -------------------------------------------------------------------------
    # LEADERSHIP
    # -------------------------------------------------------------------------

    "ES_AI_LEADERSHIP": source_spec(
        descriptor_weights={
            "relative_spy_20d": 0.40,
            "member_outperformance": 0.35,
            "members_above_ema50": 0.25,
        },
        transforms={
            "relative_spy_20d": (
                multi_series_mean_percentile(
                    polarity="positive",
                )
            ),
            "member_outperformance": (
                multi_series_mean_percentile(
                    polarity="positive",
                )
            ),
            "members_above_ema50": (
                multi_series_mean_percentile(
                    polarity="positive",
                )
            ),
        },
        freshness_profile="PRICE_DAILY",
        minimum_descriptor_coverage=0.67,
    ),

    "ES_QQQ_SPY": source_spec(
        descriptor_weights={
            "relative_momentum": 1.00,
        },
        transforms={
            "relative_momentum": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_SOXX_QQQ": source_spec(
        descriptor_weights={
            "relative_momentum": 1.00,
        },
        transforms={
            "relative_momentum": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    # -------------------------------------------------------------------------
    # CREDIT
    # -------------------------------------------------------------------------

    "ES_HY_SPREAD": source_spec(
        descriptor_weights={
            "level": 0.35,
            "momentum": 0.35,
            "percentile": 0.30,
        },
        transforms={
            "level": rolling_percentile(
                polarity="negative",
            ),
            "momentum": rolling_percentile(
                polarity="negative",
            ),
            "percentile": percentile_value(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_DAILY",
        minimum_descriptor_coverage=0.67,
    ),

    "ES_HYG_LQD": source_spec(
        descriptor_weights={
            "trend_state": 0.45,
            "momentum": 0.55,
        },
        transforms={
            "trend_state": pair_relative_distance(
                polarity="positive",
                scale_percent=3.0,
            ),
            "momentum": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    # -------------------------------------------------------------------------
    # LIQUIDITY
    # -------------------------------------------------------------------------

    "ES_WALCL": source_spec(
        descriptor_weights={
            "year_over_year_change": 0.60,
            "trend_state": 0.40,
        },
        transforms={
            "year_over_year_change": rolling_percentile(
                polarity="positive",
            ),
            "trend_state": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="MACRO_WEEKLY",
    ),

    "ES_TGA": source_spec(
        descriptor_weights={
            "change_20obs": 1.00,
        },
        transforms={
            "change_20obs": rolling_percentile(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_WEEKLY",
    ),

    "ES_RRP": source_spec(
        descriptor_weights={
            "change_20obs": 1.00,
        },
        transforms={
            "change_20obs": rolling_percentile(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_DAILY",
    ),

    "ES_DFII10": source_spec(
        descriptor_weights={
            "change_20obs": 1.00,
        },
        transforms={
            "change_20obs": rolling_percentile(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_DAILY",
    ),

    "ES_SOFR": source_spec(
        descriptor_weights={
            "change_20obs": 1.00,
        },
        transforms={
            "change_20obs": rolling_percentile(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_DAILY",
    ),

    # -------------------------------------------------------------------------
    # STRESS BUFFER
    # -------------------------------------------------------------------------

    "ES_VIX": source_spec(
        descriptor_weights={
            "level": 0.35,
            "momentum": 0.35,
            "percentile": 0.30,
        },
        transforms={
            "level": rolling_percentile(
                polarity="negative",
            ),
            "momentum": rolling_percentile(
                polarity="negative",
            ),
            "percentile": percentile_value(
                polarity="negative",
            ),
        },
        freshness_profile="MACRO_DAILY",
        minimum_descriptor_coverage=0.67,
    ),

    "ES_SPY_VOL": source_spec(
        descriptor_weights={
            "realized_volatility": 1.00,
        },
        transforms={
            "realized_volatility": rolling_percentile(
                polarity="negative",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),

    "ES_SPY_DRAWDOWN": source_spec(
        descriptor_weights={
            "drawdown_60d": 1.00,
        },
        transforms={
            # Drawdown closer to zero is a larger raw value
            # and represents greater stress capacity.
            "drawdown_60d": rolling_percentile(
                polarity="positive",
            ),
        },
        freshness_profile="PRICE_DAILY",
    ),
}


# =============================================================================
# Layer source weights
#
# These weights affect Layer Score.
# Independence is calculated separately by cluster.
# =============================================================================

layer_scoring = {

    "TREND": {
        "source_weights": {
            "ES_SPY_TREND": 0.50,
            "ES_QQQ_TREND": 0.25,
            "ES_IWM_TREND": 0.25,
        },
    },

    "PARTICIPATION": {
        "source_weights": {
            "ES_RSP_SPY": 0.40,
            "ES_IWM_SPY": 0.35,
            "ES_SOXX_SPY": 0.25,
        },
    },

    "ROTATION": {
        "source_weights": {
            "ES_SECTOR_CYCLICAL": 0.60,
            "ES_SECTOR_DEFENSIVE": 0.40,
        },
    },

    "LEADERSHIP": {
        "source_weights": {
            "ES_AI_LEADERSHIP": 0.40,
            "ES_QQQ_SPY": 0.30,
            "ES_SOXX_QQQ": 0.30,
        },
    },

    "CREDIT": {
        "source_weights": {
            "ES_HY_SPREAD": 0.60,
            "ES_HYG_LQD": 0.40,
        },
    },

    "LIQUIDITY": {
        "source_weights": {
            "ES_WALCL": 0.25,
            "ES_TGA": 0.20,
            "ES_RRP": 0.15,
            "ES_DFII10": 0.25,
            "ES_SOFR": 0.15,
        },
    },

    "STRESS_BUFFER": {
        "source_weights": {
            "ES_VIX": 0.45,
            "ES_SPY_VOL": 0.35,
            "ES_SPY_DRAWDOWN": 0.20,
        },
    },
}

for layer_config in layer_scoring.values():
    layer_config["aggregation"] = {
        "method": "cluster_normalized_weighted_mean",
        "neutral_score": 50.0,
        "renormalize_available_weights": True,
        "minimum_source_coverage": 0.50,
    }


# Credit availability policy:
# - FULL: HY OAS + HYG/LQD
# - PROXY: HYG/LQD only when primary HY OAS history is unavailable
# Proxy mode receives a separate confidence ceiling in the v2.1 stage.
layer_scoring["CREDIT"]["aggregation"].update({
    "minimum_source_coverage": 0.40,
    "availability_modes": {
        "FULL": {
            "required_sources": ["ES_HY_SPREAD", "ES_HYG_LQD"],
            "confidence_ceiling": 94.0,
        },
        "PROXY": {
            "required_sources": ["ES_HYG_LQD"],
            "excluded_sources": ["ES_HY_SPREAD"],
            "confidence_ceiling": 65.0,
        },
    },
})


# =============================================================================
# Calibration document
# =============================================================================

calibration = {
    "schema_name": "IVMOS Evidence Calibration",
    "schema_version": "0.1.0-draft",
    "target_registry_version": registry.get("schema_version"),
    "target_engine_version": "2.0.0",

    "score_contract": {
        "minimum": 0.0,
        "neutral": 50.0,
        "maximum": 100.0,
        "orientation": (
            "Higher score always means more risk-supportive."
        ),
    },

    "quality_factors": {
        "A": 1.00,
        "B": 0.85,
        "C": 0.65,
    },

    "freshness_profiles": {
        "PRICE_DAILY": {
            "full_credit_age_days": 1,
            "zero_credit_age_days": 5,
            "decay": "linear",
        },
        "MACRO_DAILY": {
            "full_credit_age_days": 3,
            "zero_credit_age_days": 10,
            "decay": "linear",
        },
        "MACRO_WEEKLY": {
            "full_credit_age_days": 8,
            "zero_credit_age_days": 21,
            "decay": "linear",
        },
    },

    "stability": {
        "lookback_observations": 20,
        "minimum_observations": 10,
        "method": "inverse_score_volatility",
        "full_credit_max_score_std": 8.0,
        "zero_credit_score_std": 30.0,
    },

    "source_confidence": {
        # Coverage is deliberately excluded from base weights.
        # It is applied once as a multiplier below.
        "base_weights": {
            "source_quality": 0.40,
            "freshness": 0.30,
            "stability": 0.30,
        },
        "descriptor_coverage_multiplier": True,
        "formula": (
            "base_confidence * descriptor_coverage_ratio"
        ),
    },

    "layer_confidence": {
        # Coverage is deliberately excluded from base weights.
        # This avoids counting missingness twice.
        "base_weights": {
            "independence": 0.35,
            "agreement": 0.25,
            "freshness": 0.20,
            "source_quality": 0.20,
        },
        "source_coverage_multiplier": True,
        "formula": (
            "base_confidence * source_coverage_ratio"
        ),
    },

    "independence": {
        "unit": "independence_cluster",
        "references_count_in_independence": False,
        "method": "effective_cluster_weight",
        "effective_sample_size_method": "kish",
        "maximum_credit_per_cluster": 1.0,
    },

    "agreement": {
        "method": "weighted_median_absolute_deviation",
        "neutral_score": 50.0,
        "maximum_dispersion": 50.0,
        "minimum_sources": 2,
        "single_source_agreement_credit": 0.50,
    },

    "structural_break_policy": {
        "flagged_source_confidence_multiplier": 0.70,
        "manual_review_required": True,
        "automatic_recalibration": False,
    },

    "curated_bias_policy": {
        "curated_source_confidence_multiplier": 0.75,
        "eligible_for_layer_score": True,
        "eligible_for_independent_validation": False,
        "eligible_for_system_backtest_claim": False,
    },

    "source_scoring": source_scoring,
    "layer_scoring": layer_scoring,

    "metadata": {
        "status": "INITIAL_REQUIRES_BACKTEST",
        "created_for": "IVMOS Step 6 Evidence Engine v2",
        "notes": [
            "No threshold is considered final before historical audit.",
            "Coverage is multiplied once and is not included in base confidence.",
            "Reference use affects layer score only when explicitly configured.",
            "All layer scores share the same risk-supportive orientation.",
        ],
    },
}


# =============================================================================
# Validation
# =============================================================================

allowed_transform_types = {
    "rolling_percentile",
    "pair_relative_distance",
    "ordered_structure",
    "multi_series_mean_percentile",
    "percentile_value",
}

validation_errors = []
validation_checks = {}


# 1. Source ID coverage
registry_source_ids = set(registry_by_id)
calibrated_source_ids = set(source_scoring)

validation_checks["all_registry_sources_calibrated"] = (
    registry_source_ids == calibrated_source_ids
)

if registry_source_ids != calibrated_source_ids:
    missing = sorted(
        registry_source_ids - calibrated_source_ids
    )
    unexpected = sorted(
        calibrated_source_ids - registry_source_ids
    )

    validation_errors.append(
        f"Missing source calibration: {missing}"
    )
    validation_errors.append(
        f"Unexpected source calibration: {unexpected}"
    )


# 2. Descriptor contracts
descriptor_contract_pass = True

for source_id, source in registry_by_id.items():

    registry_descriptors = set(
        source.get("descriptors", {}).keys()
    )

    calibration_spec = source_scoring.get(
        source_id,
        {},
    )

    calibrated_descriptors = set(
        calibration_spec
        .get("descriptor_weights", {})
        .keys()
    )

    transform_descriptors = set(
        calibration_spec
        .get("transforms", {})
        .keys()
    )

    if (
        registry_descriptors != calibrated_descriptors
        or registry_descriptors != transform_descriptors
    ):
        descriptor_contract_pass = False

        validation_errors.append(
            f"{source_id}: descriptor contract mismatch "
            f"registry={sorted(registry_descriptors)}, "
            f"weights={sorted(calibrated_descriptors)}, "
            f"transforms={sorted(transform_descriptors)}"
        )

validation_checks["descriptor_contracts_match"] = (
    descriptor_contract_pass
)


# 3. Descriptor weights sum to one
descriptor_weight_pass = True

for source_id, spec in source_scoring.items():

    weight_sum = sum(
        spec["descriptor_weights"].values()
    )

    if abs(weight_sum - 1.0) > 1e-9:
        descriptor_weight_pass = False

        validation_errors.append(
            f"{source_id}: descriptor weights sum to "
            f"{weight_sum:.12f}"
        )

validation_checks["descriptor_weights_sum_to_one"] = (
    descriptor_weight_pass
)


# 4. Transform types supported
transform_type_pass = True

for source_id, spec in source_scoring.items():

    for descriptor, transform in (
        spec["transforms"].items()
    ):
        transform_type = transform.get("type")

        if transform_type not in allowed_transform_types:
            transform_type_pass = False

            validation_errors.append(
                f"{source_id}.{descriptor}: "
                f"unsupported transform {transform_type}"
            )

validation_checks["all_transform_types_supported"] = (
    transform_type_pass
)


# 5. Layer ownership and source coverage
layer_contract_pass = True

registry_sources_by_layer = {}

for source_id, source in registry_by_id.items():
    layer = source["owner_layer"]

    registry_sources_by_layer.setdefault(
        layer,
        set(),
    ).add(source_id)

for layer, config in layer_scoring.items():

    configured_sources = set(
        config["source_weights"]
    )

    expected_sources = registry_sources_by_layer.get(
        layer,
        set(),
    )

    if configured_sources != expected_sources:
        layer_contract_pass = False

        validation_errors.append(
            f"{layer}: source contract mismatch "
            f"expected={sorted(expected_sources)}, "
            f"configured={sorted(configured_sources)}"
        )

validation_checks["layer_source_ownership_valid"] = (
    layer_contract_pass
)


# 6. Layer source weights sum to one
layer_weight_pass = True

for layer, config in layer_scoring.items():

    weight_sum = sum(
        config["source_weights"].values()
    )

    if abs(weight_sum - 1.0) > 1e-9:
        layer_weight_pass = False

        validation_errors.append(
            f"{layer}: source weights sum to "
            f"{weight_sum:.12f}"
        )

validation_checks["layer_weights_sum_to_one"] = (
    layer_weight_pass
)


# 7. Confidence base weights sum to one
source_confidence_sum = sum(
    calibration["source_confidence"]
    ["base_weights"].values()
)

layer_confidence_sum = sum(
    calibration["layer_confidence"]
    ["base_weights"].values()
)

confidence_weight_pass = (
    abs(source_confidence_sum - 1.0) <= 1e-9
    and abs(layer_confidence_sum - 1.0) <= 1e-9
)

validation_checks["confidence_weights_sum_to_one"] = (
    confidence_weight_pass
)

if not confidence_weight_pass:
    validation_errors.append(
        "Confidence base weights do not sum to one"
    )


# 8. Coverage must not appear in confidence base weights
coverage_single_count_pass = (
    "coverage"
    not in calibration[
        "source_confidence"
    ]["base_weights"]
    and
    "coverage"
    not in calibration[
        "layer_confidence"
    ]["base_weights"]
    and
    calibration[
        "source_confidence"
    ]["descriptor_coverage_multiplier"]
    and
    calibration[
        "layer_confidence"
    ]["source_coverage_multiplier"]
)

validation_checks["coverage_counted_once"] = (
    coverage_single_count_pass
)

if not coverage_single_count_pass:
    validation_errors.append(
        "Coverage must be applied exactly once"
    )


# 9. Quality tiers
registry_quality_tiers = {
    source["quality_tier"]
    for source in registry_sources
}

configured_quality_tiers = set(
    calibration["quality_factors"]
)

quality_tier_pass = (
    registry_quality_tiers
    <= configured_quality_tiers
)

validation_checks["quality_tiers_configured"] = (
    quality_tier_pass
)

if not quality_tier_pass:
    validation_errors.append(
        "Missing quality factors for tiers: "
        f"{sorted(registry_quality_tiers - configured_quality_tiers)}"
    )


# 10. Freshness profiles
freshness_pass = True

configured_profiles = set(
    calibration["freshness_profiles"]
)

for source_id, spec in source_scoring.items():

    profile = spec["freshness_profile"]

    if profile not in configured_profiles:
        freshness_pass = False

        validation_errors.append(
            f"{source_id}: unknown freshness profile "
            f"{profile}"
        )

validation_checks["freshness_profiles_valid"] = (
    freshness_pass
)


overall_status = (
    "PASS"
    if all(validation_checks.values())
    and not validation_errors
    else "FAIL"
)


# =============================================================================
# Export
# =============================================================================

with open(
    CALIBRATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        calibration,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=120,
    )

validation_report = {
    "engine": "IVMOS Evidence Calibration Validation",
    "schema_version": calibration["schema_version"],
    "target_registry_version": (
        calibration["target_registry_version"]
    ),
    "overall_status": overall_status,
    "source_count": len(source_scoring),
    "layer_count": len(layer_scoring),
    "checks": validation_checks,
    "errors": validation_errors,
    "files_created": [
        str(CALIBRATION_PATH),
        str(VALIDATION_JSON_PATH),
        str(TEST_PATH),
    ],
}

with open(
    VALIDATION_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        validation_report,
        file,
        ensure_ascii=False,
        indent=2,
    )

test_lines = [
    "=" * 78,
    "IVMOS STEP 6 v2 — CALIBRATION VALIDATION",
    "=" * 78,
    f"Overall Status: {overall_status}",
    f"Sources: {len(source_scoring)}",
    f"Layers: {len(layer_scoring)}",
    "",
]

for check_name, passed in validation_checks.items():
    test_lines.append(
        f"{'PASS' if passed else 'FAIL'} — {check_name}"
    )

if validation_errors:
    test_lines.extend([
        "",
        "Validation errors:",
        *[
            f"  - {error}"
            for error in validation_errors
        ],
    ])

test_lines.extend([
    "",
    "Files created:",
    f"  {CALIBRATION_PATH}",
    f"  {VALIDATION_JSON_PATH}",
    f"  {TEST_PATH}",
    "=" * 78,
])

TEST_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Final report
# =============================================================================

print("\n" + "=" * 78)
print("IVMOS STEP 6 v2 — CALIBRATION VALIDATION")
print("=" * 78)
print("Overall Status:", overall_status)
print("Sources:", len(source_scoring))
print("Layers:", len(layer_scoring))

print("\nChecks:")
for check_name, passed in validation_checks.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {check_name}"
    )

if validation_errors:
    print("\nValidation errors:")
    for error in validation_errors:
        print(" ", error)

print("\nFiles created:")
for path in validation_report["files_created"]:
    print(" ", path)

print("=" * 78)


Evidence Source Registry loaded
Registry version: 0.1.0-draft
Sources: 21

IVMOS STEP 6 v2 — CALIBRATION VALIDATION
Overall Status: PASS
Sources: 21
Layers: 7

Checks:
  PASS — all_registry_sources_calibrated
  PASS — descriptor_contracts_match
  PASS — descriptor_weights_sum_to_one
  PASS — all_transform_types_supported
  PASS — layer_source_ownership_valid
  PASS — layer_weights_sum_to_one
  PASS — confidence_weights_sum_to_one
  PASS — coverage_counted_once
  PASS — quality_tiers_configured
  PASS — freshness_profiles_valid

Files created:
  /content/drive/MyDrive/IVMOS/config/evidence_calibration_v2_draft.yaml
  /content/drive/MyDrive/IVMOS/outputs/evidence_v2_calibration_validation.json
  /content/drive/MyDrive/IVMOS/tests/step6_v2_calibration_validation.txt


## Step 6B — Evidence Runtime and Confidence v2.1


In [10]:
# =============================================================================
# IVMOS STEP 6 v2
# Evidence Engine Runtime v0.1.0
#
# Inputs
# ------
# processed/features.parquet
# config/evidence_source_registry_v2_draft.yaml
# config/evidence_calibration_v2_draft.yaml
#
# Outputs
# -------
# processed/evidence_sources_v2.parquet
# processed/evidence_layers_v2.parquet
# processed/evidence_contributions_v2.parquet
# outputs/evidence_v2_latest.json
# outputs/evidence_v2_quality.json
# logs/evidence_v2_runtime_log.json
# tests/step6_v2_runtime_tests.txt
#
# Important
# ---------
# - Does not overwrite Evidence Engine v1.1
# - All rolling transforms are backward-looking
# - Higher score always means more risk-supportive
# - Coverage is applied once as a confidence multiplier
# =============================================================================

from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

FEATURE_PATH = (
    PROJECT_ROOT
    / "processed"
    / "features.parquet"
)

RAW_MACRO_PATH = (
    PROJECT_ROOT
    / "raw"
    / "macro"
    / "fred_raw.parquet"
)

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_source_registry_v2_draft.yaml"
)

CALIBRATION_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_calibration_v2_draft.yaml"
)

SOURCE_OUTPUT_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_sources_v2.parquet"
)

LAYER_OUTPUT_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_layers_v2.parquet"
)

CONTRIBUTION_OUTPUT_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_contributions_v2.parquet"
)

LATEST_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_latest.json"
)

QUALITY_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_quality.json"
)

LOG_OUTPUT_PATH = (
    PROJECT_ROOT
    / "logs"
    / "evidence_v2_runtime_log.json"
)

TEST_OUTPUT_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step6_v2_runtime_tests.txt"
)

for directory in [
    PROJECT_ROOT / "processed",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "logs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Runtime metadata
# =============================================================================

ENGINE_NAME = "IVMOS Evidence Engine v2"
ENGINE_VERSION = "0.1.0-runtime"
RUN_STARTED = pd.Timestamp.now(tz="UTC")

runtime_warnings: list[str] = []
runtime_notes: list[str] = []


# =============================================================================
# Generic helpers
# =============================================================================

def normalize_datetime_index(
    values: Any,
) -> pd.DatetimeIndex:

    index = pd.DatetimeIndex(
        pd.to_datetime(
            values,
            errors="coerce",
        )
    )

    index = index[~index.isna()]

    if index.tz is not None:
        index = index.tz_convert(None)

    return index.normalize()


def json_safe(value: Any) -> Any:

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.datetime64):
        return pd.Timestamp(value).isoformat()

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if pd.isna(value):
        return None

    return value


def linear_score(
    value: pd.Series,
    full_credit: float,
    zero_credit: float,
) -> pd.Series:

    result = pd.Series(
        np.nan,
        index=value.index,
        dtype=float,
    )

    valid = value.notna()

    result.loc[
        valid & (value <= full_credit)
    ] = 100.0

    result.loc[
        valid & (value >= zero_credit)
    ] = 0.0

    middle = (
        valid
        & (value > full_credit)
        & (value < zero_credit)
    )

    denominator = (
        zero_credit - full_credit
    )

    if denominator <= 0:
        raise ValueError(
            "zero_credit must be greater than full_credit"
        )

    result.loc[middle] = (
        100.0
        * (
            zero_credit
            - value.loc[middle]
        )
        / denominator
    )

    return result.clip(0.0, 100.0)


def weighted_median(
    values: np.ndarray,
    weights: np.ndarray,
) -> float:

    valid = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[valid]
    weights = weights[valid]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)

    cutoff = 0.5 * weights.sum()

    position = np.searchsorted(
        cumulative,
        cutoff,
        side="left",
    )

    position = min(
        position,
        len(values) - 1,
    )

    return float(values[position])


def kish_effective_sample_size(
    weights: np.ndarray,
) -> float:

    weights = np.asarray(
        weights,
        dtype=float,
    )

    weights = weights[
        np.isfinite(weights)
        & (weights > 0)
    ]

    if len(weights) == 0:
        return 0.0

    normalized = (
        weights / weights.sum()
    )

    denominator = np.sum(
        normalized ** 2
    )

    if denominator <= 0:
        return 0.0

    return float(
        1.0 / denominator
    )


# =============================================================================
# Load inputs
# =============================================================================

for required_path in [
    FEATURE_PATH,
    REGISTRY_PATH,
    CALIBRATION_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required input: {required_path}"
        )


features = pd.read_parquet(
    FEATURE_PATH
)

if "date" in features.columns:
    features["date"] = pd.to_datetime(
        features["date"],
        errors="coerce",
    )

    features = features.set_index(
        "date"
    )

features.index = pd.to_datetime(
    features.index,
    errors="coerce",
)

features = features.loc[
    ~features.index.isna()
].copy()

if features.index.tz is not None:
    features.index = (
        features.index.tz_convert(None)
    )

features.index = (
    features.index.normalize()
)

features = (
    features
    .sort_index()
    .groupby(level=0)
    .last()
)

features = features.apply(
    pd.to_numeric,
    errors="coerce",
)


with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    registry = yaml.safe_load(file)


with open(
    CALIBRATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    calibration = yaml.safe_load(file)


registry_sources = (
    registry.get("sources", [])
)

registry_by_id = {
    source["source_id"]: source
    for source in registry_sources
}

source_scoring = (
    calibration["source_scoring"]
)

layer_scoring = (
    calibration["layer_scoring"]
)

quality_factors = (
    calibration["quality_factors"]
)

freshness_profiles = (
    calibration["freshness_profiles"]
)


if set(registry_by_id) != set(source_scoring):
    raise ValueError(
        "Registry and calibration source IDs do not match"
    )


print("Feature store loaded")
print("Shape:", features.shape)
print(
    "Date range:",
    features.index.min().date(),
    "to",
    features.index.max().date(),
)

print("\nEvidence configuration loaded")
print(
    "Registry version:",
    registry.get("schema_version"),
)
print(
    "Calibration version:",
    calibration.get("schema_version"),
)
print(
    "Evidence sources:",
    len(registry_sources),
)
print(
    "Evidence layers:",
    len(layer_scoring),
)


# =============================================================================
# Point-in-time percentile transform
#
# Rolling.rank uses only the current and preceding observations.
# Expanding rank is used during the initial warm-up period.
# =============================================================================

def rolling_percentile_fraction(
    series: pd.Series,
    window: int,
    minimum_observations: int,
    fallback_minimum_observations: int,
) -> pd.Series:

    series = pd.to_numeric(
        series,
        errors="coerce",
    )

    try:
        rolling_rank = (
            series
            .rolling(
                window=window,
                min_periods=minimum_observations,
            )
            .rank(
                pct=True,
            )
        )

    except Exception:
        rolling_rank = (
            series
            .rolling(
                window=window,
                min_periods=minimum_observations,
            )
            .apply(
                lambda values: (
                    pd.Series(values)
                    .rank(pct=True)
                    .iloc[-1]
                ),
                raw=False,
            )
        )

    try:
        expanding_rank = (
            series
            .expanding(
                min_periods=(
                    fallback_minimum_observations
                )
            )
            .rank(
                pct=True,
            )
        )

    except Exception:
        expanding_rank = (
            series
            .expanding(
                min_periods=(
                    fallback_minimum_observations
                )
            )
            .apply(
                lambda values: (
                    pd.Series(values)
                    .rank(pct=True)
                    .iloc[-1]
                ),
                raw=False,
            )
        )

    result = rolling_rank.combine_first(
        expanding_rank
    )

    return result.clip(
        lower=0.0,
        upper=1.0,
    )


def percentile_score(
    series: pd.Series,
    transform: dict[str, Any],
) -> pd.Series:

    window = int(
        transform.get(
            "window_observations",
            756,
        )
    )

    minimum_observations = int(
        transform.get(
            "minimum_observations",
            252,
        )
    )

    fallback = transform.get(
        "fallback",
        {},
    )

    fallback_minimum = int(
        fallback.get(
            "minimum_observations",
            60,
        )
    )

    percentile = (
        rolling_percentile_fraction(
            series=series,
            window=window,
            minimum_observations=(
                minimum_observations
            ),
            fallback_minimum_observations=(
                fallback_minimum
            ),
        )
    )

    polarity = transform.get(
        "polarity",
        "positive",
    )

    if polarity == "positive":
        score = 100.0 * percentile

    elif polarity == "negative":
        score = (
            100.0
            * (
                1.0
                - percentile
            )
        )

    else:
        raise ValueError(
            f"Unknown polarity: {polarity}"
        )

    return score.clip(
        0.0,
        100.0,
    )


# =============================================================================
# Descriptor transforms
# =============================================================================

def transform_descriptor(
    feature_frame: pd.DataFrame,
    feature_names: list[str],
    transform: dict[str, Any],
) -> pd.Series:

    missing = [
        feature_name
        for feature_name in feature_names
        if feature_name not in feature_frame.columns
    ]

    if missing:
        raise KeyError(
            f"Missing descriptor features: {missing}"
        )

    inputs = (
        feature_frame[
            feature_names
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    transform_type = transform["type"]

    # -------------------------------------------------------------------------
    # Rolling percentile
    # -------------------------------------------------------------------------

    if transform_type == "rolling_percentile":

        if len(feature_names) == 1:
            raw_series = inputs.iloc[:, 0]

        else:
            raw_series = inputs.mean(
                axis=1,
                skipna=True,
            )

        return percentile_score(
            raw_series,
            transform,
        )

    # -------------------------------------------------------------------------
    # Pair relative distance
    # -------------------------------------------------------------------------

    if transform_type == "pair_relative_distance":

        if len(feature_names) < 2:
            raise ValueError(
                "pair_relative_distance requires "
                "at least two inputs"
            )

        left_index = int(
            transform.get(
                "left_input_index",
                0,
            )
        )

        right_index = int(
            transform.get(
                "right_input_index",
                1,
            )
        )

        left = inputs.iloc[
            :,
            left_index,
        ]

        right = inputs.iloc[
            :,
            right_index,
        ].replace(
            0.0,
            np.nan,
        )

        distance_percent = (
            (
                left / right
            )
            - 1.0
        ) * 100.0

        neutral_band = float(
            transform.get(
                "neutral_band_percent",
                0.0,
            )
        )

        scale_percent = float(
            transform.get(
                "scale_percent",
                5.0,
            )
        )

        effective_scale = max(
            scale_percent
            - neutral_band,
            1e-9,
        )

        adjusted_distance = (
            np.sign(distance_percent)
            * np.maximum(
                np.abs(distance_percent)
                - neutral_band,
                0.0,
            )
        )

        score = (
            50.0
            + 50.0
            * (
                adjusted_distance
                / effective_scale
            )
        )

        polarity = transform.get(
            "polarity",
            "positive",
        )

        if polarity == "negative":
            score = 100.0 - score

        elif polarity != "positive":
            raise ValueError(
                f"Unknown polarity: {polarity}"
            )

        return score.clip(
            0.0,
            100.0,
        )

    # -------------------------------------------------------------------------
    # Ordered structure
    # -------------------------------------------------------------------------

    if transform_type == "ordered_structure":

        if len(feature_names) != 3:
            raise ValueError(
                "ordered_structure requires "
                "exactly three inputs"
            )

        first = inputs.iloc[:, 0]
        second = inputs.iloc[:, 1]
        third = inputs.iloc[:, 2]

        valid = (
            first.notna()
            & second.notna()
            & third.notna()
        )

        fully_bullish = (
            valid
            & (first > second)
            & (second > third)
        )

        fully_bearish = (
            valid
            & (first < second)
            & (second < third)
        )

        partially_bullish = (
            valid
            & ~fully_bullish
            & ~fully_bearish
            & (first > third)
        )

        partially_bearish = (
            valid
            & ~fully_bullish
            & ~fully_bearish
            & (first < third)
        )

        mixed = (
            valid
            & ~fully_bullish
            & ~fully_bearish
            & ~partially_bullish
            & ~partially_bearish
        )

        score_map = transform["scores"]

        result = pd.Series(
            np.nan,
            index=feature_frame.index,
            dtype=float,
        )

        result.loc[fully_bullish] = float(
            score_map["fully_bullish"]
        )

        result.loc[partially_bullish] = float(
            score_map["partially_bullish"]
        )

        result.loc[mixed] = float(
            score_map["mixed"]
        )

        result.loc[partially_bearish] = float(
            score_map["partially_bearish"]
        )

        result.loc[fully_bearish] = float(
            score_map["fully_bearish"]
        )

        return result.clip(
            0.0,
            100.0,
        )

    # -------------------------------------------------------------------------
    # Multi-series mean percentile
    # -------------------------------------------------------------------------

    if transform_type == "multi_series_mean_percentile":

        minimum_coverage = float(
            transform.get(
                "minimum_series_coverage",
                0.60,
            )
        )

        row_coverage = (
            inputs.notna().mean(axis=1)
        )

        raw_series = inputs.mean(
            axis=1,
            skipna=True,
        )

        raw_series = raw_series.where(
            row_coverage
            >= minimum_coverage
        )

        percentile_transform = {
            "type": "rolling_percentile",
            "polarity": transform.get(
                "polarity",
                "positive",
            ),
            "window_observations": int(
                transform.get(
                    "window_observations",
                    756,
                )
            ),
            "minimum_observations": int(
                transform.get(
                    "minimum_observations",
                    252,
                )
            ),
            "fallback": {
                "type": "expanding_percentile",
                "minimum_observations": 60,
            },
        }

        return percentile_score(
            raw_series,
            percentile_transform,
        )

    # -------------------------------------------------------------------------
    # Existing percentile field
    # -------------------------------------------------------------------------

    if transform_type == "percentile_value":

        if len(feature_names) == 1:
            raw_series = inputs.iloc[:, 0]

        else:
            raw_series = inputs.mean(
                axis=1,
                skipna=True,
            )

        # Supports percentile fields stored as 0–1
        # or as 0–100 without a global/future scale check.
        normalized = raw_series.where(
            raw_series.abs() > 1.5,
            raw_series * 100.0,
        )

        normalized = normalized.clip(
            0.0,
            100.0,
        )

        polarity = transform.get(
            "polarity",
            "positive",
        )

        if polarity == "negative":
            normalized = (
                100.0 - normalized
            )

        elif polarity != "positive":
            raise ValueError(
                f"Unknown polarity: {polarity}"
            )

        return normalized.clip(
            0.0,
            100.0,
        )

    raise ValueError(
        f"Unsupported transform type: {transform_type}"
    )


# =============================================================================
# Raw macro observation calendar
#
# This is used for staleness estimates only.
# Historical FRED values may still contain later revisions.
# =============================================================================

def load_macro_observation_dates(
    path: Path,
) -> tuple[
    dict[str, pd.DatetimeIndex],
    str,
]:

    if not path.exists():
        return {}, "RAW_MACRO_FILE_MISSING"

    macro = pd.read_parquet(
        path
    )

    if isinstance(
        macro.index,
        pd.DatetimeIndex,
    ):
        macro = (
            macro
            .reset_index()
            .rename(
                columns={
                    macro.index.name or "index": "date"
                }
            )
        )

    columns_lower = {
        str(column).lower(): column
        for column in macro.columns
    }

    date_candidates = [
        "date",
        "observation_date",
        "timestamp",
        "datetime",
    ]

    series_candidates = [
        "series_id",
        "instrument_id",
        "ticker",
        "series",
        "symbol",
    ]

    value_candidates = [
        "value",
        "close",
        "observation_value",
    ]

    date_column = next(
        (
            columns_lower[candidate]
            for candidate in date_candidates
            if candidate in columns_lower
        ),
        None,
    )

    if date_column is None:
        return {}, "RAW_MACRO_DATE_COLUMN_NOT_FOUND"

    macro[date_column] = pd.to_datetime(
        macro[date_column],
        errors="coerce",
    )

    series_column = next(
        (
            columns_lower[candidate]
            for candidate in series_candidates
            if candidate in columns_lower
        ),
        None,
    )

    value_column = next(
        (
            columns_lower[candidate]
            for candidate in value_candidates
            if candidate in columns_lower
        ),
        None,
    )

    observation_map: dict[
        str,
        pd.DatetimeIndex
    ] = {}

    # Long format
    if (
        series_column is not None
        and value_column is not None
    ):

        usable = macro.loc[
            macro[date_column].notna()
            & macro[value_column].notna()
        ].copy()

        for series_id, group in usable.groupby(
            series_column
        ):
            observation_map[str(series_id)] = (
                normalize_datetime_index(
                    group[date_column]
                )
                .unique()
                .sort_values()
            )

        return observation_map, "RAW_MACRO_LONG_FORMAT"

    # Wide format
    excluded = {date_column}

    for column in macro.columns:

        if column in excluded:
            continue

        valid_dates = macro.loc[
            macro[column].notna(),
            date_column,
        ]

        if len(valid_dates) == 0:
            continue

        observation_map[str(column)] = (
            normalize_datetime_index(
                valid_dates
            )
            .unique()
            .sort_values()
        )

    return observation_map, "RAW_MACRO_WIDE_FORMAT"


macro_observation_map, macro_observation_mode = (
    load_macro_observation_dates(
        RAW_MACRO_PATH
    )
)

if not macro_observation_map:
    runtime_warnings.append(
        "Raw macro observation calendar unavailable. "
        "Macro freshness falls back to feature availability."
    )

runtime_warnings.append(
    "FRED historical observations are not vintage/revision safe "
    "unless an ALFRED-style point-in-time dataset is used."
)


def age_since_observation(
    index: pd.DatetimeIndex,
    observation_dates: pd.DatetimeIndex,
) -> pd.Series:

    observation_dates = (
        normalize_datetime_index(
            observation_dates
        )
        .unique()
        .sort_values()
    )

    combined_index = (
        index.union(
            observation_dates
        )
        .sort_values()
    )

    marker = pd.Series(
        pd.NaT,
        index=combined_index,
        dtype="datetime64[ns]",
    )

    marker.loc[observation_dates] = (
        observation_dates
    )

    last_observation = (
        marker
        .ffill()
        .reindex(index)
    )

    index_series = pd.Series(
        index,
        index=index,
    )

    age = (
        index_series
        - last_observation
    ).dt.days.astype(float)

    return age


def source_freshness_age(
    source: dict[str, Any],
    source_available: pd.Series,
    freshness_profile: str,
) -> tuple[pd.Series, str]:

    if freshness_profile == "PRICE_DAILY":

        age = pd.Series(
            np.where(
                source_available,
                0.0,
                np.nan,
            ),
            index=features.index,
            dtype=float,
        )

        return age, "PRICE_FEATURE_DATE"

    candidate_series = [
        str(series_id)
        for series_id
        in source.get(
            "underlying_series",
            [],
        )
        if str(series_id)
        in macro_observation_map
    ]

    if candidate_series:

        ages = pd.DataFrame(
            {
                series_id: age_since_observation(
                    index=features.index,
                    observation_dates=(
                        macro_observation_map[
                            series_id
                        ]
                    ),
                )
                for series_id
                in candidate_series
            },
            index=features.index,
        )

        # Conservatively use the stalest required series.
        age = ages.max(
            axis=1,
            skipna=True,
        )

        return age, "RAW_MACRO_OBSERVATION_DATE"

    age = pd.Series(
        np.where(
            source_available,
            0.0,
            np.nan,
        ),
        index=features.index,
        dtype=float,
    )

    return age, "FEATURE_AVAILABILITY_FALLBACK"


def freshness_score_from_age(
    age: pd.Series,
    profile_name: str,
) -> pd.Series:

    profile = freshness_profiles[
        profile_name
    ]

    full_credit = float(
        profile[
            "full_credit_age_days"
        ]
    )

    zero_credit = float(
        profile[
            "zero_credit_age_days"
        ]
    )

    if profile.get("decay") != "linear":
        raise ValueError(
            "Runtime currently supports "
            "linear freshness decay only"
        )

    return linear_score(
        value=age,
        full_credit=full_credit,
        zero_credit=zero_credit,
    )


# =============================================================================
# Compute descriptor and source evidence
# =============================================================================

descriptor_score_cache: dict[
    tuple[str, str],
    pd.Series
] = {}

source_frames: dict[
    str,
    pd.DataFrame
] = {}

descriptor_contribution_frames: list[
    pd.DataFrame
] = []

freshness_modes_used: dict[
    str,
    str
] = {}


print("\nCalculating Evidence Sources...")


for source_number, source in enumerate(
    registry_sources,
    start=1,
):

    source_id = source["source_id"]

    source_config = (
        source_scoring[source_id]
    )

    descriptor_features = (
        source["descriptors"]
    )

    descriptor_weights = (
        source_config[
            "descriptor_weights"
        ]
    )

    transforms = (
        source_config[
            "transforms"
        ]
    )

    descriptor_scores = pd.DataFrame(
        index=features.index
    )

    for descriptor_name, feature_names in (
        descriptor_features.items()
    ):

        score = transform_descriptor(
            feature_frame=features,
            feature_names=feature_names,
            transform=transforms[
                descriptor_name
            ],
        )

        descriptor_scores[
            descriptor_name
        ] = score

        descriptor_score_cache[
            (
                source_id,
                descriptor_name,
            )
        ] = score

    weight_series = pd.Series(
        descriptor_weights,
        dtype=float,
    )

    total_descriptor_weight = float(
        weight_series.sum()
    )

    availability_matrix = (
        descriptor_scores.notna()
    )

    available_weight = (
        availability_matrix
        .mul(
            weight_series,
            axis=1,
        )
        .sum(axis=1)
    )

    descriptor_coverage = (
        available_weight
        / total_descriptor_weight
    ).clip(
        0.0,
        1.0,
    )

    weighted_numerator = (
        descriptor_scores
        .mul(
            weight_series,
            axis=1,
        )
        .sum(
            axis=1,
            min_count=1,
        )
    )

    raw_source_score = (
        weighted_numerator
        / available_weight.replace(
            0.0,
            np.nan,
        )
    )

    minimum_descriptor_coverage = float(
        source_config[
            "aggregation"
        ].get(
            "minimum_descriptor_coverage",
            0.50,
        )
    )

    source_available = (
        descriptor_coverage
        >= minimum_descriptor_coverage
    ) & raw_source_score.notna()

    neutral_score = float(
        source_config[
            "aggregation"
        ].get(
            "neutral_score_when_unavailable",
            50.0,
        )
    )

    source_score = (
        raw_source_score
        .where(
            source_available,
            neutral_score,
        )
        .clip(
            0.0,
            100.0,
        )
    )

    score_for_stability = (
        source_score.where(
            source_available
        )
    )

    stability_config = (
        calibration["stability"]
    )

    stability_std = (
        score_for_stability
        .rolling(
            window=int(
                stability_config[
                    "lookback_observations"
                ]
            ),
            min_periods=int(
                stability_config[
                    "minimum_observations"
                ]
            ),
        )
        .std(ddof=0)
    )

    stability_score = linear_score(
        value=stability_std,
        full_credit=float(
            stability_config[
                "full_credit_max_score_std"
            ]
        ),
        zero_credit=float(
            stability_config[
                "zero_credit_score_std"
            ]
        ),
    ).fillna(0.0)

    freshness_profile = (
        source_config[
            "freshness_profile"
        ]
    )

    freshness_age, freshness_mode = (
        source_freshness_age(
            source=source,
            source_available=source_available,
            freshness_profile=freshness_profile,
        )
    )

    freshness_modes_used[
        source_id
    ] = freshness_mode

    freshness_score = (
        freshness_score_from_age(
            age=freshness_age,
            profile_name=freshness_profile,
        )
        .fillna(0.0)
    )

    quality_tier = source[
        "quality_tier"
    ]

    quality_factor = float(
        quality_factors[
            quality_tier
        ]
    )

    quality_score = (
        quality_factor * 100.0
    )

    source_confidence_weights = (
        calibration[
            "source_confidence"
        ]["base_weights"]
    )

    base_source_confidence = (
        float(
            source_confidence_weights[
                "source_quality"
            ]
        )
        * quality_score
        +
        float(
            source_confidence_weights[
                "freshness"
            ]
        )
        * freshness_score
        +
        float(
            source_confidence_weights[
                "stability"
            ]
        )
        * stability_score
    )

    source_confidence = (
        base_source_confidence
        * descriptor_coverage
    )

    curated_bias_flag = bool(
        source.get(
            "curated_bias_flag",
            False,
        )
    )

    if curated_bias_flag:
        curated_multiplier = float(
            calibration[
                "curated_bias_policy"
            ][
                "curated_source_confidence_multiplier"
            ]
        )

        source_confidence = (
            source_confidence
            * curated_multiplier
        )

    # Structural sensitivity is metadata.
    # It is not the same as an active structural-break event.
    structural_break_flag = pd.Series(
        False,
        index=features.index,
        dtype=bool,
    )

    source_confidence = (
        source_confidence
        .where(
            source_available,
            0.0,
        )
        .clip(
            0.0,
            100.0,
        )
    )

    source_frame = pd.DataFrame(
        {
            "score": source_score,
            "confidence": source_confidence,
            "descriptor_coverage": (
                descriptor_coverage
            ),
            "freshness_score": (
                freshness_score
            ),
            "freshness_age_days": (
                freshness_age
            ),
            "stability_score": (
                stability_score
            ),
            "stability_std": (
                stability_std
            ),
            "quality_score": (
                quality_score
            ),
            "is_available": (
                source_available
            ),
            "structural_break_flag": (
                structural_break_flag
            ),
        },
        index=features.index,
    )

    source_frames[source_id] = (
        source_frame
    )

    # Descriptor → Source contribution records
    normalized_descriptor_weights = (
        availability_matrix
        .mul(
            weight_series,
            axis=1,
        )
        .div(
            available_weight.replace(
                0.0,
                np.nan,
            ),
            axis=0,
        )
        .fillna(0.0)
    )

    for descriptor_name in (
        descriptor_scores.columns
    ):

        contribution = pd.DataFrame(
            {
                "date": features.index,
                "record_type": (
                    "DESCRIPTOR_TO_SOURCE"
                ),
                "layer": source[
                    "owner_layer"
                ],
                "source_id": source_id,
                "descriptor": descriptor_name,
                "independence_cluster": source[
                    "independence_cluster"
                ],
                "raw_weight": float(
                    descriptor_weights[
                        descriptor_name
                    ]
                ),
                "effective_weight": (
                    normalized_descriptor_weights[
                        descriptor_name
                    ].to_numpy()
                ),
                "input_score": (
                    descriptor_scores[
                        descriptor_name
                    ].to_numpy()
                ),
                "weighted_contribution": (
                    descriptor_scores[
                        descriptor_name
                    ]
                    * normalized_descriptor_weights[
                        descriptor_name
                    ]
                ).to_numpy(),
                "is_available": (
                    descriptor_scores[
                        descriptor_name
                    ].notna()
                    .to_numpy()
                ),
            }
        )

        descriptor_contribution_frames.append(
            contribution
        )

    if (
        source_number % 5 == 0
        or source_number == len(registry_sources)
    ):
        print(
            f"  Processed "
            f"{source_number}/"
            f"{len(registry_sources)} sources"
        )


# =============================================================================
# Source output table
# =============================================================================

source_output_frames = []

for source in registry_sources:

    source_id = source["source_id"]

    frame = (
        source_frames[source_id]
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": "date"
            }
        )
    )

    frame.insert(
        1,
        "source_id",
        source_id,
    )

    frame.insert(
        2,
        "source_name",
        source["name"],
    )

    frame.insert(
        3,
        "owner_layer",
        source["owner_layer"],
    )

    frame.insert(
        4,
        "source_group",
        source["source_group"],
    )

    frame.insert(
        5,
        "independence_cluster",
        source[
            "independence_cluster"
        ],
    )

    frame.insert(
        6,
        "quality_tier",
        source["quality_tier"],
    )

    frame["curated_bias_flag"] = bool(
        source.get(
            "curated_bias_flag",
            False,
        )
    )

    frame[
        "structural_break_sensitive"
    ] = bool(
        source.get(
            "structural_break_sensitive",
            False,
        )
    )

    frame["freshness_mode"] = (
        freshness_modes_used[source_id]
    )

    frame["registry_version"] = (
        registry["schema_version"]
    )

    frame["calibration_version"] = (
        calibration["schema_version"]
    )

    frame["engine_version"] = (
        ENGINE_VERSION
    )

    source_output_frames.append(
        frame
    )


source_output = pd.concat(
    source_output_frames,
    ignore_index=True,
)

source_output["date"] = pd.to_datetime(
    source_output["date"]
)


# =============================================================================
# Layer aggregation
# =============================================================================

layer_records: list[
    dict[str, Any]
] = []

layer_contribution_records: list[
    dict[str, Any]
] = []

maximum_credit_per_cluster = float(
    calibration[
        "independence"
    ].get(
        "maximum_credit_per_cluster",
        1.0,
    )
)

agreement_config = (
    calibration["agreement"]
)

layer_confidence_weights = (
    calibration[
        "layer_confidence"
    ]["base_weights"]
)


print("\nCalculating Evidence Layers...")


for layer_name, layer_config in (
    layer_scoring.items()
):

    configured_weights = pd.Series(
        layer_config[
            "source_weights"
        ],
        dtype=float,
    )

    source_ids = list(
        configured_weights.index
    )

    source_scores = pd.DataFrame(
        {
            source_id: (
                source_frames[source_id][
                    "score"
                ]
            )
            for source_id in source_ids
        },
        index=features.index,
    )

    source_available = pd.DataFrame(
        {
            source_id: (
                source_frames[source_id][
                    "is_available"
                ]
            )
            for source_id in source_ids
        },
        index=features.index,
    )

    source_freshness = pd.DataFrame(
        {
            source_id: (
                source_frames[source_id][
                    "freshness_score"
                ]
            )
            for source_id in source_ids
        },
        index=features.index,
    )

    source_quality = pd.Series(
        {
            source_id: (
                float(
                    quality_factors[
                        registry_by_id[
                            source_id
                        ][
                            "quality_tier"
                        ]
                    ]
                )
                * 100.0
            )
            for source_id in source_ids
        },
        dtype=float,
    )

    source_clusters = {
        source_id: (
            registry_by_id[
                source_id
            ][
                "independence_cluster"
            ]
        )
        for source_id in source_ids
    }

    minimum_source_coverage = float(
        layer_config[
            "aggregation"
        ].get(
            "minimum_source_coverage",
            0.50,
        )
    )

    neutral_layer_score = float(
        layer_config[
            "aggregation"
        ].get(
            "neutral_score",
            50.0,
        )
    )

    total_configured_weight = float(
        configured_weights.sum()
    )

    for date in features.index:

        available_mask = (
            source_available.loc[date]
            & source_scores.loc[date].notna()
        )

        available_source_ids = [
            source_id
            for source_id in source_ids
            if bool(
                available_mask[
                    source_id
                ]
            )
        ]

        available_raw_weights = (
            configured_weights.loc[
                available_source_ids
            ]
        )

        source_coverage_ratio = (
            float(
                available_raw_weights.sum()
            )
            / total_configured_weight
            if total_configured_weight > 0
            else 0.0
        )

        layer_is_available = (
            len(available_source_ids) > 0
            and source_coverage_ratio
            >= minimum_source_coverage
        )

        if not available_source_ids:

            layer_records.append(
                {
                    "date": date,
                    "layer": layer_name,
                    "score": neutral_layer_score,
                    "confidence": 0.0,
                    "source_coverage": 0.0,
                    "independence_score": 0.0,
                    "agreement_score": 0.0,
                    "freshness_score": 0.0,
                    "source_quality_score": 0.0,
                    "available_source_count": 0,
                    "configured_source_count": (
                        len(source_ids)
                    ),
                    "available_cluster_count": 0,
                    "effective_source_count": 0.0,
                    "effective_cluster_count": 0.0,
                    "dominant_source_id": None,
                    "is_available": False,
                }
            )

            for source_id in source_ids:
                layer_contribution_records.append(
                    {
                        "date": date,
                        "record_type": (
                            "SOURCE_TO_LAYER"
                        ),
                        "layer": layer_name,
                        "source_id": source_id,
                        "descriptor": None,
                        "independence_cluster": (
                            source_clusters[
                                source_id
                            ]
                        ),
                        "raw_weight": float(
                            configured_weights[
                                source_id
                            ]
                        ),
                        "effective_weight": 0.0,
                        "input_score": float(
                            source_scores.loc[
                                date,
                                source_id,
                            ]
                        ),
                        "weighted_contribution": 0.0,
                        "is_available": False,
                    }
                )

            continue

        # ---------------------------------------------------------------------
        # Cluster-normalized score weights
        #
        # A duplicate cluster receives no more total weight than its strongest
        # member. This prevents multiple similar sources from inflating a layer.
        # ---------------------------------------------------------------------

        effective_weights = (
            available_raw_weights.copy()
        )

        cluster_to_sources: dict[
            str,
            list[str]
        ] = {}

        for source_id in available_source_ids:
            cluster = source_clusters[
                source_id
            ]

            cluster_to_sources.setdefault(
                cluster,
                [],
            ).append(source_id)

        for cluster, members in (
            cluster_to_sources.items()
        ):

            member_weights = (
                available_raw_weights.loc[
                    members
                ]
            )

            cluster_raw_total = float(
                member_weights.sum()
            )

            strongest_member_weight = float(
                member_weights.max()
            )

            cluster_budget = min(
                cluster_raw_total,
                strongest_member_weight
                * maximum_credit_per_cluster,
            )

            if cluster_raw_total > 0:
                effective_weights.loc[
                    members
                ] = (
                    member_weights
                    / cluster_raw_total
                    * cluster_budget
                )

        normalized_effective_weights = (
            effective_weights
            / effective_weights.sum()
        )

        available_scores = (
            source_scores.loc[
                date,
                available_source_ids,
            ]
            .astype(float)
        )

        layer_score = float(
            np.sum(
                available_scores.to_numpy()
                * normalized_effective_weights.to_numpy()
            )
        )

        # ---------------------------------------------------------------------
        # Agreement
        # ---------------------------------------------------------------------

        if len(available_source_ids) == 1:

            agreement_score = float(
                agreement_config[
                    "single_source_agreement_credit"
                ]
                * 100.0
            )

        else:

            median_score = weighted_median(
                values=(
                    available_scores.to_numpy()
                ),
                weights=(
                    normalized_effective_weights
                    .to_numpy()
                ),
            )

            absolute_deviation = np.abs(
                available_scores.to_numpy()
                - median_score
            )

            weighted_mad = float(
                np.sum(
                    absolute_deviation
                    * normalized_effective_weights
                    .to_numpy()
                )
            )

            maximum_dispersion = float(
                agreement_config[
                    "maximum_dispersion"
                ]
            )

            agreement_score = float(
                np.clip(
                    100.0
                    * (
                        1.0
                        - weighted_mad
                        / maximum_dispersion
                    ),
                    0.0,
                    100.0,
                )
            )

        # ---------------------------------------------------------------------
        # Independence
        # ---------------------------------------------------------------------

        source_weight_array = (
            available_raw_weights
            .to_numpy(dtype=float)
        )

        source_ess = (
            kish_effective_sample_size(
                source_weight_array
            )
        )

        cluster_weights = []

        for cluster, members in (
            cluster_to_sources.items()
        ):
            cluster_weights.append(
                float(
                    available_raw_weights.loc[
                        members
                    ].sum()
                )
            )

        cluster_ess = (
            kish_effective_sample_size(
                np.asarray(
                    cluster_weights,
                    dtype=float,
                )
            )
        )

        if source_ess > 0:
            independence_score = float(
                np.clip(
                    100.0
                    * (
                        cluster_ess
                        / source_ess
                    ),
                    0.0,
                    100.0,
                )
            )
        else:
            independence_score = 0.0

        # ---------------------------------------------------------------------
        # Freshness and quality
        # ---------------------------------------------------------------------

        layer_freshness_score = float(
            np.sum(
                source_freshness.loc[
                    date,
                    available_source_ids,
                ].to_numpy(dtype=float)
                * normalized_effective_weights
                .to_numpy(dtype=float)
            )
        )

        layer_source_quality = float(
            np.sum(
                source_quality.loc[
                    available_source_ids
                ].to_numpy(dtype=float)
                * normalized_effective_weights
                .to_numpy(dtype=float)
            )
        )

        base_layer_confidence = (
            float(
                layer_confidence_weights[
                    "independence"
                ]
            )
            * independence_score
            +
            float(
                layer_confidence_weights[
                    "agreement"
                ]
            )
            * agreement_score
            +
            float(
                layer_confidence_weights[
                    "freshness"
                ]
            )
            * layer_freshness_score
            +
            float(
                layer_confidence_weights[
                    "source_quality"
                ]
            )
            * layer_source_quality
        )

        layer_confidence = float(
            np.clip(
                base_layer_confidence
                * source_coverage_ratio,
                0.0,
                100.0,
            )
        )

        if not layer_is_available:
            layer_score = neutral_layer_score
            layer_confidence = 0.0

        dominant_source_id = str(
            normalized_effective_weights.idxmax()
        )

        layer_records.append(
            {
                "date": date,
                "layer": layer_name,
                "score": float(
                    np.clip(
                        layer_score,
                        0.0,
                        100.0,
                    )
                ),
                "confidence": layer_confidence,
                "source_coverage": float(
                    np.clip(
                        source_coverage_ratio,
                        0.0,
                        1.0,
                    )
                ),
                "independence_score": (
                    independence_score
                ),
                "agreement_score": (
                    agreement_score
                ),
                "freshness_score": (
                    layer_freshness_score
                ),
                "source_quality_score": (
                    layer_source_quality
                ),
                "available_source_count": (
                    len(available_source_ids)
                ),
                "configured_source_count": (
                    len(source_ids)
                ),
                "available_cluster_count": (
                    len(cluster_to_sources)
                ),
                "effective_source_count": (
                    source_ess
                ),
                "effective_cluster_count": (
                    cluster_ess
                ),
                "dominant_source_id": (
                    dominant_source_id
                ),
                "is_available": bool(
                    layer_is_available
                ),
            }
        )

        # Source → Layer contribution records
        for source_id in source_ids:

            source_is_available = (
                source_id
                in available_source_ids
            )

            effective_weight = (
                float(
                    normalized_effective_weights[
                        source_id
                    ]
                )
                if source_is_available
                else 0.0
            )

            input_score = float(
                source_scores.loc[
                    date,
                    source_id,
                ]
            )

            layer_contribution_records.append(
                {
                    "date": date,
                    "record_type": (
                        "SOURCE_TO_LAYER"
                    ),
                    "layer": layer_name,
                    "source_id": source_id,
                    "descriptor": None,
                    "independence_cluster": (
                        source_clusters[
                            source_id
                        ]
                    ),
                    "raw_weight": float(
                        configured_weights[
                            source_id
                        ]
                    ),
                    "effective_weight": (
                        effective_weight
                    ),
                    "input_score": input_score,
                    "weighted_contribution": (
                        input_score
                        * effective_weight
                    ),
                    "is_available": bool(
                        source_is_available
                    ),
                }
            )

    print(
        f"  Processed layer: {layer_name}"
    )


layer_output = pd.DataFrame(
    layer_records
)

layer_output["date"] = pd.to_datetime(
    layer_output["date"]
)

layer_output["registry_version"] = (
    registry["schema_version"]
)

layer_output["calibration_version"] = (
    calibration["schema_version"]
)

layer_output["engine_version"] = (
    ENGINE_VERSION
)


# =============================================================================
# Combined contribution table
# =============================================================================

descriptor_contributions = pd.concat(
    descriptor_contribution_frames,
    ignore_index=True,
)

layer_contributions = pd.DataFrame(
    layer_contribution_records
)

contribution_output = pd.concat(
    [
        descriptor_contributions,
        layer_contributions,
    ],
    ignore_index=True,
    sort=False,
)

contribution_output["date"] = pd.to_datetime(
    contribution_output["date"]
)

contribution_output["registry_version"] = (
    registry["schema_version"]
)

contribution_output["calibration_version"] = (
    calibration["schema_version"]
)

contribution_output["engine_version"] = (
    ENGINE_VERSION
)


# =============================================================================
# Point-in-time causality test for every transform family
# =============================================================================

transform_examples: dict[
    str,
    tuple[
        str,
        str,
        list[str],
        dict[str, Any],
    ]
] = {}

for source in registry_sources:

    source_id = source["source_id"]

    for descriptor_name, feature_names in (
        source["descriptors"].items()
    ):

        transform = (
            source_scoring[source_id][
                "transforms"
            ][descriptor_name]
        )

        transform_type = transform[
            "type"
        ]

        if transform_type not in transform_examples:
            transform_examples[
                transform_type
            ] = (
                source_id,
                descriptor_name,
                feature_names,
                transform,
            )


causality_results = {}

cutoff_position = max(
    int(
        len(features)
        * 0.80
    ),
    1,
)

truncated_features = features.iloc[
    :cutoff_position
].copy()


for transform_type, example in (
    transform_examples.items()
):

    (
        source_id,
        descriptor_name,
        feature_names,
        transform,
    ) = example

    full_result = (
        descriptor_score_cache[
            (
                source_id,
                descriptor_name,
            )
        ]
        .loc[
            truncated_features.index
        ]
    )

    truncated_result = transform_descriptor(
        feature_frame=truncated_features,
        feature_names=feature_names,
        transform=transform,
    )

    overlap = (
        full_result.notna()
        | truncated_result.notna()
    )

    if overlap.any():

        passed = bool(
            np.allclose(
                full_result.loc[
                    overlap
                ].to_numpy(dtype=float),
                truncated_result.loc[
                    overlap
                ].to_numpy(dtype=float),
                equal_nan=True,
                atol=1e-10,
                rtol=1e-10,
            )
        )

    else:
        passed = True

    causality_results[
        transform_type
    ] = passed


# =============================================================================
# Runtime validation
# =============================================================================

def numeric_columns_are_finite(
    frame: pd.DataFrame,
    columns: list[str],
) -> bool:

    values = (
        frame[columns]
        .to_numpy(dtype=float)
    )

    return bool(
        np.isfinite(
            values[
                ~np.isnan(values)
            ]
        ).all()
    )


def values_within_bounds(
    series: pd.Series,
    lower: float,
    upper: float,
) -> bool:

    usable = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()

    if usable.empty:
        return False

    return bool(
        (
            usable.between(
                lower,
                upper,
                inclusive="both",
            )
        ).all()
    )


latest_date = features.index.max()

latest_sources = (
    source_output.loc[
        source_output["date"]
        == latest_date
    ]
    .copy()
)

latest_layers = (
    layer_output.loc[
        layer_output["date"]
        == latest_date
    ]
    .copy()
)


runtime_tests = {
    "source_count_is_21": (
        source_output[
            "source_id"
        ].nunique()
        == 21
    ),

    "layer_count_is_7": (
        layer_output[
            "layer"
        ].nunique()
        == 7
    ),

    "source_row_count_complete": (
        len(source_output)
        == len(features)
        * len(registry_sources)
    ),

    "layer_row_count_complete": (
        len(layer_output)
        == len(features)
        * len(layer_scoring)
    ),

    "no_duplicate_date_source": (
        not source_output.duplicated(
            subset=[
                "date",
                "source_id",
            ]
        ).any()
    ),

    "no_duplicate_date_layer": (
        not layer_output.duplicated(
            subset=[
                "date",
                "layer",
            ]
        ).any()
    ),

    "source_scores_within_bounds": (
        values_within_bounds(
            source_output["score"],
            0.0,
            100.0,
        )
    ),

    "source_confidence_within_bounds": (
        values_within_bounds(
            source_output[
                "confidence"
            ],
            0.0,
            100.0,
        )
    ),

    "layer_scores_within_bounds": (
        values_within_bounds(
            layer_output["score"],
            0.0,
            100.0,
        )
    ),

    "layer_confidence_within_bounds": (
        values_within_bounds(
            layer_output[
                "confidence"
            ],
            0.0,
            100.0,
        )
    ),

    "coverage_within_bounds": (
        source_output[
            "descriptor_coverage"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        ).all()
        and
        layer_output[
            "source_coverage"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        ).all()
    ),

    "source_numeric_values_finite": (
        numeric_columns_are_finite(
            source_output,
            [
                "score",
                "confidence",
                "descriptor_coverage",
                "freshness_score",
                "stability_score",
                "quality_score",
            ],
        )
    ),

    "layer_numeric_values_finite": (
        numeric_columns_are_finite(
            layer_output,
            [
                "score",
                "confidence",
                "source_coverage",
                "independence_score",
                "agreement_score",
                "freshness_score",
                "source_quality_score",
            ],
        )
    ),

    "latest_sources_complete": (
        latest_sources[
            "source_id"
        ].nunique()
        == 21
    ),

    "latest_layers_complete": (
        latest_layers[
            "layer"
        ].nunique()
        == 7
    ),

    "registry_version_matches": (
        calibration[
            "target_registry_version"
        ]
        == registry[
            "schema_version"
        ]
    ),

    "coverage_counted_once": (
        "coverage"
        not in calibration[
            "source_confidence"
        ]["base_weights"]
        and
        "coverage"
        not in calibration[
            "layer_confidence"
        ]["base_weights"]
    ),

    "curated_ai_bias_flag_present": bool(
        registry_by_id[
            "ES_AI_LEADERSHIP"
        ][
            "curated_bias_flag"
        ]
    ),

    "all_transform_families_causal": (
        all(
            causality_results.values()
        )
    ),

    "v1_output_names_not_overwritten": (
        SOURCE_OUTPUT_PATH.name
        == "evidence_sources_v2.parquet"
        and
        LAYER_OUTPUT_PATH.name
        == "evidence_layers_v2.parquet"
    ),
}


overall_status = (
    "PASS"
    if all(
        runtime_tests.values()
    )
    else "FAIL"
)


# =============================================================================
# Quality summaries
# =============================================================================

source_first_available = {}

for source_id, group in source_output.groupby(
    "source_id"
):

    available_dates = group.loc[
        group["is_available"],
        "date",
    ]

    source_first_available[
        source_id
    ] = (
        available_dates.min()
        if not available_dates.empty
        else None
    )


layer_first_available = {}

for layer_name, group in layer_output.groupby(
    "layer"
):

    available_dates = group.loc[
        group["is_available"],
        "date",
    ]

    layer_first_available[
        layer_name
    ] = (
        available_dates.min()
        if not available_dates.empty
        else None
    )


quality_report = {
    "engine": ENGINE_NAME,
    "engine_version": ENGINE_VERSION,
    "overall_status": overall_status,
    "run_started_utc": RUN_STARTED,
    "run_finished_utc": (
        pd.Timestamp.now(tz="UTC")
    ),
    "feature_store": {
        "rows": int(features.shape[0]),
        "columns": int(features.shape[1]),
        "start_date": features.index.min(),
        "end_date": features.index.max(),
    },
    "registry_version": (
        registry["schema_version"]
    ),
    "calibration_version": (
        calibration["schema_version"]
    ),
    "source_count": int(
        source_output[
            "source_id"
        ].nunique()
    ),
    "layer_count": int(
        layer_output[
            "layer"
        ].nunique()
    ),
    "source_first_available_date": (
        source_first_available
    ),
    "layer_first_available_date": (
        layer_first_available
    ),
    "latest_source_availability": {
        source_id: bool(
            group["is_available"].iloc[0]
        )
        for source_id, group
        in latest_sources.groupby(
            "source_id"
        )
    },
    "latest_layer_availability": {
        layer_name: bool(
            group["is_available"].iloc[0]
        )
        for layer_name, group
        in latest_layers.groupby(
            "layer"
        )
    },
    "latest_layer_coverage": {
        row["layer"]: float(
            row["source_coverage"]
        )
        for _, row
        in latest_layers.iterrows()
    },
    "freshness_observation_mode": (
        macro_observation_mode
    ),
    "source_freshness_modes": (
        freshness_modes_used
    ),
    "causality_results": (
        causality_results
    ),
    "runtime_tests": runtime_tests,
    "warnings": runtime_warnings,
    "notes": [
        "Higher Evidence Score always means more risk-supportive.",
        "Strength and confidence remain separate fields.",
        "Unavailable evidence receives neutral score but zero confidence.",
        "Downstream engines must respect is_available and confidence.",
        "Macro transform logic is point-in-time, but revised FRED history "
        "is not a true historical vintage dataset.",
    ],
}


# =============================================================================
# Latest JSON
# =============================================================================

latest_source_records = {}

for _, row in latest_sources.iterrows():

    source_id = row[
        "source_id"
    ]

    latest_source_records[
        source_id
    ] = {
        "name": row[
            "source_name"
        ],
        "owner_layer": row[
            "owner_layer"
        ],
        "score": row["score"],
        "confidence": row[
            "confidence"
        ],
        "coverage": row[
            "descriptor_coverage"
        ],
        "freshness": row[
            "freshness_score"
        ],
        "stability": row[
            "stability_score"
        ],
        "quality_tier": row[
            "quality_tier"
        ],
        "is_available": row[
            "is_available"
        ],
        "curated_bias_flag": row[
            "curated_bias_flag"
        ],
    }


latest_layer_records = {}

for _, row in latest_layers.iterrows():

    layer_name = row["layer"]

    latest_layer_records[
        layer_name
    ] = {
        "score": row["score"],
        "confidence": row[
            "confidence"
        ],
        "coverage": row[
            "source_coverage"
        ],
        "independence": row[
            "independence_score"
        ],
        "agreement": row[
            "agreement_score"
        ],
        "freshness": row[
            "freshness_score"
        ],
        "quality": row[
            "source_quality_score"
        ],
        "dominant_source": row[
            "dominant_source_id"
        ],
        "is_available": row[
            "is_available"
        ],
    }


latest_report = {
    "engine": ENGINE_NAME,
    "engine_version": ENGINE_VERSION,
    "as_of_date": latest_date,
    "registry_version": (
        registry["schema_version"]
    ),
    "calibration_version": (
        calibration["schema_version"]
    ),
    "layers": latest_layer_records,
    "sources": latest_source_records,
    "warnings": runtime_warnings,
}


# =============================================================================
# Export outputs
# =============================================================================

source_output.to_parquet(
    SOURCE_OUTPUT_PATH,
    index=False,
)

layer_output.to_parquet(
    LAYER_OUTPUT_PATH,
    index=False,
)

contribution_output.to_parquet(
    CONTRIBUTION_OUTPUT_PATH,
    index=False,
)


with open(
    LATEST_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            latest_report
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


with open(
    QUALITY_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            quality_report
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


runtime_log = {
    "engine": ENGINE_NAME,
    "engine_version": ENGINE_VERSION,
    "status": overall_status,
    "run_started_utc": RUN_STARTED,
    "run_finished_utc": (
        pd.Timestamp.now(tz="UTC")
    ),
    "input_files": [
        str(FEATURE_PATH),
        str(REGISTRY_PATH),
        str(CALIBRATION_PATH),
        str(RAW_MACRO_PATH),
    ],
    "output_files": [
        str(SOURCE_OUTPUT_PATH),
        str(LAYER_OUTPUT_PATH),
        str(CONTRIBUTION_OUTPUT_PATH),
        str(LATEST_OUTPUT_PATH),
        str(QUALITY_OUTPUT_PATH),
        str(LOG_OUTPUT_PATH),
        str(TEST_OUTPUT_PATH),
    ],
    "row_counts": {
        "evidence_sources": int(
            len(source_output)
        ),
        "evidence_layers": int(
            len(layer_output)
        ),
        "evidence_contributions": int(
            len(contribution_output)
        ),
    },
    "warnings": runtime_warnings,
    "tests": runtime_tests,
}


with open(
    LOG_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            runtime_log
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# =============================================================================
# Test report
# =============================================================================

test_lines = [
    "=" * 88,
    "IVMOS STEP 6 v2 — EVIDENCE ENGINE RUNTIME TESTS",
    "=" * 88,
    f"Overall Status: {overall_status}",
    f"Engine Version: {ENGINE_VERSION}",
    f"Registry Version: {registry['schema_version']}",
    f"Calibration Version: {calibration['schema_version']}",
    f"Date Range: {features.index.min().date()} to {features.index.max().date()}",
    f"Evidence Sources: {source_output['source_id'].nunique()}",
    f"Evidence Layers: {layer_output['layer'].nunique()}",
    "",
    "Tests:",
]

for test_name, passed in runtime_tests.items():

    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {test_name}"
    )


test_lines.extend([
    "",
    "Transform causality:",
])

for transform_type, passed in causality_results.items():

    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {transform_type}"
    )


if runtime_warnings:

    test_lines.extend([
        "",
        "Warnings:",
        *[
            f"  - {warning}"
            for warning in runtime_warnings
        ],
    ])


test_lines.extend([
    "",
    "Files created:",
    f"  {SOURCE_OUTPUT_PATH}",
    f"  {LAYER_OUTPUT_PATH}",
    f"  {CONTRIBUTION_OUTPUT_PATH}",
    f"  {LATEST_OUTPUT_PATH}",
    f"  {QUALITY_OUTPUT_PATH}",
    f"  {LOG_OUTPUT_PATH}",
    f"  {TEST_OUTPUT_PATH}",
    "=" * 88,
])


TEST_OUTPUT_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Final runtime report
# =============================================================================

print("\n" + "=" * 88)
print("IVMOS STEP 6 v2 — EVIDENCE ENGINE RUNTIME")
print("=" * 88)
print("Overall Status:", overall_status)
print("Engine Version:", ENGINE_VERSION)
print("Evidence Source Rows:", len(source_output))
print("Evidence Layer Rows:", len(layer_output))
print(
    "Contribution Rows:",
    len(contribution_output),
)

print("\nRuntime tests:")

for test_name, passed in runtime_tests.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {test_name}"
    )


print("\nTransform causality:")

for transform_type, passed in causality_results.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {transform_type}"
    )


latest_layer_display = (
    latest_layers[
        [
            "layer",
            "score",
            "confidence",
            "source_coverage",
            "independence_score",
            "agreement_score",
            "dominant_source_id",
            "is_available",
        ]
    ]
    .sort_values("layer")
    .copy()
)

for column in [
    "score",
    "confidence",
    "source_coverage",
    "independence_score",
    "agreement_score",
]:
    latest_layer_display[column] = (
        latest_layer_display[column]
        .round(2)
    )


print(
    "\nLatest Evidence Layers "
    f"({latest_date.date()}):"
)

print(
    latest_layer_display.to_string(
        index=False
    )
)


if runtime_warnings:

    print("\nWarnings:")

    for warning in runtime_warnings:
        print(" ", warning)


print("\nFiles created:")

for path in runtime_log[
    "output_files"
]:
    print(" ", path)

print("=" * 88)


Feature store loaded
Shape: (2916, 2843)
Date range: 2015-01-02 to 2026-08-06

Evidence configuration loaded
Registry version: 0.1.0-draft
Calibration version: 0.1.0-draft
Evidence sources: 21
Evidence layers: 7

Calculating Evidence Sources...
  Processed 5/21 sources
  Processed 10/21 sources
  Processed 15/21 sources
  Processed 20/21 sources
  Processed 21/21 sources

Calculating Evidence Layers...
  Processed layer: TREND
  Processed layer: PARTICIPATION
  Processed layer: ROTATION
  Processed layer: LEADERSHIP
  Processed layer: CREDIT
  Processed layer: LIQUIDITY
  Processed layer: STRESS_BUFFER

IVMOS STEP 6 v2 — EVIDENCE ENGINE RUNTIME
Overall Status: PASS
Engine Version: 0.1.0-runtime
Evidence Source Rows: 61236
Evidence Layer Rows: 20412
Contribution Rows: 169128

Runtime tests:
  PASS — source_count_is_21
  PASS — layer_count_is_7
  PASS — source_row_count_complete
  PASS — layer_row_count_complete
  PASS — no_duplicate_date_source
  PASS — no_duplicate_date_layer
  PASS — 

In [11]:
# =============================================================================
# IVMOS STEP 6 v2.1
# Confidence Calibration Patch
#
# Changes
# -------
# 1. Preserve every Evidence Score from v2
# 2. Recalculate Source Confidence using descriptor breadth
# 3. Recalculate Layer Confidence using patched source confidence
# 4. Replace near-degenerate independence credit with evidence breadth
# 5. Apply agreement and effective-evidence ceilings
# 6. Create separate v2.1 outputs without overwriting v2
# =============================================================================

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_source_registry_v2_draft.yaml"
)

BASE_CALIBRATION_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_calibration_v2_draft.yaml"
)

SOURCE_V2_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_sources_v2.parquet"
)

LAYER_V2_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_layers_v2.parquet"
)

CONTRIBUTION_V2_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_contributions_v2.parquet"
)


PATCHED_CALIBRATION_PATH = (
    PROJECT_ROOT
    / "config"
    / "evidence_calibration_v2_1_draft.yaml"
)

SOURCE_V21_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_sources_v2_1.parquet"
)

LAYER_V21_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_layers_v2_1.parquet"
)

CONTRIBUTION_V21_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_contributions_v2_1.parquet"
)

AUDIT_CSV_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_1_confidence_audit.csv"
)

LATEST_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_1_latest.json"
)

AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "evidence_v2_1_confidence_audit.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step6_v2_1_confidence_patch.txt"
)

for directory in [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "processed",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# Runtime contract
# =============================================================================

ENGINE_VERSION = "0.2.0-confidence-patch"
CALIBRATION_VERSION = "0.2.0-draft"


# =============================================================================
# Helpers
# =============================================================================

def normalize_date(
    values: pd.Series,
) -> pd.Series:

    result = pd.to_datetime(
        values,
        errors="coerce",
    )

    try:
        result = result.dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    return result.dt.normalize()


def json_safe(value: Any) -> Any:

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value

    if pd.isna(value):
        return None

    return value


def bounded_series(
    series: pd.Series,
    lower: float = 0.0,
    upper: float = 100.0,
) -> pd.Series:

    return (
        pd.to_numeric(
            series,
            errors="coerce",
        )
        .clip(
            lower=lower,
            upper=upper,
        )
    )


def saturating_breadth_score(
    effective_count: pd.Series,
    decay: float,
) -> pd.Series:
    """
    Smooth evidence breadth.

    Approximate values with decay=0.55:
      1 effective cluster  -> 42
      2 effective clusters -> 67
      3 effective clusters -> 81
      5 effective clusters -> 94
    """

    count = (
        pd.to_numeric(
            effective_count,
            errors="coerce",
        )
        .fillna(0.0)
        .clip(lower=0.0)
    )

    return (
        100.0
        * (
            1.0
            - np.exp(
                -decay * count
            )
        )
    ).clip(
        0.0,
        100.0,
    )


# =============================================================================
# Load inputs
# =============================================================================

required_paths = [
    REGISTRY_PATH,
    BASE_CALIBRATION_PATH,
    SOURCE_V2_PATH,
    LAYER_V2_PATH,
    CONTRIBUTION_V2_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )


with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    registry = yaml.safe_load(file)


with open(
    BASE_CALIBRATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    base_calibration = yaml.safe_load(file)


sources = pd.read_parquet(
    SOURCE_V2_PATH
)

layers = pd.read_parquet(
    LAYER_V2_PATH
)

contributions = pd.read_parquet(
    CONTRIBUTION_V2_PATH
)


for frame in [
    sources,
    layers,
    contributions,
]:
    frame["date"] = normalize_date(
        frame["date"]
    )


registry_sources = registry["sources"]

registry_by_id = {
    source["source_id"]: source
    for source in registry_sources
}

descriptor_count_map = {
    source["source_id"]: len(
        source.get(
            "descriptors",
            {},
        )
    )
    for source in registry_sources
}


print("Inputs loaded")
print("Source rows:", len(sources))
print("Layer rows:", len(layers))
print("Contribution rows:", len(contributions))


# =============================================================================
# 1. Patch Source Confidence
#
# Source confidence components:
#   35% source quality
#   25% freshness
#   20% stability
#   20% descriptor breadth
#
# Coverage is multiplied once.
# =============================================================================

patched_sources = sources.copy()

patched_sources[
    "confidence_v2_original"
] = patched_sources[
    "confidence"
].astype(float)


patched_sources[
    "descriptor_count"
] = (
    patched_sources[
        "source_id"
    ]
    .map(
        descriptor_count_map
    )
    .fillna(0)
    .astype(int)
)


patched_sources[
    "descriptor_breadth_score"
] = saturating_breadth_score(
    effective_count=patched_sources[
        "descriptor_count"
    ],
    decay=0.70,
)


source_base_confidence = (
    0.35
    * bounded_series(
        patched_sources[
            "quality_score"
        ]
    )
    +
    0.25
    * bounded_series(
        patched_sources[
            "freshness_score"
        ]
    )
    +
    0.20
    * bounded_series(
        patched_sources[
            "stability_score"
        ]
    )
    +
    0.20
    * bounded_series(
        patched_sources[
            "descriptor_breadth_score"
        ]
    )
)


source_coverage = (
    pd.to_numeric(
        patched_sources[
            "descriptor_coverage"
        ],
        errors="coerce",
    )
    .fillna(0.0)
    .clip(
        0.0,
        1.0,
    )
)


source_confidence_raw = (
    source_base_confidence
    * source_coverage
)


# More descriptors can justify a higher ceiling,
# but no single source reaches 100.
source_ceiling = (
    82.0
    +
    4.0
    * np.log2(
        patched_sources[
            "descriptor_count"
        ]
        .clip(lower=1)
    )
).clip(
    upper=92.0
)


# Curated AI basket remains limited because it is not
# valid independent historical confirmation.
curated_mask = (
    patched_sources[
        "curated_bias_flag"
    ]
    .fillna(False)
    .astype(bool)
)

source_ceiling.loc[
    curated_mask
] = np.minimum(
    source_ceiling.loc[
        curated_mask
    ],
    70.0,
)


# Structurally sensitive macro sources retain a lower ceiling
# until a structural-break detector is implemented.
structural_sensitive_mask = (
    patched_sources[
        "structural_break_sensitive"
    ]
    .fillna(False)
    .astype(bool)
)

source_ceiling.loc[
    structural_sensitive_mask
] = np.minimum(
    source_ceiling.loc[
        structural_sensitive_mask
    ],
    88.0,
)


patched_source_confidence = pd.concat(
    [
        source_confidence_raw.rename(
            "raw"
        ),
        source_ceiling.rename(
            "ceiling"
        ),
    ],
    axis=1,
).min(axis=1)


source_available = (
    patched_sources[
        "is_available"
    ]
    .fillna(False)
    .astype(bool)
)


patched_source_confidence = (
    patched_source_confidence
    .where(
        source_available,
        0.0,
    )
    .clip(
        0.0,
        100.0,
    )
)


patched_sources[
    "confidence"
] = patched_source_confidence

patched_sources[
    "confidence_ceiling"
] = source_ceiling

patched_sources[
    "confidence_patch_delta"
] = (
    patched_sources[
        "confidence"
    ]
    -
    patched_sources[
        "confidence_v2_original"
    ]
)

patched_sources[
    "base_calibration_version"
] = patched_sources[
    "calibration_version"
]

patched_sources[
    "calibration_version"
] = CALIBRATION_VERSION

patched_sources[
    "engine_version"
] = ENGINE_VERSION


# =============================================================================
# 2. Calculate Layer Input Confidence
#
# Uses existing Source-to-Layer effective weights.
# Layer scores and contribution weights are not changed.
# =============================================================================

source_to_layer = contributions.loc[
    contributions[
        "record_type"
    ].eq(
        "SOURCE_TO_LAYER"
    )
].copy()


source_to_layer = (
    source_to_layer.merge(
        patched_sources[
            [
                "date",
                "source_id",
                "confidence",
            ]
        ].rename(
            columns={
                "confidence": (
                    "patched_source_confidence"
                )
            }
        ),
        on=[
            "date",
            "source_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


source_to_layer[
    "effective_weight"
] = (
    pd.to_numeric(
        source_to_layer[
            "effective_weight"
        ],
        errors="coerce",
    )
    .fillna(0.0)
)


source_to_layer[
    "weighted_source_confidence"
] = (
    source_to_layer[
        "patched_source_confidence"
    ].fillna(0.0)
    *
    source_to_layer[
        "effective_weight"
    ]
)


layer_input_confidence = (
    source_to_layer.groupby(
        [
            "date",
            "layer",
        ],
        as_index=False,
    )
    .agg(
        weighted_confidence_sum=(
            "weighted_source_confidence",
            "sum",
        ),
        effective_weight_sum=(
            "effective_weight",
            "sum",
        ),
    )
)


layer_input_confidence[
    "input_confidence"
] = (
    layer_input_confidence[
        "weighted_confidence_sum"
    ]
    /
    layer_input_confidence[
        "effective_weight_sum"
    ].replace(
        0.0,
        np.nan,
    )
)


# =============================================================================
# 3. Patch Layer Confidence
#
# Components:
#   35% patched source confidence
#   30% agreement
#   15% effective evidence breadth
#   10% freshness
#   10% source quality
#
# Then:
#   × source coverage
#
# Ceilings:
#   - effective evidence ceiling
#   - agreement + 18
#   - input confidence + 10
#   - absolute maximum 94
# =============================================================================

patched_layers = layers.copy()

patched_layers[
    "confidence_v2_original"
] = patched_layers[
    "confidence"
].astype(float)


patched_layers = (
    patched_layers.merge(
        layer_input_confidence[
            [
                "date",
                "layer",
                "input_confidence",
            ]
        ],
        on=[
            "date",
            "layer",
        ],
        how="left",
        validate="one_to_one",
    )
)


patched_layers[
    "evidence_breadth_score"
] = saturating_breadth_score(
    effective_count=patched_layers[
        "effective_cluster_count"
    ],
    decay=0.55,
)


layer_base_confidence = (
    0.35
    * bounded_series(
        patched_layers[
            "input_confidence"
        ]
    )
    +
    0.30
    * bounded_series(
        patched_layers[
            "agreement_score"
        ]
    )
    +
    0.15
    * bounded_series(
        patched_layers[
            "evidence_breadth_score"
        ]
    )
    +
    0.10
    * bounded_series(
        patched_layers[
            "freshness_score"
        ]
    )
    +
    0.10
    * bounded_series(
        patched_layers[
            "source_quality_score"
        ]
    )
)


layer_coverage = (
    pd.to_numeric(
        patched_layers[
            "source_coverage"
        ],
        errors="coerce",
    )
    .fillna(0.0)
    .clip(
        0.0,
        1.0,
    )
)


layer_confidence_raw = (
    layer_base_confidence
    * layer_coverage
)


effective_cluster_count = (
    pd.to_numeric(
        patched_layers[
            "effective_cluster_count"
        ],
        errors="coerce",
    )
    .fillna(0.0)
    .clip(lower=0.0)
)


# Smooth ceiling:
#   1 effective source  ≈ 74
#   2 effective sources ≈ 81
#   3 effective sources ≈ 86
#   5 effective sources ≈ 93
evidence_ceiling = (
    62.0
    +
    12.0
    * np.log2(
        1.0
        + effective_cluster_count
    )
).clip(
    lower=0.0,
    upper=94.0,
)


agreement_ceiling = (
    bounded_series(
        patched_layers[
            "agreement_score"
        ]
    )
    + 18.0
).clip(
    upper=94.0,
)


input_confidence_ceiling = (
    bounded_series(
        patched_layers[
            "input_confidence"
        ]
    )
    + 10.0
).clip(
    upper=94.0,
)


absolute_ceiling = pd.Series(
    94.0,
    index=patched_layers.index,
)


patched_layer_confidence = pd.concat(
    [
        layer_confidence_raw.rename(
            "raw"
        ),
        evidence_ceiling.rename(
            "evidence_ceiling"
        ),
        agreement_ceiling.rename(
            "agreement_ceiling"
        ),
        input_confidence_ceiling.rename(
            "input_ceiling"
        ),
        absolute_ceiling.rename(
            "absolute_ceiling"
        ),
    ],
    axis=1,
).min(axis=1)


layer_available = (
    patched_layers[
        "is_available"
    ]
    .fillna(False)
    .astype(bool)
)


patched_layer_confidence = (
    patched_layer_confidence
    .where(
        layer_available,
        0.0,
    )
    .clip(
        0.0,
        100.0,
    )
)


patched_layers[
    "confidence"
] = patched_layer_confidence


# =============================================================================
# CREDIT FULL / PROXY AVAILABILITY MODE
# =============================================================================

patched_layers["evidence_mode"] = "STANDARD"
patched_layers["primary_source_available"] = pd.NA
patched_layers["proxy_source_available"] = pd.NA

hy_available_by_date = (
    patched_sources.loc[
        patched_sources["source_id"].eq("ES_HY_SPREAD"),
        ["date", "is_available"],
    ]
    .drop_duplicates("date", keep="last")
    .set_index("date")["is_available"]
)

hyg_lqd_available_by_date = (
    patched_sources.loc[
        patched_sources["source_id"].eq("ES_HYG_LQD"),
        ["date", "is_available"],
    ]
    .drop_duplicates("date", keep="last")
    .set_index("date")["is_available"]
)

credit_mask = patched_layers["layer"].eq("CREDIT")
credit_indices = patched_layers.index[credit_mask]

credit_primary_available = (
    patched_layers.loc[credit_mask, "date"]
    .map(hy_available_by_date)
    .fillna(False)
    .astype(bool)
)

credit_proxy_available = (
    patched_layers.loc[credit_mask, "date"]
    .map(hyg_lqd_available_by_date)
    .fillna(False)
    .astype(bool)
)

patched_layers.loc[
    credit_mask,
    "primary_source_available",
] = credit_primary_available.to_numpy()

patched_layers.loc[
    credit_mask,
    "proxy_source_available",
] = credit_proxy_available.to_numpy()

full_mode = credit_primary_available & credit_proxy_available
proxy_mode = ~credit_primary_available & credit_proxy_available
primary_only_mode = credit_primary_available & ~credit_proxy_available
unavailable_mode = ~credit_primary_available & ~credit_proxy_available

patched_layers.loc[
    credit_indices[full_mode.to_numpy()],
    "evidence_mode",
] = "FULL"

patched_layers.loc[
    credit_indices[proxy_mode.to_numpy()],
    "evidence_mode",
] = "PROXY"

patched_layers.loc[
    credit_indices[primary_only_mode.to_numpy()],
    "evidence_mode",
] = "PRIMARY_ONLY"

patched_layers.loc[
    credit_indices[unavailable_mode.to_numpy()],
    "evidence_mode",
] = "UNAVAILABLE"

PROXY_CONFIDENCE_CEILING = 65.0
proxy_indices = credit_indices[proxy_mode.to_numpy()]

patched_layers.loc[
    proxy_indices,
    "confidence",
] = np.minimum(
    patched_layers.loc[proxy_indices, "confidence"],
    PROXY_CONFIDENCE_CEILING,
)


patched_layers[
    "confidence_raw_v2_1"
] = layer_confidence_raw

patched_layers[
    "evidence_confidence_ceiling"
] = evidence_ceiling

patched_layers[
    "agreement_confidence_ceiling"
] = agreement_ceiling

patched_layers[
    "input_confidence_ceiling"
] = input_confidence_ceiling

patched_layers[
    "confidence_patch_delta"
] = (
    patched_layers[
        "confidence"
    ]
    -
    patched_layers[
        "confidence_v2_original"
    ]
)

patched_layers[
    "base_calibration_version"
] = patched_layers[
    "calibration_version"
]

patched_layers[
    "calibration_version"
] = CALIBRATION_VERSION

patched_layers[
    "engine_version"
] = ENGINE_VERSION


# =============================================================================
# 4. Version contribution output
# =============================================================================

patched_contributions = contributions.copy()

patched_contributions[
    "base_calibration_version"
] = patched_contributions[
    "calibration_version"
]

patched_contributions[
    "calibration_version"
] = CALIBRATION_VERSION

patched_contributions[
    "engine_version"
] = ENGINE_VERSION


# =============================================================================
# 5. Create patched calibration document
# =============================================================================

patched_calibration = dict(
    base_calibration
)

patched_calibration[
    "schema_version"
] = CALIBRATION_VERSION

patched_calibration[
    "base_schema_version"
] = base_calibration.get(
    "schema_version"
)

patched_calibration[
    "target_engine_version"
] = ENGINE_VERSION


patched_calibration[
    "source_confidence_v2_1"
] = {
    "base_weights": {
        "source_quality": 0.35,
        "freshness": 0.25,
        "stability": 0.20,
        "descriptor_breadth": 0.20,
    },
    "coverage_multiplier": True,
    "descriptor_breadth_decay": 0.70,
    "general_ceiling_formula": (
        "min(92, 82 + 4 * log2(descriptor_count))"
    ),
    "curated_source_ceiling": 70.0,
    "structural_sensitive_ceiling": 88.0,
}


patched_calibration[
    "layer_confidence_v2_1"
] = {
    "base_weights": {
        "input_confidence": 0.35,
        "agreement": 0.30,
        "evidence_breadth": 0.15,
        "freshness": 0.10,
        "source_quality": 0.10,
    },
    "coverage_multiplier": True,
    "evidence_breadth_decay": 0.55,
    "ceilings": {
        "effective_evidence": (
            "min(94, 62 + 12 * log2(1 + effective_cluster_count))"
        ),
        "agreement": "agreement_score + 18",
        "input_confidence": (
            "input_confidence + 10"
        ),
        "absolute": 94.0,
    },
}


patched_calibration[
    "metadata"
] = {
    **patched_calibration.get(
        "metadata",
        {},
    ),
    "status": (
        "CONFIDENCE_PATCH_REQUIRES_AUDIT"
    ),
    "credit_availability_policy": {
        "full_mode": ["ES_HY_SPREAD", "ES_HYG_LQD"],
        "proxy_mode": ["ES_HYG_LQD"],
        "proxy_confidence_ceiling": 65.0,
    },
    "patch_reason": [
        "Historical mean confidence exceeded 85 in five layers.",
        "Liquidity confidence exceeded agreement by more than 20.",
        "Raw independence ratio provided insufficient confidence discrimination.",
        "Evidence scores remain unchanged.",
    ],
}


with open(
    PATCHED_CALIBRATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        patched_calibration,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=120,
    )


# =============================================================================
# 6. Audit v2 versus v2.1
# =============================================================================

confidence_audit = (
    patched_layers.groupby(
        "layer",
        as_index=False,
    )
    .agg(
        confidence_mean_v2=(
            "confidence_v2_original",
            "mean",
        ),
        confidence_mean_v2_1=(
            "confidence",
            "mean",
        ),
        confidence_median_v2_1=(
            "confidence",
            "median",
        ),
        confidence_p95_v2_1=(
            "confidence",
            lambda values: values.quantile(
                0.95
            ),
        ),
        agreement_mean=(
            "agreement_score",
            "mean",
        ),
        evidence_breadth_mean=(
            "evidence_breadth_score",
            "mean",
        ),
        input_confidence_mean=(
            "input_confidence",
            "mean",
        ),
        patch_delta_mean=(
            "confidence_patch_delta",
            "mean",
        ),
    )
)


confidence_audit[
    "confidence_minus_agreement_v2_1"
] = (
    confidence_audit[
        "confidence_mean_v2_1"
    ]
    -
    confidence_audit[
        "agreement_mean"
    ]
)


high_confidence_ratios = (
    patched_layers.assign(
        high_85=(
            patched_layers[
                "confidence"
            ]
            >= 85.0
        ),
        high_90=(
            patched_layers[
                "confidence"
            ]
            >= 90.0
        ),
    )
    .groupby(
        "layer",
        as_index=False,
    )
    .agg(
        high_confidence_85_ratio_v2_1=(
            "high_85",
            "mean",
        ),
        high_confidence_90_ratio_v2_1=(
            "high_90",
            "mean",
        ),
    )
)


confidence_audit = (
    confidence_audit.merge(
        high_confidence_ratios,
        on="layer",
        how="left",
        validate="one_to_one",
    )
)


# =============================================================================
# 7. Tests
# =============================================================================

source_scores_unchanged = bool(
    np.allclose(
        sources[
            "score"
        ].to_numpy(dtype=float),
        patched_sources[
            "score"
        ].to_numpy(dtype=float),
        equal_nan=True,
        atol=0.0,
        rtol=0.0,
    )
)


layer_scores_unchanged = bool(
    np.allclose(
        layers[
            "score"
        ].to_numpy(dtype=float),
        patched_layers[
            "score"
        ].to_numpy(dtype=float),
        equal_nan=True,
        atol=0.0,
        rtol=0.0,
    )
)


row_level_agreement_gap = (
    patched_layers[
        "confidence"
    ]
    -
    patched_layers[
        "agreement_score"
    ]
)


tests = {
    "source_scores_unchanged": (
        source_scores_unchanged
    ),

    "layer_scores_unchanged": (
        layer_scores_unchanged
    ),

    "source_confidence_within_bounds": bool(
        patched_sources[
            "confidence"
        ].between(
            0.0,
            100.0,
            inclusive="both",
        ).all()
    ),

    "layer_confidence_within_bounds": bool(
        patched_layers[
            "confidence"
        ].between(
            0.0,
            100.0,
            inclusive="both",
        ).all()
    ),

    "source_confidence_never_reaches_100": bool(
        patched_sources[
            "confidence"
        ].max()
        < 100.0
    ),

    "layer_confidence_never_exceeds_94": bool(
        patched_layers[
            "confidence"
        ].max()
        <= 94.0
    ),

    "mean_layer_confidence_not_above_85": bool(
        (
            confidence_audit[
                "confidence_mean_v2_1"
            ]
            <= 85.0
        ).all()
    ),

    "row_level_confidence_agreement_gap_not_above_18": bool(
        (
            row_level_agreement_gap
            <= 18.0 + 1e-9
        ).all()
    ),

    "curated_ai_source_confidence_not_above_70": bool(
        (
            patched_sources.loc[
                patched_sources[
                    "source_id"
                ].eq(
                    "ES_AI_LEADERSHIP"
                ),
                "confidence",
            ]
            <= 70.0
        ).all()
    ),

    "latest_sources_complete": bool(
        patched_sources.loc[
            patched_sources[
                "date"
            ].eq(
                patched_sources[
                    "date"
                ].max()
            ),
            "source_id",
        ].nunique()
        == 21
    ),

    "latest_layers_complete": bool(
        patched_layers.loc[
            patched_layers[
                "date"
            ].eq(
                patched_layers[
                    "date"
                ].max()
            ),
            "layer",
        ].nunique()
        == 7
    ),

    "no_duplicate_date_source": bool(
        not patched_sources.duplicated(
            [
                "date",
                "source_id",
            ]
        ).any()
    ),

    "no_duplicate_date_layer": bool(
        not patched_layers.duplicated(
            [
                "date",
                "layer",
            ]
        ).any()
    ),

    "v2_outputs_not_overwritten": bool(
        SOURCE_V21_PATH.name
        != SOURCE_V2_PATH.name
        and
        LAYER_V21_PATH.name
        != LAYER_V2_PATH.name
    ),
}


overall_status = (
    "PASS"
    if all(tests.values())
    else "FAIL"
)


# =============================================================================
# 8. Latest report
# =============================================================================

latest_date = patched_layers[
    "date"
].max()


latest_layers = (
    patched_layers.loc[
        patched_layers[
            "date"
        ].eq(
            latest_date
        )
    ]
    .copy()
    .sort_values(
        "layer"
    )
)


latest_sources = (
    patched_sources.loc[
        patched_sources[
            "date"
        ].eq(
            latest_date
        )
    ]
    .copy()
    .sort_values(
        [
            "owner_layer",
            "source_id",
        ]
    )
)


latest_report = {
    "engine": (
        "IVMOS Evidence Engine v2.1"
    ),
    "engine_version": ENGINE_VERSION,
    "calibration_version": (
        CALIBRATION_VERSION
    ),
    "as_of_date": latest_date,
    "overall_status": overall_status,
    "layers": {
        row["layer"]: {
            "score": row["score"],
            "confidence": row[
                "confidence"
            ],
            "confidence_v2_original": row[
                "confidence_v2_original"
            ],
            "agreement": row[
                "agreement_score"
            ],
            "evidence_breadth": row[
                "evidence_breadth_score"
            ],
            "input_confidence": row[
                "input_confidence"
            ],
            "coverage": row[
                "source_coverage"
            ],
            "dominant_source": row[
                "dominant_source_id"
            ],
        }
        for _, row
        in latest_layers.iterrows()
    },
}


# =============================================================================
# 9. Export
# =============================================================================

patched_sources.to_parquet(
    SOURCE_V21_PATH,
    index=False,
)

patched_layers.to_parquet(
    LAYER_V21_PATH,
    index=False,
)

patched_contributions.to_parquet(
    CONTRIBUTION_V21_PATH,
    index=False,
)

confidence_audit.to_csv(
    AUDIT_CSV_PATH,
    index=False,
)


with open(
    LATEST_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            latest_report
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


audit_report = {
    "engine": (
        "IVMOS Evidence Engine v2.1 "
        "Confidence Patch"
    ),
    "overall_status": overall_status,
    "engine_version": ENGINE_VERSION,
    "calibration_version": (
        CALIBRATION_VERSION
    ),
    "tests": tests,
    "layer_confidence_audit": (
        confidence_audit.to_dict(
            orient="records"
        )
    ),
    "files_created": [
        str(PATCHED_CALIBRATION_PATH),
        str(SOURCE_V21_PATH),
        str(LAYER_V21_PATH),
        str(CONTRIBUTION_V21_PATH),
        str(AUDIT_CSV_PATH),
        str(LATEST_JSON_PATH),
        str(AUDIT_JSON_PATH),
        str(TEST_PATH),
    ],
}


with open(
    AUDIT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            audit_report
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


test_lines = [
    "=" * 96,
    "IVMOS STEP 6 v2.1 — CONFIDENCE PATCH",
    "=" * 96,
    f"Overall Status: {overall_status}",
    f"Engine Version: {ENGINE_VERSION}",
    f"Calibration Version: {CALIBRATION_VERSION}",
    f"Latest Date: {latest_date.date()}",
    "",
    "Tests:",
]

for test_name, passed in tests.items():
    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — "
        f"{test_name}"
    )


test_lines.extend([
    "",
    "Files created:",
    *[
        f"  {path}"
        for path
        in audit_report[
            "files_created"
        ]
    ],
    "=" * 96,
])


TEST_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Final report
# =============================================================================

display_audit = confidence_audit.copy()

numeric_columns = (
    display_audit.select_dtypes(
        include=[np.number]
    ).columns
)

display_audit[
    numeric_columns
] = (
    display_audit[
        numeric_columns
    ].round(3)
)


latest_display = latest_layers[
    [
        "layer",
        "score",
        "confidence_v2_original",
        "confidence",
        "agreement_score",
        "evidence_breadth_score",
        "input_confidence",
        "source_coverage",
        "dominant_source_id",
    ]
].copy()


for column in [
    "score",
    "confidence_v2_original",
    "confidence",
    "agreement_score",
    "evidence_breadth_score",
    "input_confidence",
    "source_coverage",
]:
    latest_display[
        column
    ] = (
        latest_display[
            column
        ].round(2)
    )


print("\n" + "=" * 96)
print("IVMOS STEP 6 v2.1 — CONFIDENCE PATCH")
print("=" * 96)
print("Overall Status:", overall_status)
print("Engine Version:", ENGINE_VERSION)
print(
    "Calibration Version:",
    CALIBRATION_VERSION,
)

print("\nTests:")

for test_name, passed in tests.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — "
        f"{test_name}"
    )


print("\nHistorical confidence audit:")
print(
    display_audit.to_string(
        index=False
    )
)


print(
    f"\nLatest Layers ({latest_date.date()}):"
)

print(
    latest_display.to_string(
        index=False
    )
)


print("\nFiles created:")

for path in audit_report[
    "files_created"
]:
    print(" ", path)

print("=" * 96)


Inputs loaded
Source rows: 61236
Layer rows: 20412
Contribution rows: 169128

IVMOS STEP 6 v2.1 — CONFIDENCE PATCH
Overall Status: PASS
Engine Version: 0.2.0-confidence-patch
Calibration Version: 0.2.0-draft

Tests:
  PASS — source_scores_unchanged
  PASS — layer_scores_unchanged
  PASS — source_confidence_within_bounds
  PASS — layer_confidence_within_bounds
  PASS — source_confidence_never_reaches_100
  PASS — layer_confidence_never_exceeds_94
  PASS — mean_layer_confidence_not_above_85
  PASS — row_level_confidence_agreement_gap_not_above_18
  PASS — curated_ai_source_confidence_not_above_70
  PASS — latest_sources_complete
  PASS — latest_layers_complete
  PASS — no_duplicate_date_source
  PASS — no_duplicate_date_layer
  PASS — v2_outputs_not_overwritten

Historical confidence audit:
        layer  confidence_mean_v2  confidence_mean_v2_1  confidence_median_v2_1  confidence_p95_v2_1  agreement_mean  evidence_breadth_mean  input_confidence_mean  patch_delta_mean  confidence_minus_a

## Final Validation


In [12]:
# =============================================================================
# IVMOS — FINAL PRODUCTION-CANDIDATE CHECK
# =============================================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/IVMOS")

required_outputs = [
    ROOT / "raw" / "prices" / "prices_daily.parquet",
    ROOT / "raw" / "macro" / "fred_raw.parquet",
    ROOT / "processed" / "features.parquet",
    ROOT / "processed" / "evidence_sources_v2_1.parquet",
    ROOT / "processed" / "evidence_layers_v2_1.parquet",
    ROOT / "processed" / "evidence_contributions_v2_1.parquet",
    ROOT / "outputs" / "evidence_v2_1_latest.json",
]

missing = [str(path) for path in required_outputs if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Missing production outputs:\n" + "\n".join(missing)
    )

layers = pd.read_parquet(
    ROOT / "processed" / "evidence_layers_v2_1.parquet"
)
layers["date"] = pd.to_datetime(layers["date"], errors="coerce")

latest_date = layers["date"].max()
latest = (
    layers.loc[
        layers["date"].eq(latest_date),
        [
            "layer",
            "score",
            "confidence",
            "source_coverage",
            "agreement_score",
            "dominant_source_id",
            "evidence_mode",
            "is_available",
        ],
    ]
    .sort_values("layer")
    .copy()
)

for column in [
    "score",
    "confidence",
    "source_coverage",
    "agreement_score",
]:
    latest[column] = latest[column].round(2)

credit = layers.loc[layers["layer"].eq("CREDIT")].copy()

tests = {
    "all_required_outputs_exist": not missing,
    "latest_layer_count_is_7": latest["layer"].nunique() == 7,
    "no_duplicate_date_layer": not layers.duplicated(["date", "layer"]).any(),
    "scores_within_bounds": layers["score"].between(0, 100).all(),
    "confidence_within_bounds": layers["confidence"].between(0, 100).all(),
    "credit_mode_present": credit["evidence_mode"].isin(
        ["FULL", "PROXY", "PRIMARY_ONLY", "UNAVAILABLE"]
    ).all(),
    "credit_proxy_history_present": credit["evidence_mode"].eq("PROXY").any(),
    "credit_proxy_confidence_capped": (
        credit.loc[credit["evidence_mode"].eq("PROXY"), "confidence"] <= 65.0
    ).all(),
}

status = "PASS" if all(tests.values()) else "FAIL"

print("\n" + "=" * 88)
print("IVMOS CLEAN MARKET ENGINE — FINAL CHECK")
print("=" * 88)
print("Overall Status:", status)
print("Latest Date:", latest_date.date())

print("\nTests:")
for name, passed in tests.items():
    print(f"  {'PASS' if passed else 'FAIL'} — {name}")

print("\nLatest Evidence Layers:")
print(latest.to_string(index=False))
print("=" * 88)



IVMOS CLEAN MARKET ENGINE — FINAL CHECK
Overall Status: PASS
Latest Date: 2026-08-06

Tests:
  PASS — all_required_outputs_exist
  PASS — latest_layer_count_is_7
  PASS — no_duplicate_date_layer
  PASS — scores_within_bounds
  PASS — confidence_within_bounds
  PASS — credit_mode_present
  PASS — credit_proxy_history_present
  PASS — credit_proxy_confidence_capped

Latest Evidence Layers:
        layer  score  confidence  source_coverage  agreement_score dominant_source_id evidence_mode  is_available
       CREDIT  66.10       80.57              1.0            97.17       ES_HY_SPREAD          FULL          True
   LEADERSHIP  13.50       81.21              1.0            82.13   ES_AI_LEADERSHIP      STANDARD          True
    LIQUIDITY  49.27       74.27              1.0            56.27           ES_WALCL      STANDARD          True
PARTICIPATION  36.48       80.17              1.0            66.88         ES_RSP_SPY      STANDARD          True
     ROTATION  71.51       80.57      

# IVMOS STEP 7A — Mechanism Registry v0.1

In [13]:
# =============================================================================
# IVMOS STEP 7A — Mechanism Registry v0.1
#
# Purpose
# -------
# Define mechanisms that interpret Evidence layers.
# This step creates and validates the registry only.
# It does not calculate mechanism states yet.
# =============================================================================

from __future__ import annotations

import json
from pathlib import Path

import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "mechanism_registry_v0_1_draft.yaml"
)

VALIDATION_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_registry_v0_1_validation.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step7a_mechanism_registry_validation.txt"
)

for directory in [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# Evidence contract
# =============================================================================

VALID_LAYERS = {
    "TREND",
    "PARTICIPATION",
    "LEADERSHIP",
    "ROTATION",
    "CREDIT",
    "LIQUIDITY",
    "STRESS_BUFFER",
}

VALID_STATES = {
    "INACTIVE",
    "WATCH",
    "ACTIVE",
    "INVALIDATED",
}


# =============================================================================
# Mechanism registry
# =============================================================================

registry = {
    "registry_name": "IVMOS Mechanism Registry",
    "registry_version": "0.1.0-draft",

    "score_semantics": {
        "evidence_score": (
            "Higher Evidence score means more risk-supportive."
        ),
        "alignment": (
            "alignment = polarity * (evidence_score - 50) / 50"
        ),
        "alignment_range": [
            -1.0,
            1.0,
        ],
    },

    "state_contract": {
        "states": [
            "INACTIVE",
            "WATCH",
            "ACTIVE",
            "INVALIDATED",
        ],

        "default_thresholds": {
            "watch_strength": 0.20,
            "active_strength": 0.40,
            "invalidated_strength": -0.25,
        },

        "persistence": {
            "watch_days": 2,
            "active_days": 3,
            "invalidated_days": 2,
        },
    },

    "mechanisms": {
        # ---------------------------------------------------------------------
        # 1. Trend continuation
        # ---------------------------------------------------------------------
        "M_TREND_CONTINUATION": {
            "name": "Trend Continuation",
            "description": (
                "Price trend remains supportive and is reinforced by "
                "risk-supportive rotation."
            ),
            "category": "MARKET_STRUCTURE",

            "inputs": {
                "TREND": {
                    "weight": 0.65,
                    "polarity": 1,
                    "role": "PRIMARY",
                },
                "ROTATION": {
                    "weight": 0.35,
                    "polarity": 1,
                    "role": "CONFIRMATION",
                },
            },

            "minimum_input_coverage": 0.65,

            "invalidation_inputs": {
                "STRESS_BUFFER": {
                    "threshold_below": 25.0,
                    "consecutive_days": 2,
                },
            },
        },

        # ---------------------------------------------------------------------
        # 2. Broad participation confirmation
        # ---------------------------------------------------------------------
        "M_BREADTH_CONFIRMATION": {
            "name": "Breadth Confirmation",
            "description": (
                "Participation and leadership confirm that the market move "
                "is broad rather than concentrated."
            ),
            "category": "MARKET_STRUCTURE",

            "inputs": {
                "PARTICIPATION": {
                    "weight": 0.55,
                    "polarity": 1,
                    "role": "PRIMARY",
                },
                "LEADERSHIP": {
                    "weight": 0.45,
                    "polarity": 1,
                    "role": "CONFIRMATION",
                },
            },

            "minimum_input_coverage": 0.55,

            "invalidation_inputs": {
                "PARTICIPATION": {
                    "threshold_below": 30.0,
                    "consecutive_days": 3,
                },
            },
        },

        # ---------------------------------------------------------------------
        # 3. Credit and liquidity support
        # ---------------------------------------------------------------------
        "M_FINANCIAL_CONDITIONS_SUPPORT": {
            "name": "Financial Conditions Support",
            "description": (
                "Credit conditions and system liquidity support risk assets."
            ),
            "category": "FINANCIAL_CONDITIONS",

            "inputs": {
                "CREDIT": {
                    "weight": 0.55,
                    "polarity": 1,
                    "role": "PRIMARY",
                },
                "LIQUIDITY": {
                    "weight": 0.45,
                    "polarity": 1,
                    "role": "CONFIRMATION",
                },
            },

            "minimum_input_coverage": 0.55,

            "confidence_policy": {
                "credit_proxy_mode_penalty": 0.15,
                "credit_proxy_confidence_ceiling": 65.0,
            },

            "invalidation_inputs": {
                "CREDIT": {
                    "threshold_below": 30.0,
                    "consecutive_days": 2,
                },
            },
        },

        # ---------------------------------------------------------------------
        # 4. Stress resilience
        # ---------------------------------------------------------------------
        "M_STRESS_RESILIENCE": {
            "name": "Stress Resilience",
            "description": (
                "Volatility, drawdown and stress indicators remain contained."
            ),
            "category": "RISK_CONTROL",

            "inputs": {
                "STRESS_BUFFER": {
                    "weight": 0.70,
                    "polarity": 1,
                    "role": "PRIMARY",
                },
                "CREDIT": {
                    "weight": 0.30,
                    "polarity": 1,
                    "role": "CONFIRMATION",
                },
            },

            "minimum_input_coverage": 0.70,

            "invalidation_inputs": {
                "STRESS_BUFFER": {
                    "threshold_below": 20.0,
                    "consecutive_days": 2,
                },
            },
        },

        # ---------------------------------------------------------------------
        # 5. Narrow rally / concentration risk
        # ---------------------------------------------------------------------
        "M_NARROW_RALLY_RISK": {
            "name": "Narrow Rally Risk",
            "description": (
                "The headline trend remains positive while participation "
                "and leadership fail to confirm."
            ),
            "category": "DIVERGENCE",

            "inputs": {
                "TREND": {
                    "weight": 0.30,
                    "polarity": 1,
                    "role": "CONTEXT",
                },
                "PARTICIPATION": {
                    "weight": 0.35,
                    "polarity": -1,
                    "role": "PRIMARY",
                },
                "LEADERSHIP": {
                    "weight": 0.35,
                    "polarity": -1,
                    "role": "PRIMARY",
                },
            },

            "minimum_input_coverage": 0.65,

            "activation_constraints": {
                "TREND": {
                    "threshold_above": 55.0,
                },
            },

            "invalidation_inputs": {
                "PARTICIPATION": {
                    "threshold_above": 65.0,
                    "consecutive_days": 3,
                },
                "LEADERSHIP": {
                    "threshold_above": 65.0,
                    "consecutive_days": 3,
                },
            },
        },

        # ---------------------------------------------------------------------
        # 6. Systemic risk-off
        # ---------------------------------------------------------------------
        "M_SYSTEMIC_RISK_OFF": {
            "name": "Systemic Risk-Off",
            "description": (
                "Trend, credit, liquidity and stress conditions deteriorate "
                "at the same time."
            ),
            "category": "RISK_OFF",

            "inputs": {
                "TREND": {
                    "weight": 0.25,
                    "polarity": -1,
                    "role": "PRIMARY",
                },
                "CREDIT": {
                    "weight": 0.25,
                    "polarity": -1,
                    "role": "PRIMARY",
                },
                "LIQUIDITY": {
                    "weight": 0.20,
                    "polarity": -1,
                    "role": "CONFIRMATION",
                },
                "STRESS_BUFFER": {
                    "weight": 0.30,
                    "polarity": -1,
                    "role": "PRIMARY",
                },
            },

            "minimum_input_coverage": 0.70,

            "activation_constraints": {
                "STRESS_BUFFER": {
                    "threshold_below": 40.0,
                },
            },

            "invalidation_inputs": {
                "STRESS_BUFFER": {
                    "threshold_above": 65.0,
                    "consecutive_days": 3,
                },
                "CREDIT": {
                    "threshold_above": 65.0,
                    "consecutive_days": 3,
                },
            },
        },
    },
}


# =============================================================================
# Validation
# =============================================================================

checks = {}
errors = []

mechanisms = registry["mechanisms"]

checks["mechanism_count_is_6"] = (
    len(mechanisms) == 6
)

checks["mechanism_ids_unique"] = (
    len(mechanisms)
    == len(set(mechanisms))
)

all_weights_valid = True
all_polarities_valid = True
all_layers_valid = True
all_coverage_valid = True
all_mechanisms_have_inputs = True
all_weight_sums_equal_one = True


mechanism_summary = []


for mechanism_id, mechanism in mechanisms.items():

    inputs = mechanism.get(
        "inputs",
        {},
    )

    if not inputs:
        all_mechanisms_have_inputs = False
        errors.append(
            f"{mechanism_id}: no inputs"
        )
        continue

    weight_sum = sum(
        float(spec["weight"])
        for spec in inputs.values()
    )

    if abs(weight_sum - 1.0) > 1e-9:
        all_weight_sums_equal_one = False
        errors.append(
            f"{mechanism_id}: weight sum = {weight_sum}"
        )

    for layer, spec in inputs.items():

        if layer not in VALID_LAYERS:
            all_layers_valid = False
            errors.append(
                f"{mechanism_id}: invalid layer {layer}"
            )

        weight = float(
            spec["weight"]
        )

        if not (
            0.0 < weight <= 1.0
        ):
            all_weights_valid = False
            errors.append(
                f"{mechanism_id}: invalid weight "
                f"{layer}={weight}"
            )

        polarity = int(
            spec["polarity"]
        )

        if polarity not in {
            -1,
            1,
        }:
            all_polarities_valid = False
            errors.append(
                f"{mechanism_id}: invalid polarity "
                f"{layer}={polarity}"
            )

    minimum_coverage = float(
        mechanism[
            "minimum_input_coverage"
        ]
    )

    if not (
        0.0
        < minimum_coverage
        <= 1.0
    ):
        all_coverage_valid = False
        errors.append(
            f"{mechanism_id}: invalid minimum coverage"
        )

    mechanism_summary.append({
        "mechanism_id": mechanism_id,
        "name": mechanism["name"],
        "category": mechanism["category"],
        "input_count": len(inputs),
        "weight_sum": weight_sum,
        "minimum_input_coverage": minimum_coverage,
        "input_layers": list(inputs.keys()),
    })


checks[
    "all_mechanisms_have_inputs"
] = all_mechanisms_have_inputs

checks[
    "all_input_layers_valid"
] = all_layers_valid

checks[
    "all_weights_valid"
] = all_weights_valid

checks[
    "all_weight_sums_equal_one"
] = all_weight_sums_equal_one

checks[
    "all_polarities_valid"
] = all_polarities_valid

checks[
    "all_minimum_coverage_valid"
] = all_coverage_valid


required_categories = {
    "MARKET_STRUCTURE",
    "FINANCIAL_CONDITIONS",
    "RISK_CONTROL",
    "DIVERGENCE",
    "RISK_OFF",
}

actual_categories = {
    mechanism["category"]
    for mechanism in mechanisms.values()
}

checks["required_categories_present"] = (
    required_categories
    .issubset(actual_categories)
)


overall_status = (
    "PASS"
    if all(checks.values())
    else "FAIL"
)


# =============================================================================
# Write outputs
# =============================================================================

with open(
    REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        registry,
        file,
        allow_unicode=True,
        sort_keys=False,
    )


validation_report = {
    "registry_version": (
        registry[
            "registry_version"
        ]
    ),
    "overall_status": overall_status,
    "mechanism_count": len(mechanisms),
    "checks": checks,
    "errors": errors,
    "mechanisms": mechanism_summary,
    "files_created": [
        str(REGISTRY_PATH),
        str(VALIDATION_JSON_PATH),
        str(TEST_PATH),
    ],
}


with open(
    VALIDATION_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        validation_report,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


test_lines = [
    "=" * 96,
    "IVMOS STEP 7A — MECHANISM REGISTRY VALIDATION",
    "=" * 96,
    f"Overall Status: {overall_status}",
    (
        "Registry Version: "
        f"{registry['registry_version']}"
    ),
    f"Mechanism Count: {len(mechanisms)}",
    "",
    "Tests:",
]

for name, passed in checks.items():
    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


if errors:
    test_lines.extend([
        "",
        "Errors:",
        *[
            f"  - {error}"
            for error in errors
        ],
    ])


test_lines.extend([
    "",
    "Files created:",
    f"  {REGISTRY_PATH}",
    f"  {VALIDATION_JSON_PATH}",
    f"  {TEST_PATH}",
    "=" * 96,
])


TEST_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Display
# =============================================================================

print("\n" + "=" * 96)
print("IVMOS STEP 7A — MECHANISM REGISTRY VALIDATION")
print("=" * 96)
print("Overall Status:", overall_status)
print(
    "Registry Version:",
    registry[
        "registry_version"
    ],
)
print(
    "Mechanism Count:",
    len(mechanisms),
)

print("\nTests:")

for name, passed in checks.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


print("\nMechanisms:")

for row in mechanism_summary:
    print(
        f"  {row['mechanism_id']}"
        f" | {row['category']}"
        f" | inputs={row['input_count']}"
        f" | weight_sum={row['weight_sum']:.2f}"
        f" | coverage={row['minimum_input_coverage']:.2f}"
    )


print("\nFiles created:")
print(" ", REGISTRY_PATH)
print(" ", VALIDATION_JSON_PATH)
print(" ", TEST_PATH)
print("=" * 96)


IVMOS STEP 7A — MECHANISM REGISTRY VALIDATION
Overall Status: PASS
Registry Version: 0.1.0-draft
Mechanism Count: 6

Tests:
  PASS — mechanism_count_is_6
  PASS — mechanism_ids_unique
  PASS — all_mechanisms_have_inputs
  PASS — all_input_layers_valid
  PASS — all_weights_valid
  PASS — all_weight_sums_equal_one
  PASS — all_polarities_valid
  PASS — all_minimum_coverage_valid
  PASS — required_categories_present

Mechanisms:
  M_TREND_CONTINUATION | MARKET_STRUCTURE | inputs=2 | weight_sum=1.00 | coverage=0.65
  M_BREADTH_CONFIRMATION | MARKET_STRUCTURE | inputs=2 | weight_sum=1.00 | coverage=0.55
  M_FINANCIAL_CONDITIONS_SUPPORT | FINANCIAL_CONDITIONS | inputs=2 | weight_sum=1.00 | coverage=0.55
  M_STRESS_RESILIENCE | RISK_CONTROL | inputs=2 | weight_sum=1.00 | coverage=0.70
  M_NARROW_RALLY_RISK | DIVERGENCE | inputs=3 | weight_sum=1.00 | coverage=0.65
  M_SYSTEMIC_RISK_OFF | RISK_OFF | inputs=4 | weight_sum=1.00 | coverage=0.70

Files created:
  /content/drive/MyDrive/IVMOS/confi

# IVMOS STEP 7B — Mechanism Engine Runtime v0.1

In [14]:
# =============================================================================
# IVMOS STEP 7B — Mechanism Engine Runtime v0.1
#
# Inputs
# ------
# processed/evidence_layers_v2_1.parquet
# config/mechanism_registry_v0_1_draft.yaml
#
# Outputs
# -------
# processed/mechanism_states_v0_1.parquet
# processed/mechanism_contributions_v0_1.parquet
# outputs/mechanism_latest_v0_1.json
# outputs/mechanism_quality_v0_1.json
# tests/step7b_mechanism_runtime_tests.txt
# =============================================================================

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

EVIDENCE_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_layers_v2_1.parquet"
)

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "mechanism_registry_v0_1_draft.yaml"
)

STATE_OUTPUT_PATH = (
    PROJECT_ROOT
    / "processed"
    / "mechanism_states_v0_1.parquet"
)

CONTRIBUTION_OUTPUT_PATH = (
    PROJECT_ROOT
    / "processed"
    / "mechanism_contributions_v0_1.parquet"
)

LATEST_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_latest_v0_1.json"
)

QUALITY_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_quality_v0_1.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step7b_mechanism_runtime_tests.txt"
)

for directory in [
    PROJECT_ROOT / "processed",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# Runtime constants
# =============================================================================

ENGINE_VERSION = "0.1.0-runtime"

SMOOTHING_SPAN = 5

VALID_STATES = {
    "INACTIVE",
    "WATCH",
    "ACTIVE",
    "INVALIDATED",
}


# =============================================================================
# Helpers
# =============================================================================

def normalize_date(
    values: pd.Series,
) -> pd.Series:

    result = pd.to_datetime(
        values,
        errors="coerce",
    )

    try:
        result = result.dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    return result.dt.normalize()


def json_safe(
    value: Any,
) -> Any:

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value

    if pd.isna(value):
        return None

    return value


def consecutive_true_count(
    values: pd.Series,
) -> pd.Series:

    values = (
        values
        .fillna(False)
        .astype(bool)
    )

    groups = (
        (~values)
        .cumsum()
    )

    return (
        values
        .groupby(groups)
        .cumsum()
        .astype(int)
    )


def evaluate_constraint(
    score: pd.Series,
    specification: dict[str, Any],
) -> pd.Series:

    result = pd.Series(
        True,
        index=score.index,
        dtype=bool,
    )

    if "threshold_above" in specification:
        result &= (
            score
            > float(
                specification[
                    "threshold_above"
                ]
            )
        )

    if "threshold_below" in specification:
        result &= (
            score
            < float(
                specification[
                    "threshold_below"
                ]
            )
        )

    return result


def weighted_average(
    values: np.ndarray,
    weights: np.ndarray,
) -> float:

    valid = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    if not valid.any():
        return np.nan

    return float(
        np.average(
            values[valid],
            weights=weights[valid],
        )
    )


# =============================================================================
# Load inputs
# =============================================================================

for path in [
    EVIDENCE_PATH,
    REGISTRY_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )


evidence = pd.read_parquet(
    EVIDENCE_PATH
)


with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    registry = yaml.safe_load(file)


evidence["date"] = normalize_date(
    evidence["date"]
)

evidence["layer"] = (
    evidence["layer"]
    .astype(str)
    .str.upper()
)


required_evidence_columns = {
    "date",
    "layer",
    "score",
    "confidence",
    "source_coverage",
    "is_available",
    "evidence_mode",
}

missing_evidence_columns = (
    required_evidence_columns
    - set(evidence.columns)
)

if missing_evidence_columns:
    raise ValueError(
        "Evidence schema missing columns: "
        f"{sorted(missing_evidence_columns)}"
    )


mechanisms = registry["mechanisms"]

state_contract = registry[
    "state_contract"
]

default_thresholds = state_contract[
    "default_thresholds"
]

persistence_contract = state_contract[
    "persistence"
]


print("Inputs loaded")
print("Evidence rows:", len(evidence))
print("Mechanisms:", len(mechanisms))
print(
    "Date range:",
    evidence["date"].min().date(),
    "to",
    evidence["date"].max().date(),
)


# =============================================================================
# Pivot Evidence layer fields
# =============================================================================

score_wide = evidence.pivot(
    index="date",
    columns="layer",
    values="score",
).sort_index()

confidence_wide = evidence.pivot(
    index="date",
    columns="layer",
    values="confidence",
).sort_index()

coverage_wide = evidence.pivot(
    index="date",
    columns="layer",
    values="source_coverage",
).sort_index()

available_wide = evidence.pivot(
    index="date",
    columns="layer",
    values="is_available",
).sort_index()

mode_wide = evidence.pivot(
    index="date",
    columns="layer",
    values="evidence_mode",
).sort_index()


all_dates = score_wide.index


# =============================================================================
# Calculate mechanisms
# =============================================================================

state_rows = []
contribution_rows = []


for mechanism_id, mechanism in mechanisms.items():

    input_specs = mechanism["inputs"]

    minimum_input_coverage = float(
        mechanism[
            "minimum_input_coverage"
        ]
    )

    input_layers = list(
        input_specs.keys()
    )

    frame = pd.DataFrame(
        index=all_dates
    )

    weighted_alignment_sum = pd.Series(
        0.0,
        index=all_dates,
    )

    available_weight_sum = pd.Series(
        0.0,
        index=all_dates,
    )

    weighted_confidence_sum = pd.Series(
        0.0,
        index=all_dates,
    )

    weighted_coverage_sum = pd.Series(
        0.0,
        index=all_dates,
    )

    for layer_name, specification in (
        input_specs.items()
    ):

        weight = float(
            specification["weight"]
        )

        polarity = int(
            specification["polarity"]
        )

        score = pd.to_numeric(
            score_wide[
                layer_name
            ],
            errors="coerce",
        )

        confidence = pd.to_numeric(
            confidence_wide[
                layer_name
            ],
            errors="coerce",
        )

        layer_coverage = pd.to_numeric(
            coverage_wide[
                layer_name
            ],
            errors="coerce",
        )

        layer_available = (
            available_wide[
                layer_name
            ]
            .fillna(False)
            .astype(bool)
        )

        alignment = (
            polarity
            * (
                score
                - 50.0
            )
            / 50.0
        ).clip(
            -1.0,
            1.0,
        )

        effective_weight = (
            weight
            * layer_available.astype(float)
        )

        weighted_alignment = (
            alignment.fillna(0.0)
            * effective_weight
        )

        weighted_alignment_sum += (
            weighted_alignment
        )

        available_weight_sum += (
            effective_weight
        )

        weighted_confidence_sum += (
            confidence.fillna(0.0)
            * effective_weight
        )

        weighted_coverage_sum += (
            layer_coverage.fillna(0.0)
            * effective_weight
        )

        layer_contribution = pd.DataFrame({
            "date": all_dates,
            "mechanism_id": mechanism_id,
            "mechanism_name": mechanism["name"],
            "category": mechanism["category"],
            "layer": layer_name,
            "role": specification["role"],
            "weight": weight,
            "polarity": polarity,
            "score": score.to_numpy(),
            "confidence": confidence.to_numpy(),
            "layer_coverage": layer_coverage.to_numpy(),
            "is_available": layer_available.to_numpy(),
            "alignment": alignment.to_numpy(),
            "effective_weight": effective_weight.to_numpy(),
            "weighted_alignment": weighted_alignment.to_numpy(),
            "engine_version": ENGINE_VERSION,
            "registry_version": registry[
                "registry_version"
            ],
        })

        contribution_rows.append(
            layer_contribution
        )

    raw_strength = (
        weighted_alignment_sum
        / available_weight_sum.replace(
            0.0,
            np.nan,
        )
    )

    raw_strength = raw_strength.clip(
        -1.0,
        1.0,
    )

    input_coverage = (
        available_weight_sum
    ).clip(
        0.0,
        1.0,
    )

    input_confidence = (
        weighted_confidence_sum
        / available_weight_sum.replace(
            0.0,
            np.nan,
        )
    )

    average_layer_coverage = (
        weighted_coverage_sum
        / available_weight_sum.replace(
            0.0,
            np.nan,
        )
    )

    smoothed_strength = (
        raw_strength
        .ewm(
            span=SMOOTHING_SPAN,
            adjust=False,
            min_periods=1,
        )
        .mean()
    )

    # -------------------------------------------------------------------------
    # Activation constraints
    # -------------------------------------------------------------------------

    activation_constraint_pass = pd.Series(
        True,
        index=all_dates,
        dtype=bool,
    )

    for layer_name, constraint in (
        mechanism.get(
            "activation_constraints",
            {},
        ).items()
    ):

        layer_score = pd.to_numeric(
            score_wide[layer_name],
            errors="coerce",
        )

        activation_constraint_pass &= (
            evaluate_constraint(
                layer_score,
                constraint,
            )
        )

    # -------------------------------------------------------------------------
    # Invalidation conditions
    # -------------------------------------------------------------------------

    invalidation_trigger = pd.Series(
        False,
        index=all_dates,
        dtype=bool,
    )

    invalidation_details = []

    for layer_name, constraint in (
        mechanism.get(
            "invalidation_inputs",
            {},
        ).items()
    ):

        layer_score = pd.to_numeric(
            score_wide[layer_name],
            errors="coerce",
        )

        condition = evaluate_constraint(
            layer_score,
            constraint,
        )

        required_days = int(
            constraint.get(
                "consecutive_days",
                persistence_contract[
                    "invalidated_days"
                ],
            )
        )

        persistent_condition = (
            consecutive_true_count(
                condition
            )
            >= required_days
        )

        invalidation_trigger |= (
            persistent_condition
        )

        invalidation_details.append({
            "layer": layer_name,
            "required_days": required_days,
            "trigger_count": int(
                persistent_condition.sum()
            ),
        })

    # -------------------------------------------------------------------------
    # Confidence
    # -------------------------------------------------------------------------

    mechanism_confidence = (
        0.55
        * input_confidence.fillna(0.0)
        +
        0.25
        * (
            input_coverage
            * 100.0
        )
        +
        0.20
        * (
            average_layer_coverage.fillna(0.0)
            * 100.0
        )
    )

    mechanism_confidence = (
        mechanism_confidence
        * input_coverage
    )

    credit_proxy_flag = pd.Series(
        False,
        index=all_dates,
    )

    if "CREDIT" in input_layers:

        credit_proxy_flag = (
            mode_wide[
                "CREDIT"
            ]
            .astype(str)
            .str.upper()
            .eq("PROXY")
        )

        confidence_policy = (
            mechanism.get(
                "confidence_policy",
                {},
            )
        )

        proxy_penalty = float(
            confidence_policy.get(
                "credit_proxy_mode_penalty",
                0.0,
            )
        )

        proxy_ceiling = float(
            confidence_policy.get(
                "credit_proxy_confidence_ceiling",
                100.0,
            )
        )

        mechanism_confidence.loc[
            credit_proxy_flag
        ] *= (
            1.0
            - proxy_penalty
        )

        mechanism_confidence.loc[
            credit_proxy_flag
        ] = np.minimum(
            mechanism_confidence.loc[
                credit_proxy_flag
            ],
            proxy_ceiling,
        )

    mechanism_confidence = (
        mechanism_confidence.clip(
            0.0,
            94.0,
        )
    )

    # -------------------------------------------------------------------------
    # Persistence
    # -------------------------------------------------------------------------

    watch_threshold = float(
        default_thresholds[
            "watch_strength"
        ]
    )

    active_threshold = float(
        default_thresholds[
            "active_strength"
        ]
    )

    invalidated_threshold = float(
        default_thresholds[
            "invalidated_strength"
        ]
    )

    watch_days = int(
        persistence_contract[
            "watch_days"
        ]
    )

    active_days = int(
        persistence_contract[
            "active_days"
        ]
    )

    invalidated_days = int(
        persistence_contract[
            "invalidated_days"
        ]
    )

    coverage_pass = (
        input_coverage
        >= minimum_input_coverage
    )

    watch_condition = (
        coverage_pass
        &
        activation_constraint_pass
        &
        (
            smoothed_strength
            >= watch_threshold
        )
    )

    active_condition = (
        coverage_pass
        &
        activation_constraint_pass
        &
        (
            smoothed_strength
            >= active_threshold
        )
    )

    strength_invalidation_condition = (
        coverage_pass
        &
        (
            smoothed_strength
            <= invalidated_threshold
        )
    )

    watch_persistence = (
        consecutive_true_count(
            watch_condition
        )
    )

    active_persistence = (
        consecutive_true_count(
            active_condition
        )
    )

    invalidated_persistence = (
        consecutive_true_count(
            strength_invalidation_condition
        )
    )

    state = pd.Series(
        "INACTIVE",
        index=all_dates,
        dtype="object",
    )

    state.loc[
        watch_persistence
        >= watch_days
    ] = "WATCH"

    state.loc[
        active_persistence
        >= active_days
    ] = "ACTIVE"

    state.loc[
        (
            invalidated_persistence
            >= invalidated_days
        )
        |
        invalidation_trigger
    ] = "INVALIDATED"

    state.loc[
        ~coverage_pass
    ] = "INACTIVE"

    # -------------------------------------------------------------------------
    # Output
    # -------------------------------------------------------------------------

    mechanism_frame = pd.DataFrame({
        "date": all_dates,
        "mechanism_id": mechanism_id,
        "mechanism_name": mechanism["name"],
        "category": mechanism["category"],

        "raw_strength": raw_strength.to_numpy(),
        "smoothed_strength": (
            smoothed_strength.to_numpy()
        ),

        "input_coverage": input_coverage.to_numpy(),
        "minimum_input_coverage": (
            minimum_input_coverage
        ),

        "input_confidence": (
            input_confidence.to_numpy()
        ),

        "average_layer_coverage": (
            average_layer_coverage.to_numpy()
        ),

        "confidence": (
            mechanism_confidence.to_numpy()
        ),

        "activation_constraint_pass": (
            activation_constraint_pass.to_numpy()
        ),

        "invalidation_trigger": (
            invalidation_trigger.to_numpy()
        ),

        "credit_proxy_mode": (
            credit_proxy_flag.to_numpy()
        ),

        "watch_persistence_days": (
            watch_persistence.to_numpy()
        ),

        "active_persistence_days": (
            active_persistence.to_numpy()
        ),

        "invalidated_persistence_days": (
            invalidated_persistence.to_numpy()
        ),

        "state": state.to_numpy(),

        "is_available": (
            coverage_pass.to_numpy()
        ),

        "engine_version": ENGINE_VERSION,
        "registry_version": registry[
            "registry_version"
        ],
    })

    state_rows.append(
        mechanism_frame
    )


mechanism_states = pd.concat(
    state_rows,
    ignore_index=True,
)

mechanism_contributions = pd.concat(
    contribution_rows,
    ignore_index=True,
)


# =============================================================================
# Validation
# =============================================================================

expected_row_count = (
    evidence["date"].nunique()
    * len(mechanisms)
)

latest_date = mechanism_states[
    "date"
].max()

latest_states = (
    mechanism_states.loc[
        mechanism_states[
            "date"
        ].eq(latest_date)
    ]
    .sort_values(
        "mechanism_id"
    )
)


duplicate_state_rows = int(
    mechanism_states.duplicated(
        [
            "date",
            "mechanism_id",
        ]
    ).sum()
)


duplicate_contribution_rows = int(
    mechanism_contributions.duplicated(
        [
            "date",
            "mechanism_id",
            "layer",
        ]
    ).sum()
)


checks = {
    "state_row_count_correct": (
        len(mechanism_states)
        == expected_row_count
    ),

    "latest_mechanism_count_is_6": (
        len(latest_states)
        == len(mechanisms)
    ),

    "no_duplicate_date_mechanism": (
        duplicate_state_rows == 0
    ),

    "no_duplicate_contributions": (
        duplicate_contribution_rows == 0
    ),

    "raw_strength_within_bounds": bool(
        mechanism_states[
            "raw_strength"
        ]
        .dropna()
        .between(
            -1.0,
            1.0,
            inclusive="both",
        )
        .all()
    ),

    "smoothed_strength_within_bounds": bool(
        mechanism_states[
            "smoothed_strength"
        ]
        .dropna()
        .between(
            -1.0,
            1.0,
            inclusive="both",
        )
        .all()
    ),

    "confidence_within_bounds": bool(
        mechanism_states[
            "confidence"
        ]
        .between(
            0.0,
            94.0,
            inclusive="both",
        )
        .all()
    ),

    "input_coverage_within_bounds": bool(
        mechanism_states[
            "input_coverage"
        ]
        .between(
            0.0,
            1.0,
            inclusive="both",
        )
        .all()
    ),

    "states_valid": bool(
        set(
            mechanism_states[
                "state"
            ].dropna().unique()
        )
        .issubset(
            VALID_STATES
        )
    ),

    "credit_proxy_confidence_capped": bool(
        (
            mechanism_states.loc[
                mechanism_states[
                    "credit_proxy_mode"
                ],
                "confidence",
            ]
            <= 65.0
        ).all()
    ),

    "latest_all_available": bool(
        latest_states[
            "is_available"
        ].all()
    ),
}


overall_status = (
    "PASS"
    if all(checks.values())
    else "FAIL"
)


# =============================================================================
# Quality diagnostics
# =============================================================================

quality_rows = []


for mechanism_id, frame in (
    mechanism_states.groupby(
        "mechanism_id"
    )
):

    state_counts = (
        frame["state"]
        .value_counts(
            normalize=True
        )
        .to_dict()
    )

    quality_rows.append({
        "mechanism_id": mechanism_id,
        "available_ratio": float(
            frame[
                "is_available"
            ].mean()
        ),
        "confidence_mean": float(
            frame[
                "confidence"
            ].mean()
        ),
        "confidence_median": float(
            frame[
                "confidence"
            ].median()
        ),
        "raw_strength_mean": float(
            frame[
                "raw_strength"
            ].mean()
        ),
        "smoothed_strength_mean": float(
            frame[
                "smoothed_strength"
            ].mean()
        ),
        "inactive_ratio": float(
            state_counts.get(
                "INACTIVE",
                0.0,
            )
        ),
        "watch_ratio": float(
            state_counts.get(
                "WATCH",
                0.0,
            )
        ),
        "active_ratio": float(
            state_counts.get(
                "ACTIVE",
                0.0,
            )
        ),
        "invalidated_ratio": float(
            state_counts.get(
                "INVALIDATED",
                0.0,
            )
        ),
        "state_transition_count": int(
            (
                frame
                .sort_values("date")[
                    "state"
                ]
                .ne(
                    frame
                    .sort_values("date")[
                        "state"
                    ]
                    .shift(1)
                )
                .sum()
                - 1
            )
        ),
    })


quality_df = pd.DataFrame(
    quality_rows
)


# =============================================================================
# Save outputs
# =============================================================================

mechanism_states.to_parquet(
    STATE_OUTPUT_PATH,
    index=False,
)

mechanism_contributions.to_parquet(
    CONTRIBUTION_OUTPUT_PATH,
    index=False,
)


latest_payload = {
    "engine_version": ENGINE_VERSION,
    "registry_version": registry[
        "registry_version"
    ],
    "overall_status": overall_status,
    "latest_date": latest_date,
    "mechanisms": latest_states[
        [
            "mechanism_id",
            "mechanism_name",
            "category",
            "raw_strength",
            "smoothed_strength",
            "confidence",
            "input_coverage",
            "state",
            "credit_proxy_mode",
            "is_available",
        ]
    ].to_dict(
        orient="records"
    ),
}


with open(
    LATEST_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            latest_payload
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


quality_payload = {
    "engine_version": ENGINE_VERSION,
    "registry_version": registry[
        "registry_version"
    ],
    "overall_status": overall_status,
    "checks": checks,
    "quality": quality_df.to_dict(
        orient="records"
    ),
    "files_created": [
        str(STATE_OUTPUT_PATH),
        str(CONTRIBUTION_OUTPUT_PATH),
        str(LATEST_OUTPUT_PATH),
        str(QUALITY_OUTPUT_PATH),
        str(TEST_PATH),
    ],
}


with open(
    QUALITY_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            quality_payload
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


test_lines = [
    "=" * 100,
    "IVMOS STEP 7B — MECHANISM ENGINE RUNTIME",
    "=" * 100,
    f"Overall Status: {overall_status}",
    f"Engine Version: {ENGINE_VERSION}",
    (
        "Registry Version: "
        f"{registry['registry_version']}"
    ),
    f"Latest Date: {latest_date.date()}",
    "",
    "Tests:",
]


for name, passed in checks.items():
    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


test_lines.extend([
    "",
    "Files created:",
    f"  {STATE_OUTPUT_PATH}",
    f"  {CONTRIBUTION_OUTPUT_PATH}",
    f"  {LATEST_OUTPUT_PATH}",
    f"  {QUALITY_OUTPUT_PATH}",
    f"  {TEST_PATH}",
    "=" * 100,
])


TEST_PATH.write_text(
    "\n".join(test_lines),
    encoding="utf-8",
)


# =============================================================================
# Display
# =============================================================================

latest_display = latest_states[
    [
        "mechanism_id",
        "mechanism_name",
        "category",
        "raw_strength",
        "smoothed_strength",
        "confidence",
        "input_coverage",
        "state",
        "credit_proxy_mode",
    ]
].copy()


for column in [
    "raw_strength",
    "smoothed_strength",
    "confidence",
    "input_coverage",
]:
    latest_display[column] = (
        latest_display[column]
        .round(3)
    )


quality_display = quality_df.copy()

for column in (
    quality_display
    .select_dtypes(
        include=[np.number]
    )
    .columns
):
    quality_display[column] = (
        quality_display[column]
        .round(3)
    )


print("\n" + "=" * 100)
print("IVMOS STEP 7B — MECHANISM ENGINE RUNTIME")
print("=" * 100)
print("Overall Status:", overall_status)
print("Engine Version:", ENGINE_VERSION)
print(
    "Registry Version:",
    registry[
        "registry_version"
    ],
)
print(
    "Latest Date:",
    latest_date.date(),
)

print("\nTests:")

for name, passed in checks.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


print("\nLatest Mechanisms:")
print(
    latest_display.to_string(
        index=False
    )
)


print("\nHistorical Quality:")
print(
    quality_display.to_string(
        index=False
    )
)


print("\nFiles created:")
print(" ", STATE_OUTPUT_PATH)
print(" ", CONTRIBUTION_OUTPUT_PATH)
print(" ", LATEST_OUTPUT_PATH)
print(" ", QUALITY_OUTPUT_PATH)
print(" ", TEST_PATH)
print("=" * 100)

Inputs loaded
Evidence rows: 20412
Mechanisms: 6
Date range: 2015-01-02 to 2026-08-06

IVMOS STEP 7B — MECHANISM ENGINE RUNTIME
Overall Status: FAIL
Engine Version: 0.1.0-runtime
Registry Version: 0.1.0-draft
Latest Date: 2026-08-06

Tests:
  PASS — state_row_count_correct
  PASS — latest_mechanism_count_is_6
  PASS — no_duplicate_date_mechanism
  PASS — no_duplicate_contributions
  PASS — raw_strength_within_bounds
  PASS — smoothed_strength_within_bounds
  PASS — confidence_within_bounds
  PASS — input_coverage_within_bounds
  PASS — states_valid
  FAIL — credit_proxy_confidence_capped
  PASS — latest_all_available

Latest Mechanisms:
                  mechanism_id               mechanism_name             category  raw_strength  smoothed_strength  confidence  input_coverage       state  credit_proxy_mode
        M_BREADTH_CONFIRMATION         Breadth Confirmation     MARKET_STRUCTURE        -0.477             -0.448      89.352             1.0 INVALIDATED              False
M_FINANCI

In [15]:
# =============================================================================
# IVMOS STEP 7A.1 — Credit Proxy Confidence Policy Patch
#
# Adds the same Credit PROXY confidence policy to every mechanism
# that consumes the CREDIT evidence layer.
# =============================================================================

from pathlib import Path

import yaml


PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "mechanism_registry_v0_1_draft.yaml"
)

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(
        f"Missing registry: {REGISTRY_PATH}"
    )


with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    registry = yaml.safe_load(file)


DEFAULT_CREDIT_PROXY_POLICY = {
    "credit_proxy_mode_penalty": 0.15,
    "credit_proxy_confidence_ceiling": 65.0,
}


patched_mechanisms = []


for mechanism_id, mechanism in (
    registry["mechanisms"].items()
):

    if "CREDIT" not in mechanism.get(
        "inputs",
        {},
    ):
        continue

    existing_policy = mechanism.get(
        "confidence_policy",
        {},
    )

    existing_policy.update(
        DEFAULT_CREDIT_PROXY_POLICY
    )

    mechanism[
        "confidence_policy"
    ] = existing_policy

    patched_mechanisms.append(
        mechanism_id
    )


expected_credit_mechanisms = {
    mechanism_id
    for mechanism_id, mechanism
    in registry["mechanisms"].items()
    if "CREDIT" in mechanism.get(
        "inputs",
        {},
    )
}


patched_set = set(
    patched_mechanisms
)


checks = {
    "all_credit_mechanisms_patched": (
        patched_set
        == expected_credit_mechanisms
    ),

    "all_proxy_penalties_are_15_percent": all(
        float(
            registry["mechanisms"][
                mechanism_id
            ]["confidence_policy"][
                "credit_proxy_mode_penalty"
            ]
        )
        == 0.15
        for mechanism_id
        in expected_credit_mechanisms
    ),

    "all_proxy_ceilings_are_65": all(
        float(
            registry["mechanisms"][
                mechanism_id
            ]["confidence_policy"][
                "credit_proxy_confidence_ceiling"
            ]
        )
        == 65.0
        for mechanism_id
        in expected_credit_mechanisms
    ),
}


overall_status = (
    "PASS"
    if all(checks.values())
    else "FAIL"
)


if overall_status != "PASS":
    raise ValueError(
        "Credit proxy policy patch failed validation"
    )


with open(
    REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        registry,
        file,
        allow_unicode=True,
        sort_keys=False,
    )


print("\n" + "=" * 88)
print("IVMOS STEP 7A.1 — CREDIT PROXY POLICY PATCH")
print("=" * 88)
print("Overall Status:", overall_status)

print("\nPatched mechanisms:")

for mechanism_id in sorted(
    patched_mechanisms
):
    print(" ", mechanism_id)

print("\nTests:")

for name, passed in checks.items():
    print(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )

print("\nUpdated registry:")
print(REGISTRY_PATH)
print("=" * 88)


IVMOS STEP 7A.1 — CREDIT PROXY POLICY PATCH
Overall Status: PASS

Patched mechanisms:
  M_FINANCIAL_CONDITIONS_SUPPORT
  M_STRESS_RESILIENCE
  M_SYSTEMIC_RISK_OFF

Tests:
  PASS — all_credit_mechanisms_patched
  PASS — all_proxy_penalties_are_15_percent
  PASS — all_proxy_ceilings_are_65

Updated registry:
/content/drive/MyDrive/IVMOS/config/mechanism_registry_v0_1_draft.yaml


In [16]:
# =============================================================================
# IVMOS STEP 7C — Mechanism Semantic Audit v0.1
#
# Diagnostic only:
# - Does not modify Step 6 or Step 7B outputs
# - Audits persistence, event behavior, contradiction, state occupancy,
#   confidence inflation and latest-state consistency
# =============================================================================

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

STATE_PATH = (
    PROJECT_ROOT
    / "processed"
    / "mechanism_states_v0_1.parquet"
)

CONTRIBUTION_PATH = (
    PROJECT_ROOT
    / "processed"
    / "mechanism_contributions_v0_1.parquet"
)

EVIDENCE_PATH = (
    PROJECT_ROOT
    / "processed"
    / "evidence_layers_v2_1.parquet"
)

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "mechanism_registry_v0_1_draft.yaml"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_semantic_summary_v0_1.csv"
)

EVENT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_event_audit_v0_1.csv"
)

CONTRADICTION_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_contradiction_audit_v0_1.csv"
)

LATEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_latest_semantic_audit_v0_1.csv"
)

AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "mechanism_semantic_audit_v0_1.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step7c_mechanism_semantic_audit.txt"
)

for directory in [
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# Audit configuration
# =============================================================================

EVENT_WINDOWS = {
    "2015_RISK_OFF": (
        "2015-08-17",
        "2015-09-30",
    ),
    "2018_Q4_SELLOFF": (
        "2018-10-01",
        "2018-12-24",
    ),
    "2020_COVID_SHOCK": (
        "2020-02-19",
        "2020-04-30",
    ),
    "2022_TIGHTENING_BEAR": (
        "2022-01-03",
        "2022-10-14",
    ),
    "2023_BANK_STRESS": (
        "2023-03-01",
        "2023-05-15",
    ),
}

OPPOSING_PAIRS = [
    (
        "M_TREND_CONTINUATION",
        "M_SYSTEMIC_RISK_OFF",
    ),
    (
        "M_BREADTH_CONFIRMATION",
        "M_NARROW_RALLY_RISK",
    ),
]

EXPECTED_LATEST = {
    "M_BREADTH_CONFIRMATION": {
        "allowed_states": {
            "INVALIDATED",
            "INACTIVE",
        },
    },
    "M_NARROW_RALLY_RISK": {
        "allowed_states": {
            "WATCH",
            "ACTIVE",
        },
    },
    "M_SYSTEMIC_RISK_OFF": {
        "allowed_states": {
            "INACTIVE",
            "INVALIDATED",
        },
    },
}


# =============================================================================
# Helpers
# =============================================================================

def normalize_date(
    values: pd.Series,
) -> pd.Series:

    result = pd.to_datetime(
        values,
        errors="coerce",
    )

    try:
        result = result.dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    return result.dt.normalize()


def json_safe(
    value: Any,
) -> Any:

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value

    if pd.isna(value):
        return None

    return value


def state_transition_count(
    states: pd.Series,
) -> int:

    series = (
        states
        .dropna()
        .astype(str)
    )

    if series.empty:
        return 0

    return max(
        int(
            series.ne(
                series.shift(1)
            ).sum()
            - 1
        ),
        0,
    )


def safe_ratio(
    numerator: float,
    denominator: float,
) -> float:

    if denominator == 0:
        return np.nan

    return float(
        numerator / denominator
    )


# =============================================================================
# Load
# =============================================================================

for path in [
    STATE_PATH,
    CONTRIBUTION_PATH,
    EVIDENCE_PATH,
    REGISTRY_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )


states = pd.read_parquet(
    STATE_PATH
)

contributions = pd.read_parquet(
    CONTRIBUTION_PATH
)

evidence = pd.read_parquet(
    EVIDENCE_PATH
)


with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    registry = yaml.safe_load(file)


for frame in [
    states,
    contributions,
    evidence,
]:
    frame["date"] = normalize_date(
        frame["date"]
    )


states = states.sort_values(
    [
        "mechanism_id",
        "date",
    ]
).reset_index(
    drop=True
)


print("Inputs loaded")
print("Mechanism rows:", len(states))
print("Contribution rows:", len(contributions))
print("Evidence rows:", len(evidence))
print(
    "Date range:",
    states["date"].min().date(),
    "to",
    states["date"].max().date(),
)


# =============================================================================
# Mechanism-level summary
# =============================================================================

summary_rows = []


for mechanism_id, frame in states.groupby(
    "mechanism_id"
):

    frame = frame.sort_values(
        "date"
    )

    available = frame.loc[
        frame["is_available"]
        .fillna(False)
    ]

    state_ratios = (
        frame["state"]
        .value_counts(
            normalize=True
        )
        .to_dict()
    )

    years = max(
        (
            frame["date"].max()
            -
            frame["date"].min()
        ).days
        / 365.25,
        1e-9,
    )

    transition_count = (
        state_transition_count(
            frame["state"]
        )
    )

    confidence_minus_input = (
        frame["confidence"]
        -
        frame["input_confidence"]
    )

    active_or_watch = frame[
        "state"
    ].isin(
        [
            "WATCH",
            "ACTIVE",
        ]
    )

    positive_strength = (
        frame["smoothed_strength"]
        >= 0.20
    )

    state_strength_consistency = (
        (
            active_or_watch
            == positive_strength
        )
        .mean()
    )

    summary_rows.append({
        "mechanism_id": mechanism_id,
        "mechanism_name": (
            frame[
                "mechanism_name"
            ].iloc[0]
        ),
        "category": (
            frame[
                "category"
            ].iloc[0]
        ),
        "available_ratio": float(
            frame[
                "is_available"
            ].mean()
        ),
        "raw_strength_mean": float(
            available[
                "raw_strength"
            ].mean()
        ),
        "smoothed_strength_mean": float(
            available[
                "smoothed_strength"
            ].mean()
        ),
        "smoothed_strength_std": float(
            available[
                "smoothed_strength"
            ].std()
        ),
        "confidence_mean": float(
            available[
                "confidence"
            ].mean()
        ),
        "input_confidence_mean": float(
            available[
                "input_confidence"
            ].mean()
        ),
        "confidence_minus_input_mean": float(
            confidence_minus_input[
                frame["is_available"]
                .fillna(False)
            ].mean()
        ),
        "confidence_minus_input_p95": float(
            confidence_minus_input[
                frame["is_available"]
                .fillna(False)
            ].quantile(
                0.95
            )
        ),
        "inactive_ratio": float(
            state_ratios.get(
                "INACTIVE",
                0.0,
            )
        ),
        "watch_ratio": float(
            state_ratios.get(
                "WATCH",
                0.0,
            )
        ),
        "active_ratio": float(
            state_ratios.get(
                "ACTIVE",
                0.0,
            )
        ),
        "invalidated_ratio": float(
            state_ratios.get(
                "INVALIDATED",
                0.0,
            )
        ),
        "transition_count": (
            transition_count
        ),
        "transitions_per_year": float(
            transition_count / years
        ),
        "state_strength_consistency": float(
            state_strength_consistency
        ),
        "credit_proxy_ratio": float(
            frame[
                "credit_proxy_mode"
            ].mean()
        ),
    })


summary_df = pd.DataFrame(
    summary_rows
)


# =============================================================================
# Persistence and latest-state audit
# =============================================================================

latest_date = states[
    "date"
].max()


latest = (
    states.loc[
        states[
            "date"
        ].eq(
            latest_date
        )
    ]
    .sort_values(
        "mechanism_id"
    )
    .copy()
)


latest[
    "active_threshold"
] = float(
    registry[
        "state_contract"
    ][
        "default_thresholds"
    ][
        "active_strength"
    ]
)


latest[
    "active_required_days"
] = int(
    registry[
        "state_contract"
    ][
        "persistence"
    ][
        "active_days"
    ]
)


latest[
    "active_strength_pass"
] = (
    latest[
        "smoothed_strength"
    ]
    >=
    latest[
        "active_threshold"
    ]
)


latest[
    "active_persistence_pass"
] = (
    latest[
        "active_persistence_days"
    ]
    >=
    latest[
        "active_required_days"
    ]
)


latest[
    "state_matches_active_contract"
] = np.where(
    latest[
        "state"
    ].eq(
        "ACTIVE"
    ),
    (
        latest[
            "active_strength_pass"
        ]
        &
        latest[
            "active_persistence_pass"
        ]
        &
        latest[
            "activation_constraint_pass"
        ]
    ),
    True,
)


latest_expected_checks = {}

for mechanism_id, expectation in (
    EXPECTED_LATEST.items()
):

    state_rows = latest.loc[
        latest[
            "mechanism_id"
        ].eq(
            mechanism_id
        ),
        "state",
    ]

    latest_expected_checks[
        mechanism_id
    ] = bool(
        not state_rows.empty
        and
        state_rows.iloc[0]
        in expectation[
            "allowed_states"
        ]
    )


# =============================================================================
# Event-window audit
# =============================================================================

event_rows = []


for event_name, (
    start_date,
    end_date,
) in EVENT_WINDOWS.items():

    start = pd.Timestamp(
        start_date
    )

    end = pd.Timestamp(
        end_date
    )

    for mechanism_id, frame in (
        states.groupby(
            "mechanism_id"
        )
    ):

        event_frame = frame.loc[
            frame["date"].between(
                start,
                end,
                inclusive="both",
            )
        ].sort_values(
            "date"
        )

        if event_frame.empty:
            continue

        counts = (
            event_frame[
                "state"
            ]
            .value_counts()
            .to_dict()
        )

        event_rows.append({
            "event": event_name,
            "start_date": start,
            "end_date": end,
            "mechanism_id": mechanism_id,
            "observations": int(
                len(event_frame)
            ),
            "strength_start": float(
                event_frame[
                    "smoothed_strength"
                ].iloc[0]
            ),
            "strength_end": float(
                event_frame[
                    "smoothed_strength"
                ].iloc[-1]
            ),
            "strength_min": float(
                event_frame[
                    "smoothed_strength"
                ].min()
            ),
            "strength_max": float(
                event_frame[
                    "smoothed_strength"
                ].max()
            ),
            "strength_mean": float(
                event_frame[
                    "smoothed_strength"
                ].mean()
            ),
            "confidence_mean": float(
                event_frame[
                    "confidence"
                ].mean()
            ),
            "inactive_ratio": safe_ratio(
                counts.get(
                    "INACTIVE",
                    0,
                ),
                len(event_frame),
            ),
            "watch_ratio": safe_ratio(
                counts.get(
                    "WATCH",
                    0,
                ),
                len(event_frame),
            ),
            "active_ratio": safe_ratio(
                counts.get(
                    "ACTIVE",
                    0,
                ),
                len(event_frame),
            ),
            "invalidated_ratio": safe_ratio(
                counts.get(
                    "INVALIDATED",
                    0,
                ),
                len(event_frame),
            ),
        })


event_df = pd.DataFrame(
    event_rows
)


# =============================================================================
# Opposing-mechanism contradiction audit
# =============================================================================

state_wide = states.pivot(
    index="date",
    columns="mechanism_id",
    values="state",
)

strength_wide = states.pivot(
    index="date",
    columns="mechanism_id",
    values="smoothed_strength",
)


contradiction_rows = []


for left_id, right_id in OPPOSING_PAIRS:

    left_active = (
        state_wide[left_id]
        .eq("ACTIVE")
    )

    right_active = (
        state_wide[right_id]
        .eq("ACTIVE")
    )

    simultaneous_active = (
        left_active
        &
        right_active
    )

    simultaneous_positive = (
        strength_wide[left_id]
        .gt(0.20)
        &
        strength_wide[right_id]
        .gt(0.20)
    )

    contradiction_rows.append({
        "left_mechanism": left_id,
        "right_mechanism": right_id,
        "observation_count": int(
            len(state_wide)
        ),
        "simultaneous_active_count": int(
            simultaneous_active.sum()
        ),
        "simultaneous_active_ratio": float(
            simultaneous_active.mean()
        ),
        "simultaneous_positive_count": int(
            simultaneous_positive.sum()
        ),
        "simultaneous_positive_ratio": float(
            simultaneous_positive.mean()
        ),
        "latest_left_state": (
            state_wide[
                left_id
            ].iloc[-1]
        ),
        "latest_right_state": (
            state_wide[
                right_id
            ].iloc[-1]
        ),
    })


contradiction_df = pd.DataFrame(
    contradiction_rows
)


# =============================================================================
# Semantic checks
# =============================================================================

summary_map = (
    summary_df
    .set_index(
        "mechanism_id"
    )
)


covid_risk_off = event_df.loc[
    event_df[
        "event"
    ].eq(
        "2020_COVID_SHOCK"
    )
    &
    event_df[
        "mechanism_id"
    ].eq(
        "M_SYSTEMIC_RISK_OFF"
    )
]


covid_trend = event_df.loc[
    event_df[
        "event"
    ].eq(
        "2020_COVID_SHOCK"
    )
    &
    event_df[
        "mechanism_id"
    ].eq(
        "M_TREND_CONTINUATION"
    )
]


checks = {
    "latest_active_states_match_persistence_contract": bool(
        latest[
            "state_matches_active_contract"
        ].all()
    ),

    "latest_expected_divergence_structure": bool(
        all(
            latest_expected_checks.values()
        )
    ),

    "opposing_mechanisms_not_active_together": bool(
        (
            contradiction_df[
                "simultaneous_active_ratio"
            ]
            <= 0.01
        ).all()
    ),

    "confidence_mean_not_above_90": bool(
        (
            summary_df[
                "confidence_mean"
            ]
            <= 90.0
        ).all()
    ),

    "confidence_not_inflated_above_input_by_more_than_15": bool(
        (
            summary_df[
                "confidence_minus_input_p95"
            ]
            <= 15.0
        ).all()
    ),

    "transition_frequency_not_above_30_per_year": bool(
        (
            summary_df[
                "transitions_per_year"
            ]
            <= 30.0
        ).all()
    ),

    "invalidated_state_not_above_50_percent": bool(
        (
            summary_df[
                "invalidated_ratio"
            ]
            <= 0.50
        ).all()
    ),

    "covid_risk_off_activates_or_watches": bool(
        not covid_risk_off.empty
        and
        (
            covid_risk_off[
                "active_ratio"
            ].iloc[0]
            +
            covid_risk_off[
                "watch_ratio"
            ].iloc[0]
        )
        >= 0.20
    ),

    "covid_trend_continuation_not_dominantly_active": bool(
        not covid_trend.empty
        and
        covid_trend[
            "active_ratio"
        ].iloc[0]
        <= 0.25
    ),
}


critical_checks = {
    "no_duplicate_date_mechanism": bool(
        not states.duplicated(
            [
                "date",
                "mechanism_id",
            ]
        ).any()
    ),

    "all_six_mechanisms_present": bool(
        states[
            "mechanism_id"
        ].nunique()
        == 6
    ),

    "latest_all_mechanisms_present": bool(
        len(latest)
        == 6
    ),

    "all_states_available": bool(
        set(
            states[
                "state"
            ].unique()
        )
        .issubset({
            "INACTIVE",
            "WATCH",
            "ACTIVE",
            "INVALIDATED",
        })
    ),
}


if not all(
    critical_checks.values()
):
    overall_status = "FAIL"

elif not all(
    checks.values()
):
    overall_status = "REVIEW_REQUIRED"

else:
    overall_status = "PASS"


# =============================================================================
# Export
# =============================================================================

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)

event_df.to_csv(
    EVENT_PATH,
    index=False,
)

contradiction_df.to_csv(
    CONTRADICTION_PATH,
    index=False,
)

latest.to_csv(
    LATEST_PATH,
    index=False,
)


audit_payload = {
    "engine": (
        "IVMOS Mechanism Semantic Audit"
    ),
    "overall_status": (
        overall_status
    ),
    "latest_date": (
        latest_date
    ),
    "critical_checks": (
        critical_checks
    ),
    "semantic_checks": checks,
    "latest_expected_checks": (
        latest_expected_checks
    ),
    "files_created": [
        str(SUMMARY_PATH),
        str(EVENT_PATH),
        str(CONTRADICTION_PATH),
        str(LATEST_PATH),
        str(AUDIT_JSON_PATH),
        str(TEST_PATH),
    ],
}


with open(
    AUDIT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            audit_payload
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


test_lines = [
    "=" * 100,
    "IVMOS STEP 7C — MECHANISM SEMANTIC AUDIT",
    "=" * 100,
    f"Overall Status: {overall_status}",
    f"Latest Date: {latest_date.date()}",
    "",
    "Critical checks:",
]

for name, passed in (
    critical_checks.items()
):
    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


test_lines.extend([
    "",
    "Semantic checks:",
])

for name, passed in checks.items():
    test_lines.append(
        f"  {'PASS' if passed else 'REVIEW'} — {name}"
    )


test_lines.extend([
    "",
    "Files created:",
    *[
        f"  {path}"
        for path
        in audit_payload[
            "files_created"
        ]
    ],
    "=" * 100,
])


TEST_PATH.write_text(
    "\n".join(
        test_lines
    ),
    encoding="utf-8",
)


# =============================================================================
# Display
# =============================================================================

summary_display = (
    summary_df.copy()
)

for column in (
    summary_display
    .select_dtypes(
        include=[np.number]
    )
    .columns
):
    summary_display[column] = (
        summary_display[
            column
        ].round(3)
    )


latest_display = latest[
    [
        "mechanism_id",
        "raw_strength",
        "smoothed_strength",
        "active_persistence_days",
        "watch_persistence_days",
        "invalidated_persistence_days",
        "confidence",
        "state",
        "activation_constraint_pass",
        "invalidation_trigger",
    ]
].copy()


for column in (
    latest_display
    .select_dtypes(
        include=[np.number]
    )
    .columns
):
    latest_display[column] = (
        latest_display[
            column
        ].round(3)
    )


event_display = event_df.loc[
    event_df[
        "event"
    ].isin(
        [
            "2020_COVID_SHOCK",
            "2022_TIGHTENING_BEAR",
            "2023_BANK_STRESS",
        ]
    )
].copy()


for column in (
    event_display
    .select_dtypes(
        include=[np.number]
    )
    .columns
):
    event_display[column] = (
        event_display[
            column
        ].round(3)
    )


print("\n" + "=" * 100)
print("IVMOS STEP 7C — MECHANISM SEMANTIC AUDIT")
print("=" * 100)
print("Overall Status:", overall_status)
print("Latest Date:", latest_date.date())

print("\nCritical checks:")

for name, passed in (
    critical_checks.items()
):
    print(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )

print("\nSemantic checks:")

for name, passed in checks.items():
    print(
        f"  {'PASS' if passed else 'REVIEW'} — {name}"
    )

print("\nMechanism summary:")
print(
    summary_display.to_string(
        index=False
    )
)

print("\nLatest persistence audit:")
print(
    latest_display.to_string(
        index=False
    )
)

print("\nOpposing-pair audit:")
print(
    contradiction_df.round(
        4
    ).to_string(
        index=False
    )
)

print("\nSelected event audit:")
print(
    event_display.to_string(
        index=False
    )
)

print("\nFiles created:")

for path in audit_payload[
    "files_created"
]:
    print(" ", path)

print("=" * 100)

Inputs loaded
Mechanism rows: 17496
Contribution rows: 43740
Evidence rows: 20412
Date range: 2015-01-02 to 2026-08-06

IVMOS STEP 7C — MECHANISM SEMANTIC AUDIT
Overall Status: PASS
Latest Date: 2026-08-06

Critical checks:
  PASS — no_duplicate_date_mechanism
  PASS — all_six_mechanisms_present
  PASS — latest_all_mechanisms_present
  PASS — all_states_available

Semantic checks:
  PASS — latest_active_states_match_persistence_contract
  PASS — latest_expected_divergence_structure
  PASS — opposing_mechanisms_not_active_together
  PASS — confidence_mean_not_above_90
  PASS — confidence_not_inflated_above_input_by_more_than_15
  PASS — transition_frequency_not_above_30_per_year
  PASS — invalidated_state_not_above_50_percent
  PASS — covid_risk_off_activates_or_watches
  PASS — covid_trend_continuation_not_dominantly_active

Mechanism summary:
                  mechanism_id               mechanism_name             category  available_ratio  raw_strength_mean  smoothed_strength_mean  sm

## IVMOS STEP 8A — Regime Registry v0.1

In [17]:
# =============================================================================
# IVMOS STEP 8A — Regime Registry v0.1
#
# Purpose
# -------
# Define market regimes from validated Mechanism states.
#
# This step:
# - creates the Regime Registry
# - validates regime contracts
# - does NOT calculate historical regime states yet
# =============================================================================

from __future__ import annotations

import json
from pathlib import Path

import yaml


# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IVMOS")

REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "regime_registry_v0_1_draft.yaml"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "regime_registry_v0_1_validation.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "tests"
    / "step8a_regime_registry_validation.txt"
)

for directory in [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "tests",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# Valid mechanism universe
# =============================================================================

VALID_MECHANISMS = {
    "M_TREND_CONTINUATION",
    "M_BREADTH_CONFIRMATION",
    "M_FINANCIAL_CONDITIONS_SUPPORT",
    "M_STRESS_RESILIENCE",
    "M_NARROW_RALLY_RISK",
    "M_SYSTEMIC_RISK_OFF",
}

VALID_MECHANISM_STATES = {
    "INACTIVE",
    "WATCH",
    "ACTIVE",
    "INVALIDATED",
}


# =============================================================================
# Registry
# =============================================================================

registry = {
    "registry_name": "IVMOS Regime Registry",
    "registry_version": "0.1.0-draft",

    "semantics": {
        "purpose": (
            "Classify the market environment from validated "
            "mechanism states and strengths."
        ),

        "priority_rule": (
            "Higher-priority regimes are evaluated before lower-priority "
            "regimes. MIXED_TRANSITION is the fallback."
        ),

        "confidence_rule": (
            "Regime confidence must be derived from mechanism confidence "
            "and agreement; Evidence confidence must not be counted again."
        ),
    },

    "regime_priority": [
        "SYSTEMIC_RISK_OFF",
        "BROAD_RISK_ON",
        "NARROW_RISK_ON",
        "RECOVERY_TRANSITION",
        "MIXED_TRANSITION",
    ],

    "regimes": {

        # ---------------------------------------------------------------------
        # 1. Systemic Risk-Off
        # ---------------------------------------------------------------------

        "SYSTEMIC_RISK_OFF": {
            "name": "Systemic Risk-Off",

            "description": (
                "Systemic risk mechanism is active while trend and/or "
                "stress resilience are materially impaired."
            ),

            "required": {
                "M_SYSTEMIC_RISK_OFF": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },
            },

            "confirmation_any": {
                "M_TREND_CONTINUATION": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },

                "M_STRESS_RESILIENCE": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },
            },

            "minimum_confirmation_count": 1,

            "minimum_regime_score": 0.50,
        },

        # ---------------------------------------------------------------------
        # 2. Broad Risk-On
        # ---------------------------------------------------------------------

        "BROAD_RISK_ON": {
            "name": "Broad Risk-On",

            "description": (
                "Trend is positive and participation broadly confirms "
                "the move without systemic risk-off."
            ),

            "required": {
                "M_TREND_CONTINUATION": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_BREADTH_CONFIRMATION": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_SYSTEMIC_RISK_OFF": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },
            },

            "confirmation_any": {
                "M_FINANCIAL_CONDITIONS_SUPPORT": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_STRESS_RESILIENCE": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },
            },

            "minimum_confirmation_count": 0,

            "exclusions": {
                "M_NARROW_RALLY_RISK": {
                    "states": [
                        "ACTIVE",
                    ],
                },
            },

            "minimum_regime_score": 0.45,
        },

        # ---------------------------------------------------------------------
        # 3. Narrow Risk-On
        # ---------------------------------------------------------------------

        "NARROW_RISK_ON": {
            "name": "Narrow Risk-On",

            "description": (
                "Trend remains constructive but breadth or leadership "
                "fails to confirm, creating concentration risk."
            ),

            "required": {
                "M_TREND_CONTINUATION": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_NARROW_RALLY_RISK": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_SYSTEMIC_RISK_OFF": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },
            },

            "confirmation_any": {
                "M_BREADTH_CONFIRMATION": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },
            },

            "minimum_confirmation_count": 1,

            "minimum_regime_score": 0.40,
        },

        # ---------------------------------------------------------------------
        # 4. Recovery / Transition
        # ---------------------------------------------------------------------

        "RECOVERY_TRANSITION": {
            "name": "Recovery Transition",

            "description": (
                "Systemic risk pressure is no longer dominant while "
                "trend is improving, but broad confirmation is incomplete."
            ),

            "required": {
                "M_SYSTEMIC_RISK_OFF": {
                    "states": [
                        "INACTIVE",
                        "INVALIDATED",
                    ],
                },
            },

            "confirmation_any": {
                "M_TREND_CONTINUATION": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_STRESS_RESILIENCE": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },

                "M_FINANCIAL_CONDITIONS_SUPPORT": {
                    "states": [
                        "WATCH",
                        "ACTIVE",
                    ],
                },
            },

            "minimum_confirmation_count": 1,

            "minimum_regime_score": 0.25,
        },

        # ---------------------------------------------------------------------
        # 5. Mixed / Transition
        # ---------------------------------------------------------------------

        "MIXED_TRANSITION": {
            "name": "Mixed / Transition",

            "description": (
                "No higher-priority regime has sufficient confirmation."
            ),

            "fallback": True,

            "minimum_regime_score": 0.0,
        },
    },

    "persistence": {
        "candidate_days": 2,
        "confirmed_days": 3,

        "emergency_override": {
            "regime": "SYSTEMIC_RISK_OFF",
            "mechanism": "M_SYSTEMIC_RISK_OFF",
            "state": "ACTIVE",
            "days": 2,
        },
    },

    "confidence": {
        "maximum": 94.0,

        "components": {
            "mechanism_confidence": 0.55,
            "mechanism_agreement": 0.30,
            "regime_margin": 0.15,
        },
    },
}


# =============================================================================
# Validation
# =============================================================================

checks = {}
errors = []


regimes = registry[
    "regimes"
]

priority = registry[
    "regime_priority"
]


checks[
    "regime_count_is_5"
] = (
    len(regimes) == 5
)


checks[
    "priority_contains_all_regimes"
] = (
    set(priority)
    == set(regimes)
)


checks[
    "priority_has_no_duplicates"
] = (
    len(priority)
    == len(set(priority))
)


fallback_regimes = [
    regime_id
    for regime_id, specification
    in regimes.items()
    if specification.get(
        "fallback",
        False,
    )
]


checks[
    "exactly_one_fallback_regime"
] = (
    fallback_regimes
    == [
        "MIXED_TRANSITION"
    ]
)


all_mechanisms_valid = True
all_states_valid = True
all_scores_valid = True
all_confirmation_counts_valid = True


for regime_id, specification in (
    regimes.items()
):

    minimum_score = float(
        specification.get(
            "minimum_regime_score",
            0.0,
        )
    )

    if not (
        0.0
        <= minimum_score
        <= 1.0
    ):
        all_scores_valid = False

        errors.append(
            f"{regime_id}: invalid minimum_regime_score"
        )

    for contract_name in [
        "required",
        "confirmation_any",
        "exclusions",
    ]:

        contract = specification.get(
            contract_name,
            {},
        )

        for mechanism_id, rule in (
            contract.items()
        ):

            if mechanism_id not in (
                VALID_MECHANISMS
            ):
                all_mechanisms_valid = False

                errors.append(
                    f"{regime_id}: invalid mechanism "
                    f"{mechanism_id}"
                )

            rule_states = set(
                rule.get(
                    "states",
                    [],
                )
            )

            if not rule_states.issubset(
                VALID_MECHANISM_STATES
            ):
                all_states_valid = False

                errors.append(
                    f"{regime_id}: invalid states "
                    f"{sorted(rule_states)}"
                )

    confirmation_any = (
        specification.get(
            "confirmation_any",
            {},
        )
    )

    minimum_count = int(
        specification.get(
            "minimum_confirmation_count",
            0,
        )
    )

    if minimum_count > len(
        confirmation_any
    ):
        all_confirmation_counts_valid = False

        errors.append(
            f"{regime_id}: minimum confirmation "
            f"{minimum_count} exceeds available "
            f"{len(confirmation_any)}"
        )


checks[
    "all_mechanisms_valid"
] = all_mechanisms_valid

checks[
    "all_mechanism_states_valid"
] = all_states_valid

checks[
    "all_minimum_scores_valid"
] = all_scores_valid

checks[
    "all_confirmation_counts_valid"
] = (
    all_confirmation_counts_valid
)


confidence_weights = registry[
    "confidence"
][
    "components"
]


confidence_weight_sum = sum(
    float(value)
    for value in confidence_weights.values()
)


checks[
    "confidence_weights_sum_to_one"
] = (
    abs(
        confidence_weight_sum
        - 1.0
    )
    <= 1e-9
)


persistence = registry[
    "persistence"
]


checks[
    "confirmed_days_not_less_than_candidate_days"
] = (
    int(
        persistence[
            "confirmed_days"
        ]
    )
    >=
    int(
        persistence[
            "candidate_days"
        ]
    )
)


emergency = persistence[
    "emergency_override"
]


checks[
    "emergency_regime_exists"
] = (
    emergency[
        "regime"
    ]
    in regimes
)


checks[
    "emergency_mechanism_valid"
] = (
    emergency[
        "mechanism"
    ]
    in VALID_MECHANISMS
)


overall_status = (
    "PASS"
    if all(
        checks.values()
    )
    else "FAIL"
)


# =============================================================================
# Save
# =============================================================================

with open(
    REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        registry,
        file,
        allow_unicode=True,
        sort_keys=False,
    )


validation_payload = {
    "registry_version": (
        registry[
            "registry_version"
        ]
    ),
    "overall_status": (
        overall_status
    ),
    "regime_count": (
        len(regimes)
    ),
    "checks": checks,
    "errors": errors,
    "priority": priority,
    "files_created": [
        str(REGISTRY_PATH),
        str(VALIDATION_PATH),
        str(TEST_PATH),
    ],
}


with open(
    VALIDATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        validation_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )


test_lines = [
    "=" * 96,
    "IVMOS STEP 8A — REGIME REGISTRY VALIDATION",
    "=" * 96,
    f"Overall Status: {overall_status}",
    (
        "Registry Version: "
        f"{registry['registry_version']}"
    ),
    f"Regime Count: {len(regimes)}",
    "",
    "Tests:",
]


for name, passed in checks.items():

    test_lines.append(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


if errors:

    test_lines.extend([
        "",
        "Errors:",
        *[
            f"  - {error}"
            for error in errors
        ],
    ])


test_lines.extend([
    "",
    "Regime Priority:",
    *[
        f"  {index + 1}. {regime_id}"
        for index, regime_id
        in enumerate(priority)
    ],
    "",
    "Files created:",
    f"  {REGISTRY_PATH}",
    f"  {VALIDATION_PATH}",
    f"  {TEST_PATH}",
    "=" * 96,
])


TEST_PATH.write_text(
    "\n".join(
        test_lines
    ),
    encoding="utf-8",
)


# =============================================================================
# Display
# =============================================================================

print("\n" + "=" * 96)
print("IVMOS STEP 8A — REGIME REGISTRY VALIDATION")
print("=" * 96)

print(
    "Overall Status:",
    overall_status,
)

print(
    "Registry Version:",
    registry[
        "registry_version"
    ],
)

print(
    "Regime Count:",
    len(regimes),
)


print("\nTests:")

for name, passed in checks.items():

    print(
        f"  {'PASS' if passed else 'FAIL'} — {name}"
    )


print("\nRegime Priority:")

for index, regime_id in enumerate(
    priority,
    start=1,
):
    print(
        f"  {index}. {regime_id}"
    )


print("\nFiles created:")
print(" ", REGISTRY_PATH)
print(" ", VALIDATION_PATH)
print(" ", TEST_PATH)

print("=" * 96)


IVMOS STEP 8A — REGIME REGISTRY VALIDATION
Overall Status: PASS
Registry Version: 0.1.0-draft
Regime Count: 5

Tests:
  PASS — regime_count_is_5
  PASS — priority_contains_all_regimes
  PASS — priority_has_no_duplicates
  PASS — exactly_one_fallback_regime
  PASS — all_mechanisms_valid
  PASS — all_mechanism_states_valid
  PASS — all_minimum_scores_valid
  PASS — all_confirmation_counts_valid
  PASS — confidence_weights_sum_to_one
  PASS — confirmed_days_not_less_than_candidate_days
  PASS — emergency_regime_exists
  PASS — emergency_mechanism_valid

Regime Priority:
  1. SYSTEMIC_RISK_OFF
  2. BROAD_RISK_ON
  3. NARROW_RISK_ON
  4. RECOVERY_TRANSITION
  5. MIXED_TRANSITION

Files created:
  /content/drive/MyDrive/IVMOS/config/regime_registry_v0_1_draft.yaml
  /content/drive/MyDrive/IVMOS/outputs/regime_registry_v0_1_validation.json
  /content/drive/MyDrive/IVMOS/tests/step8a_regime_registry_validation.txt


# IVMOS STEP 8B — Regime Engine Runtime v0.1